In [ ]:
!pip install -q transformers datasets tokenizers accelerate evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 6.5 MB/s eta 0:00:00


In [1]:
import os
import torch
from google.colab import drive

drive.mount("/content/drive")

DRIVE = "/content/drive/MyDrive/BabyLM_Chinese_English_Translation"

corpus_file = f"{DRIVE}/final_english_training_corpus_gpt41mini_repaired.txt"

print("=" * 80)
print("SETUP CHECK")
print("=" * 80)

print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("GPU memory GB:", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))

print("Corpus exists:", os.path.exists(corpus_file))
print("Corpus path:", corpus_file)

if os.path.exists(corpus_file):
    print("Corpus size MB:", round(os.path.getsize(corpus_file) / 1024**2, 2))

Mounted at /content/drive
SETUP CHECK
CUDA available: True
GPU: NVIDIA A100-SXM4-80GB
GPU memory GB: 79.25
Corpus exists: True
Corpus path: /content/drive/MyDrive/BabyLM_Chinese_English_Translation/final_english_training_corpus_gpt41mini_repaired.txt
Corpus size MB: 580.87


95/5 train-validation split

In [ ]:
import os
import random

train_file = f"{DRIVE}/babyroberta_train_95.txt"
valid_file = f"{DRIVE}/babyroberta_valid_5.txt"

random.seed(42)

print("=" * 80)
print("CREATING 95/5 TRAIN-VALIDATION SPLIT")
print("=" * 80)

if os.path.exists(train_file) and os.path.exists(valid_file):
    print("Train/validation files already exist. Skipping split.")
else:
    train_count = 0
    valid_count = 0

    with open(corpus_file, "r", encoding="utf-8") as inp, \
         open(train_file, "w", encoding="utf-8") as train_out, \
         open(valid_file, "w", encoding="utf-8") as valid_out:

        for line in inp:
            line = line.strip()

            if not line:
                continue

            if random.random() < 0.05:
                valid_out.write(line + "\n")
                valid_count += 1
            else:
                train_out.write(line + "\n")
                train_count += 1

    print("Split complete.")
    print("Train lines:", f"{train_count:,}")
    print("Validation lines:", f"{valid_count:,}")

print()
print("Train file:", train_file)
print("Valid file:", valid_file)

print()
print("Train exists:", os.path.exists(train_file))
print("Valid exists:", os.path.exists(valid_file))

if os.path.exists(train_file):
    print("Train size MB:", round(os.path.getsize(train_file) / 1024**2, 2))

if os.path.exists(valid_file):
    print("Valid size MB:", round(os.path.getsize(valid_file) / 1024**2, 2))

CREATING 95/5 TRAIN-VALIDATION SPLIT
Train/validation files already exist. Skipping split.

Train file: /content/drive/MyDrive/BabyLM_Chinese_English_Translation/babyroberta_train_95.txt
Valid file: /content/drive/MyDrive/BabyLM_Chinese_English_Translation/babyroberta_valid_5.txt

Train exists: True
Valid exists: True
Train size MB: 551.83
Valid size MB: 29.04


Train/load Byte-Level BPE tokenizer

In [ ]:
from tokenizers import ByteLevelBPETokenizer
from transformers import RobertaTokenizerFast
import os
import shutil

tokenizer_dir = f"{DRIVE}/babyroberta_tokenizer_32k"

print("=" * 80)
print("RETRAIN TOKENIZER CLEANLY WITH tokenizer.json")
print("=" * 80)

# Remove broken/partial tokenizer folder
if os.path.exists(tokenizer_dir):
    shutil.rmtree(tokenizer_dir)

os.makedirs(tokenizer_dir, exist_ok=True)

# Train tokenizer
tokenizer_trainer = ByteLevelBPETokenizer()

tokenizer_trainer.train(
    files=[train_file],
    vocab_size=32000,
    min_frequency=2,
    special_tokens=[
        "<s>",
        "<pad>",
        "</s>",
        "<unk>",
        "<mask>",
    ],
)

# Save vocab/merges and tokenizer.json
tokenizer_trainer.save_model(tokenizer_dir)
tokenizer_trainer.save(f"{tokenizer_dir}/tokenizer.json")

print("Tokenizer files saved.")

# Load from tokenizer.json
tokenizer = RobertaTokenizerFast(
    tokenizer_file=f"{tokenizer_dir}/tokenizer.json",
    bos_token="<s>",
    eos_token="</s>",
    unk_token="<unk>",
    pad_token="<pad>",
    mask_token="<mask>",
    model_max_length=128,
)

tokenizer.save_pretrained(tokenizer_dir)

print("Tokenizer loaded successfully.")
print("Tokenizer vocab size:", tokenizer.vocab_size)
print("Mask token:", tokenizer.mask_token, tokenizer.mask_token_id)
print("Pad token:", tokenizer.pad_token, tokenizer.pad_token_id)
print("Tokenizer dir:", tokenizer_dir)

RETRAIN TOKENIZER CLEANLY WITH tokenizer.json
Tokenizer files saved.
Tokenizer loaded successfully.
Tokenizer vocab size: 32000
Mask token: <mask> 4
Pad token: <pad> 1
Tokenizer dir: /content/drive/MyDrive/BabyLM_Chinese_English_Translation/babyroberta_tokenizer_32k


Load text dataset and tokenize

In [ ]:
from datasets import load_dataset

block_size = 128

print("=" * 80)
print("LOADING AND TOKENIZING DATASET")
print("=" * 80)

raw_datasets = load_dataset(
    "text",
    data_files={
        "train": train_file,
        "validation": valid_file,
    }
)

print(raw_datasets)

def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        max_length=block_size,
        padding="max_length",
        return_special_tokens_mask=True,
    )

tokenized_datasets = raw_datasets.map(
    tokenize_function,
    batched=True,
    batch_size=1000,
    num_proc=2,
    remove_columns=["text"],
)

print(tokenized_datasets)
print("Tokenization complete.")

LOADING AND TOKENIZING DATASET


Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text'],
        num_rows: 4193879
    })
    validation: Dataset({
        features: ['text'],
        num_rows: 220645
    })
})


Map (num_proc=2):   0%|          | 0/4193879 [00:00<?, ? examples/s]

Map (num_proc=2):   0%|          | 0/220645 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask', 'special_tokens_mask'],
        num_rows: 4193879
    })
    validation: Dataset({
        features: ['input_ids', 'attention_mask', 'special_tokens_mask'],
        num_rows: 220645
    })
})
Tokenization complete.


Create same BabyRoBERTa architecture

In [ ]:
from transformers import RobertaConfig, RobertaForMaskedLM

print("=" * 80)

print("CREATING BABYROBERTA MODEL - SAME ARCHITECTURE")

print("=" * 80)

config = RobertaConfig(

    vocab_size=tokenizer.vocab_size,

    max_position_embeddings=130,   # 128 + 2 special tokens

    num_attention_heads=4,

    num_hidden_layers=4,

    hidden_size=256,

    intermediate_size=1024,

    type_vocab_size=1,

    pad_token_id=tokenizer.pad_token_id,

    bos_token_id=tokenizer.bos_token_id,

    eos_token_id=tokenizer.eos_token_id,

)

model = RobertaForMaskedLM(config)

total_params = sum(p.numel() for p in model.parameters())

print("BabyRoBERTa created from scratch.")

print("Vocab size:", tokenizer.vocab_size)

print("Total parameters:", f"{total_params:,}")

print("Total parameters in M:", round(total_params / 1e6, 2))

print(config)

CREATING BABYROBERTA MODEL - SAME ARCHITECTURE
BabyRoBERTa created from scratch.
Vocab size: 32000
Total parameters: 11,483,392
Total parameters in M: 11.48
RobertaConfig {
  "add_cross_attention": false,
  "attention_probs_dropout_prob": 0.1,
  "bos_token_id": 0,
  "classifier_dropout": null,
  "eos_token_id": 2,
  "hidden_act": "gelu",
  "hidden_dropout_prob": 0.1,
  "hidden_size": 256,
  "initializer_range": 0.02,
  "intermediate_size": 1024,
  "is_decoder": false,
  "layer_norm_eps": 1e-12,
  "max_position_embeddings": 130,
  "model_type": "roberta",
  "num_attention_heads": 4,
  "num_hidden_layers": 4,
  "pad_token_id": 1,
  "tie_word_embeddings": true,
  "transformers_version": "5.12.1",
  "type_vocab_size": 1,
  "use_cache": true,
  "vocab_size": 32000
}



Training setup + one checkpoint-safe run

In [ ]:
import os
import math
import torch
from transformers import DataCollatorForLanguageModeling, TrainingArguments, Trainer

print("=" * 80)
print("TRAINING SETUP")
print("=" * 80)

output_dir = f"{DRIVE}/babyroberta_108M_same_arch_95_5_checkpoints"
final_model_dir = f"{DRIVE}/babyroberta_108M_same_arch_95_5_final"

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=True,
    mlm_probability=0.15
)

training_args = TrainingArguments(
    output_dir=output_dir,

    do_train=True,
    do_eval=True,

    num_train_epochs=3,

    per_device_train_batch_size=128,
    per_device_eval_batch_size=128,
    gradient_accumulation_steps=1,

    learning_rate=5e-4,
    weight_decay=0.01,
    warmup_ratio=0.1,

    logging_strategy="steps",
    logging_steps=500,

    eval_strategy="steps",
    eval_steps=5000,

    save_strategy="steps",
    save_steps=5000,
    save_total_limit=5,

    bf16=True,
    fp16=False,

    dataloader_num_workers=2,
    report_to="none",

    load_best_model_at_end=False,
    seed=42,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    data_collator=data_collator,
)

print("Output checkpoint dir:", output_dir)
print("Final model dir:", final_model_dir)
print("Train examples:", len(tokenized_datasets["train"]))
print("Validation examples:", len(tokenized_datasets["validation"]))
print("Batch size:", training_args.per_device_train_batch_size)
print("BF16:", training_args.bf16)

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


TRAINING SETUP
Output checkpoint dir: /content/drive/MyDrive/BabyLM_Chinese_English_Translation/babyroberta_108M_same_arch_95_5_checkpoints
Final model dir: /content/drive/MyDrive/BabyLM_Chinese_English_Translation/babyroberta_108M_same_arch_95_5_final
Train examples: 4193879
Validation examples: 220645
Batch size: 128
BF16: True


In [ ]:
trainer.train()

trainer.save_model(final_model_dir)
tokenizer.save_pretrained(final_model_dir)

print("Training complete.")
print("Final model saved to:", final_model_dir)

Step,Training Loss,Validation Loss
5000,6.094512,5.914715
10000,4.971806,4.772932
15000,4.639884,4.437962
20000,4.472255,4.254391
25000,4.358114,4.145248
30000,4.280427,4.052437
35000,4.201414,3.974735
40000,4.147852,3.921277
45000,4.091316,3.863289
50000,4.051220,3.817227


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Training complete.
Final model saved to: /content/drive/MyDrive/BabyLM_Chinese_English_Translation/babyroberta_108M_same_arch_95_5_final


######################################preposition selection test ################################

####English BERT baseline: bert-base-uncased.#####

In [ ]:
PROJECT_DIR = "/content/drive/MyDrive/BabyLM_Chinese_English_Translation"

INPUT_FILE = f"{PROJECT_DIR}/Mandarin_Question_MASK_CLEAN.txt"
OPTIONS_FILE = f"{PROJECT_DIR}/Mandarin_answers.txt"

In [ ]:
print("Question exists:", os.path.exists(INPUT_FILE))
print("Answers exists:", os.path.exists(OPTIONS_FILE))

Question exists: True
Answers exists: True


In [ ]:
import os

import torch

import torch.nn.functional as F

import pandas as pd

from transformers import BertTokenizer, BertForMaskedLM

# =============================================================================

# PROJECT CONFIGURATION

# =============================================================================

PROJECT_DIR = "/content/drive/MyDrive/BabyLM_Chinese_English_Translation"

INPUT_FILE = f"{PROJECT_DIR}/Mandarin_Question_MASK_CLEAN.txt"

OPTIONS_FILE = f"{PROJECT_DIR}/Mandarin_answers.txt"

BASE_MODEL = "bert-base-uncased"

OUTPUT_DIR = f"{PROJECT_DIR}/bert_base_english_preposition_eval"

os.makedirs(OUTPUT_DIR, exist_ok=True)

OUTPUT_FILE = f"{OUTPUT_DIR}/bert_base_english_predictions.csv"

NORMALIZED_FILE = f"{OUTPUT_DIR}/bert_base_english_normalized_scores.csv"

OPTIONS = ["on", "at", "like", "as", "with", "inside", "of", "among", "in",
           "by", "from", "during", "about", "near", "out", "round", "until",
           "along", "for", "against", "across", "to", "off", "upon", "towards",
           "under", "around", "behind", "besides", "within", "beside", "into",
           "between", "up", "over", "before", "above", "down", "after", "till",
           "toward", "without"]

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# -------------------------------------------------
# LOAD ENGLISH BERT BASELINE
# -------------------------------------------------

tokenizer = BertTokenizer.from_pretrained(BASE_MODEL)
model = BertForMaskedLM.from_pretrained(BASE_MODEL)

model.to(device)
model.eval()

print("English BERT baseline loaded.")
print("Using device:", device)
print("Tokenizer vocab size:", tokenizer.vocab_size)
print("Model vocab size:", model.config.vocab_size)
print("Mask token:", tokenizer.mask_token)

# -------------------------------------------------
# MASK PREDICTION FUNCTION
# -------------------------------------------------

def predict_masked_word(sentence, options):
    inputs = tokenizer(
        sentence,
        return_tensors="pt",
        truncation=True,
        max_length=128
    )

    inputs = {k: v.to(device) for k, v in inputs.items()}
    input_ids = inputs["input_ids"]

    mask_positions = torch.where(input_ids == tokenizer.mask_token_id)[1]

    if mask_positions.numel() == 0:
        return {word: 0.0 for word in options}

    with torch.no_grad():
        logits = model(**inputs).logits

    mask_logits = logits[0, mask_positions[0], :]
    probs = F.softmax(mask_logits, dim=-1)

    word_probs = {}

    for word in options:
        token_ids = tokenizer.encode(word, add_special_tokens=False)

        if len(token_ids) == 1:
            word_probs[word] = probs[token_ids[0]].item()
        else:
            prob = 1.0
            for tid in token_ids:
                prob *= probs[tid].item()
            word_probs[word] = prob

    return word_probs

# -------------------------------------------------
# LOAD SENTENCES
# -------------------------------------------------

sentences = []

with open(INPUT_FILE, "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if not line:
            continue

        # Convert all mask formats to BERT mask
        line = line.replace("<mask>", "[MASK]")
        line = line.replace("____", "[MASK]")
        line = line.replace(" ___ ", " [MASK] ")
        line = line.replace("___", "[MASK]")
        line = line.replace("__", "[MASK]")

        if "[MASK]" not in line:
            parts = line.split()
            if len(parts) > 1:
                parts.insert(-1, "[MASK]")
                line = " ".join(parts)
            else:
                line = line + " [MASK]"

        line = line.replace("[MASK] [MASK]", "[MASK]")

        sentences.append(line)

print("Loaded", len(sentences), "sentences.")

# -------------------------------------------------
# RAW PREDICTIONS
# -------------------------------------------------

rows = []

for sentence in sentences:
    probs = predict_masked_word(sentence, OPTIONS)
    row = {"Question": sentence}
    row.update(probs)
    rows.append(row)

df = pd.DataFrame(rows)
df.to_csv(OUTPUT_FILE, index=False, encoding="utf-8-sig")

print("Raw predictions saved to:", OUTPUT_FILE)

# -------------------------------------------------
# NORMALIZED SCORES
# -------------------------------------------------

with open(OPTIONS_FILE, "r", encoding="utf-8") as f:
    option_lines = [line.strip() for line in f if line.strip()]

if len(option_lines) != len(sentences):
    raise ValueError("OPTIONS_FILE line count must match INPUT_FILE line count.")

normalized_rows = []

for i, allowed in enumerate(option_lines):
    allowed_list = [w.strip() for w in allowed.split(",") if w.strip()]
    sentence = sentences[i]

    row = {"sentence": sentence}

    total = sum(df.loc[i, w] for w in allowed_list if w in df.columns)

    for w in OPTIONS:
        if w in allowed_list and total > 0:
            row[w] = df.loc[i, w] / total
        else:
            row[w] = 0.0

    normalized_rows.append(row)

df_norm = pd.DataFrame(normalized_rows)
df_norm.to_csv(NORMALIZED_FILE, index=False, encoding="utf-8-sig")

print("Normalized scores saved to:", NORMALIZED_FILE)
print("DONE.")

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

[transformers] BertForMaskedLM LOAD REPORT from: bert-base-uncased
Key                         | Status     |  | 
----------------------------+------------+--+-
cls.seq_relationship.bias   | UNEXPECTED |  | 
bert.pooler.dense.bias      | UNEXPECTED |  | 
bert.pooler.dense.weight    | UNEXPECTED |  | 
cls.seq_relationship.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


English BERT baseline loaded.
Using device: cuda
Tokenizer vocab size: 30522
Model vocab size: 30522
Mask token: [MASK]
Loaded 100 sentences.
Raw predictions saved to: /content/drive/MyDrive/BabyLM_Chinese_English_Translation/bert_base_english_preposition_eval/bert_base_english_predictions.csv
Normalized scores saved to: /content/drive/MyDrive/BabyLM_Chinese_English_Translation/bert_base_english_preposition_eval/bert_base_english_normalized_scores.csv
DONE.


In [ ]:
import pandas as pd

pred = pd.read_csv(
    "/content/drive/MyDrive/BabyLM_Chinese_English_Translation/bert_base_english_preposition_eval/bert_base_english_predictions.csv"
)

pred.head(10)

,Question,on,at,like,as,with,inside,of,among,in,...,between,up,over,before,above,down,after,till,toward,without
0,"Now, you don't eat these all [MASK] once",0.000684,0.947930,1.058088e-05,1.106339e-04,3.819894e-05,8.897174e-06,1.091314e-03,1.181927e-07,0.002267,...,3.598496e-07,2.909867e-03,0.002018,1.113208e-04,1.331898e-05,2.540552e-04,9.576891e-05,1.832439e-06,2.031201e-07,1.506695e-05
1,So you're happy [MASK] your work?,0.000303,0.004220,9.239157e-06,7.700957e-06,9.429551e-01,9.275779e-06,7.944848e-05,5.457633e-07,0.006806,...,2.214326e-07,1.096062e-06,0.000027,7.766121e-07,1.435332e-06,2.018310e-07,3.935412e-05,1.949885e-08,1.412330e-06,9.743639e-05
2,Max plugs himself [MASK] again,0.007379,0.000068,2.598786e-06,2.110707e-05,1.668672e-04,2.641157e-03,7.886517e-06,3.364607e-07,0.935139,...,3.065353e-06,2.196877e-02,0.000224,1.657228e-05,1.909586e-06,5.260731e-04,1.021510e-05,5.017191e-07,9.653285e-08,1.798697e-06
3,She has a low opinion [MASK] herself,0.000426,0.000009,8.314457e-06,1.030021e-05,5.792552e-05,4.836806e-06,9.922660e-01,8.509571e-05,0.000516,...,1.148649e-05,2.776283e-08,0.000324,6.235194e-07,7.086229e-06,1.080693e-08,1.162402e-06,1.784809e-09,5.629797e-05,1.865528e-07
4,She also has the advantage [MASK] an outstandi...,0.000462,0.000027,2.825809e-05,3.822996e-03,9.682479e-04,7.288334e-07,9.853427e-01,7.800394e-06,0.001013,...,3.127602e-05,2.312929e-07,0.001482,1.543389e-06,3.555716e-06,4.469807e-07,3.735248e-05,4.698410e-09,6.376401e-06,1.783125e-05
5,Is there anyone else we should know [MASK]?,0.000561,0.000025,1.587571e-05,3.839141e-06,6.956469e-04,1.044541e-06,8.337114e-03,3.598771e-06,0.000038,...,2.064024e-06,2.396014e-06,0.000087,4.259253e-05,1.373358e-06,1.264624e-06,4.137541e-06,3.803158e-08,3.284524e-06,6.275583e-06
6,Find [MASK] which insurance area you live in,0.000028,0.000005,1.349096e-07,1.180023e-07,7.712607e-07,4.248724e-06,3.396369e-07,9.892445e-08,0.000043,...,2.420413e-08,3.290248e-05,0.000004,7.452255e-08,7.175029e-07,1.002999e-05,1.397798e-07,1.616263e-11,1.117187e-08,5.675996e-09
7,Mrs. Brooks was not usually curious [MASK] he...,0.000177,0.000297,3.609590e-05,6.558758e-05,2.782983e-03,2.525586e-06,1.623771e-02,4.275979e-04,0.000153,...,4.970841e-05,5.636801e-07,0.000426,6.470439e-05,2.240890e-06,8.538950e-07,9.509129e-04,1.717816e-07,1.214424e-04,1.503244e-05
8,I only moved [MASK] a couple of years ago,0.015936,0.000037,4.735078e-05,1.544871e-05,1.056214e-04,7.128546e-05,1.442007e-05,2.050392e-06,0.142433,...,2.556932e-05,1.903277e-02,0.007351,4.739958e-05,4.290200e-06,3.657811e-03,3.083223e-05,5.817160e-06,1.171473e-05,1.639837e-06
9,You live [MASK] home with your parents?,0.000027,0.986713,1.296427e-05,6.020574e-06,3.308467e-06,1.940760e-06,1.436317e-06,8.126786e-08,0.001775,...,2.921930e-07,2.838772e-04,0.000034,4.946897e-07,2.938076e-05,2.275589e-03,1.361449e-05,3.217179e-08,3.624517e-06,7.377744e-06


our trained BabyRoberta on chinese translated english

In [ ]:
import os
import torch
import torch.nn.functional as F
import pandas as pd
from transformers import RobertaTokenizerFast, RobertaForMaskedLM

# =============================================================================
# PROJECT CONFIGURATION
# =============================================================================

PROJECT_DIR = "/content/drive/MyDrive/BabyLM_Chinese_English_Translation"

INPUT_FILE = f"{PROJECT_DIR}/Mandarin_Question_MASK_CLEAN.txt"
OPTIONS_FILE = f"{PROJECT_DIR}/Mandarin_answers.txt"

MODEL_DIR = f"{PROJECT_DIR}/babyroberta_108M_same_arch_95_5_final"

OUTPUT_DIR = f"{PROJECT_DIR}/babyroberta_108M_preposition_eval"
os.makedirs(OUTPUT_DIR, exist_ok=True)

OUTPUT_FILE = f"{OUTPUT_DIR}/babyroberta_108M_predictions.csv"
NORMALIZED_FILE = f"{OUTPUT_DIR}/babyroberta_108M_normalized_scores.csv"

print("Question exists:", os.path.exists(INPUT_FILE))
print("Answers exists :", os.path.exists(OPTIONS_FILE))
print("Model exists   :", os.path.exists(MODEL_DIR))

OPTIONS = ["on", "at", "like", "as", "with", "inside", "of", "among", "in",
           "by", "from", "during", "about", "near", "out", "round", "until",
           "along", "for", "against", "across", "to", "off", "upon", "towards",
           "under", "around", "behind", "besides", "within", "beside", "into",
           "between", "up", "over", "before", "above", "down", "after", "till",
           "toward", "without"]

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

tokenizer = RobertaTokenizerFast.from_pretrained(MODEL_DIR)
model = RobertaForMaskedLM.from_pretrained(MODEL_DIR)

model.to(device)
model.eval()

print("BabyRoBERTa 108M model loaded.")
print("Using device:", device)
print("Tokenizer vocab size:", tokenizer.vocab_size)
print("Model vocab size:", model.config.vocab_size)
print("Mask token:", tokenizer.mask_token)

def predict_masked_word(sentence, options):
    inputs = tokenizer(
        sentence,
        return_tensors="pt",
        truncation=True,
        max_length=128
    )

    inputs = {k: v.to(device) for k, v in inputs.items()}
    input_ids = inputs["input_ids"]

    mask_positions = torch.where(input_ids == tokenizer.mask_token_id)[1]

    if mask_positions.numel() == 0:
        return {word: 0.0 for word in options}

    with torch.no_grad():
        logits = model(**inputs).logits

    mask_logits = logits[0, mask_positions[0], :]
    probs = F.softmax(mask_logits, dim=-1)

    word_probs = {}

    for word in options:
        token_ids = tokenizer.encode(word, add_special_tokens=False)

        if len(token_ids) == 1:
            word_probs[word] = probs[token_ids[0]].item()
        else:
            prob = 1.0
            for tid in token_ids:
                prob *= probs[tid].item()
            word_probs[word] = prob

    return word_probs

sentences = []

with open(INPUT_FILE, "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if not line:
            continue

        line = line.replace("[MASK]", "<mask>")
        line = line.replace("____", "<mask>")
        line = line.replace(" ___ ", " <mask> ")
        line = line.replace("___", "<mask>")
        line = line.replace("__", "<mask>")

        if "<mask>" not in line:
            parts = line.split()
            if len(parts) > 1:
                parts.insert(-1, "<mask>")
                line = " ".join(parts)
            else:
                line = line + " <mask>"

        line = line.replace("<mask> <mask>", "<mask>")
        sentences.append(line)

print("Loaded", len(sentences), "sentences.")

rows = []

for sentence in sentences:
    probs = predict_masked_word(sentence, OPTIONS)
    row = {"Question": sentence}
    row.update(probs)
    rows.append(row)

df = pd.DataFrame(rows)
df.to_csv(OUTPUT_FILE, index=False, encoding="utf-8-sig")

print("Raw predictions saved to:", OUTPUT_FILE)

with open(OPTIONS_FILE, "r", encoding="utf-8") as f:
    option_lines = [line.strip() for line in f if line.strip()]

if len(option_lines) != len(sentences):
    raise ValueError("OPTIONS_FILE line count must match INPUT_FILE line count.")

normalized_rows = []

for i, allowed in enumerate(option_lines):
    allowed_list = [w.strip() for w in allowed.split(",") if w.strip()]
    sentence = sentences[i]

    row = {"sentence": sentence}

    total = sum(df.loc[i, w] for w in allowed_list if w in df.columns)

    for w in OPTIONS:
        if w in allowed_list and total > 0:
            row[w] = df.loc[i, w] / total
        else:
            row[w] = 0.0

    normalized_rows.append(row)

df_norm = pd.DataFrame(normalized_rows)
df_norm.to_csv(NORMALIZED_FILE, index=False, encoding="utf-8-sig")

print("Normalized scores saved to:", NORMALIZED_FILE)
print("DONE.")

Question exists: True
Answers exists : True
Model exists   : True


Loading weights:   0%|          | 0/74 [00:00<?, ?it/s]

BabyRoBERTa 108M model loaded.
Using device: cuda
Tokenizer vocab size: 32000
Model vocab size: 32000
Mask token: <mask>
Loaded 100 sentences.
Raw predictions saved to: /content/drive/MyDrive/BabyLM_Chinese_English_Translation/babyroberta_108M_preposition_eval/babyroberta_108M_predictions.csv
Normalized scores saved to: /content/drive/MyDrive/BabyLM_Chinese_English_Translation/babyroberta_108M_preposition_eval/babyroberta_108M_normalized_scores.csv
DONE.


In [ ]:
import pandas as pd

pred = pd.read_csv(
    "/content/drive/MyDrive/BabyLM_Chinese_English_Translation/babyroberta_108M_preposition_eval/babyroberta_108M_predictions.csv"
)

print("Shape:", pred.shape)
pred.head(10)

Shape: (100, 43)


,Question,on,at,like,as,with,inside,of,among,in,...,between,up,over,before,above,down,after,till,toward,without
0,"Now, you don't eat these all <mask> once",0.000389,0.000019,6.488528e-10,0.000004,8.557750e-12,1.190095e-16,0.000028,1.439775e-12,0.000008,...,1.666585e-17,3.123513e-07,3.915693e-08,4.549750e-14,3.383707e-09,1.877687e-10,5.865266e-14,1.439007e-10,1.683312e-15,6.332940e-18
1,So you're happy <mask> your work?,0.000182,0.000043,2.173144e-08,0.000008,2.493530e-09,1.196286e-14,0.000035,1.580805e-10,0.000081,...,1.857211e-16,6.522860e-05,1.233455e-08,4.620901e-13,8.851921e-10,5.609288e-10,8.535330e-13,4.822489e-10,1.009178e-13,1.369633e-14
2,Max plugs himself <mask> again,0.000066,0.000041,3.170837e-09,0.000020,1.690919e-10,2.980812e-14,0.000004,9.856175e-11,0.000251,...,2.303180e-18,2.046991e-05,8.491668e-10,2.396257e-15,3.854570e-12,8.237002e-12,6.820897e-13,4.870278e-09,4.001419e-16,2.181860e-16
3,She has a low opinion <mask> herself,0.000870,0.000105,1.099143e-07,0.000066,1.662275e-09,1.147140e-12,0.000111,1.095192e-11,0.000331,...,7.546218e-13,4.157129e-06,3.360550e-08,1.353141e-11,7.000011e-09,1.724525e-09,2.605960e-11,8.342194e-10,2.052998e-14,1.988510e-15
4,She also has the advantage <mask> an outstandi...,0.000953,0.000087,6.568787e-09,0.000007,1.367833e-09,2.763142e-14,0.000044,1.483736e-09,0.000198,...,4.232167e-15,1.000779e-06,2.073381e-08,4.391436e-13,4.946126e-09,6.802682e-09,3.612366e-12,1.031172e-09,5.622166e-14,3.509424e-16
5,Is there anyone else we should know <mask>?,0.002337,0.000029,1.158395e-09,0.000047,5.779390e-10,4.391269e-15,0.000023,2.766858e-11,0.000149,...,1.875534e-16,6.418016e-08,1.181110e-09,9.881261e-16,7.306609e-11,5.184256e-09,3.804114e-14,9.892379e-12,2.156226e-15,2.690601e-17
6,Find <mask> which insurance area you live in,0.000084,0.000013,1.685657e-09,0.000010,2.717382e-11,8.762348e-15,0.000005,4.474240e-13,0.000254,...,3.531161e-19,1.038109e-05,1.486604e-10,8.920492e-16,4.368324e-12,1.378823e-12,2.398282e-14,6.247126e-13,1.038481e-17,5.501676e-17
7,Mrs. Brooks was not usually curious <mask> he...,0.000120,0.000081,4.510466e-08,0.000184,1.088892e-09,1.500516e-13,0.000031,9.687014e-12,0.000099,...,1.301556e-16,2.593131e-05,9.852225e-10,9.454076e-14,9.449435e-11,1.468017e-10,1.776987e-12,4.419631e-10,1.428471e-14,6.970229e-15
8,I only moved <mask> a couple of years ago,0.000173,0.000016,1.120187e-09,0.000004,2.424945e-12,2.493640e-16,0.000012,8.045091e-12,0.000007,...,4.082449e-17,1.476831e-06,8.515747e-08,2.318933e-13,9.778114e-10,1.592242e-11,2.304380e-13,3.069115e-12,8.910736e-16,4.932757e-18
9,You live <mask> home with your parents?,0.000283,0.000055,2.803505e-07,0.000014,3.045530e-08,5.002878e-13,0.000085,3.306321e-11,0.000293,...,8.813526e-14,2.476948e-05,1.411809e-08,3.879597e-12,2.831491e-09,2.751029e-09,7.458742e-12,1.738100e-09,4.493848e-14,6.974365e-14


In [ ]:
import pandas as pd

PROJECT_DIR = "/content/drive/MyDrive/BabyLM_Chinese_English_Translation"

bert_path = f"{PROJECT_DIR}/bert_base_english_preposition_eval/bert_base_english_normalized_scores.csv"
baby_path = f"{PROJECT_DIR}/babyroberta_108M_preposition_eval/babyroberta_108M_normalized_scores.csv"

bert = pd.read_csv(bert_path)
baby = pd.read_csv(baby_path)

ignore_cols = ["sentence", "Question"]
prep_cols = [c for c in bert.columns if c not in ignore_cols]

bert["bert_prediction"] = bert[prep_cols].idxmax(axis=1)
baby["babyroberta_prediction"] = baby[prep_cols].idxmax(axis=1)

compare = pd.DataFrame({
    "sentence": baby["sentence"],
    "BERT_prediction": bert["bert_prediction"],
    "BabyRoBERTa_prediction": baby["babyroberta_prediction"]
})

compare.head(20)

,sentence,BERT_prediction,BabyRoBERTa_prediction
0,"Now, you don't eat these all <mask> once",at,on
1,So you're happy <mask> your work?,with,on
2,Max plugs himself <mask> again,in,in
3,She has a low opinion <mask> herself,of,of
4,She also has the advantage <mask> an outstandi...,of,of
5,Is there anyone else we should know <mask>?,about,on
6,Find <mask> which insurance area you live in,out,of
7,Mrs. Brooks was not usually curious <mask> he...,about,by
8,I only moved <mask> a couple of years ago,in,on
9,You live <mask> home with your parents?,at,at


In [ ]:
import pandas as pd

gold = pd.read_csv("/content/drive/MyDrive/gold.csv")

print(gold.head())
print(gold.columns)
print(gold.shape)

   Test preposition                                       question   Precent
0     1          on       Now， you don't eat these all [MASK] once  0.045455
1     1          at       Now， you don't eat these all [MASK] once  0.625000
2     1        like       Now， you don't eat these all [MASK] once  0.193182
3     1          as       Now， you don't eat these all [MASK] once  0.136364
4     2          of   Sounds like you 're [MASK] trouble， Vologsky  0.761364
Index(['Test', 'preposition', 'question', 'Precent'], dtype='object')
(400, 4)


In [ ]:
import os
import re
import math
import torch
import numpy as np
import pandas as pd
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModelForMaskedLM

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

PROJECT_DIR = "/content/drive/MyDrive/BabyLM_Chinese_English_Translation"

models = {
    "English_BERT_Base": "bert-base-uncased",

    # English BabyRoBERTa (new baseline)
    "English_BabyRoBERTa_108M":
        f"{PROJECT_DIR}/babyroberta_english_same_arch_95_5_final",

    # Chinese-translated BabyRoBERTa
    "Chinese_BabyRoBERTa_108M":
        f"{PROJECT_DIR}/babyroberta_108M_same_arch_95_5_final",
}

essay_files = {
    "Native_English": "/content/drive/MyDrive/SLA_Project/native_wikitext.txt",
    "Native_English_ICNALE": "/content/drive/MyDrive/SLA_Project/ens_icnale_clean.txt",
    "Chinese_ICNALE": "/content/drive/MyDrive/SLA_Project/chinese_icnale_clean.txt",
    "Japanese_ICNALE": "/content/drive/MyDrive/SLA_Project/japanese_icnale_clean.txt",
    "Korean_ICNALE": "/content/drive/MyDrive/SLA_Project/kor_icnale_clean.txt",
    "Thai_ICNALE": "/content/drive/MyDrive/SLA_Project/tha_icnale_clean.txt",
    "Filipino_ICNALE": "/content/drive/MyDrive/SLA_Project/phl_icnale_clean.txt",
    "Urdu_ICNALE": "/content/drive/MyDrive/SLA_Project/urdu_icnale_clean.txt",
}

def compute_pseudo_perplexity(text, tokenizer, model, max_length=128):
    model.eval()

    encodings = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=max_length
    )

    input_ids = encodings["input_ids"].to(device)
    attention_mask = encodings["attention_mask"].to(device)

    seq_len = input_ids.size(1)

    log_likelihood = 0.0
    token_count = 0

    with torch.no_grad():
        for i in range(1, seq_len - 1):
            masked_input = input_ids.clone()
            masked_input[0, i] = tokenizer.mask_token_id

            outputs = model(masked_input, attention_mask=attention_mask)
            logits = outputs.logits

            log_probs = torch.log_softmax(logits[0, i], dim=-1)
            true_token = input_ids[0, i]

            log_likelihood += log_probs[true_token].item()
            token_count += 1

    if token_count == 0:
        return None

    return math.exp(-log_likelihood / token_count)

all_results = {}

for model_name, model_path in models.items():

    print("\n" + "=" * 80)
    print("Loading model:", model_name)
    print("Path:", model_path)
    print("=" * 80)

    tokenizer = AutoTokenizer.from_pretrained(model_path)
    model = AutoModelForMaskedLM.from_pretrained(model_path).to(device)
    model.eval()

    model_results = {}

    for group, path in essay_files.items():

        print(f"\nEvaluating {group}...")

        if not os.path.exists(path):
            print("Missing file:", path)
            continue

        with open(path, "r", encoding="utf-8", errors="ignore") as f:
            text = f.read()

        if group == "Chinese_ICNALE":
            essays = re.split(r'(?=I\s)', text)
            essays = [e.strip() for e in essays if len(e.split()) > 50]
        else:
            essays = [line.strip() for line in text.splitlines() if len(line.split()) > 5]

        essays = essays[:200]

        ppl_scores = []

        for essay in tqdm(essays):
            ppl = compute_pseudo_perplexity(
                essay,
                tokenizer,
                model,
                max_length=128
            )

            if ppl is not None and not math.isnan(ppl) and not math.isinf(ppl):
                ppl_scores.append(ppl)

        avg_ppl = np.mean(ppl_scores)
        model_results[group] = avg_ppl

        print(f"{group} Average Pseudo-PPL: {avg_ppl:.3f}")

    all_results[model_name] = model_results

print("\n" + "=" * 80)
print("FINAL PSEUDO-PERPLEXITY RESULTS")
print("=" * 80)

results_df = pd.DataFrame(all_results)
print(results_df)

output_csv = (
    f"{PROJECT_DIR}/pseudo_perplexity_three_model_comparison.csv"
)
results_df.to_csv(output_csv)

print("\nSaved results to:")
print(output_csv)


Loading model: English_BERT_Base
Path: bert-base-uncased


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

...

Thai_ICNALE Average Pseudo-PPL: 518.487

Evaluating Filipino_ICNALE...


100%|██████████| 200/200 [00:17<00:00, 11.70it/s]


Filipino_ICNALE Average Pseudo-PPL: 309.541

Evaluating Urdu_ICNALE...


100%|██████████| 200/200 [01:26<00:00,  2.32it/s]

Urdu_ICNALE Average Pseudo-PPL: 224.533

FINAL PSEUDO-PERPLEXITY RESULTS
                       English_BERT_Base  English_BabyRoBERTa_108M  \
Native_English                 15.096965                481.973946   
Native_English_ICNALE           3.740754                148.084206   
Chinese_ICNALE                  7.001253                127.343711   
Japanese_ICNALE               100.504326                114.487184   
Korean_ICNALE                  14.532735                119.390650   
Thai_ICNALE                   468.025969                141.925800   
Filipino_ICNALE                30.259848                128.347562   
Urdu_ICNALE                    11.812974                140.881237   

                       Chinese_BabyRoBERTa_108M  
Native_English                       717.502311  
Native_English_ICNALE                168.742460  
Chinese_ICNALE                        82.213764  
Japanese_ICNALE                      140.183140  
Korean_ICNALE                        125.34801

The English BabyRoBERTa achieved lower pseudo-perplexity on native English and on most non-Chinese learner corpora. In contrast, the Chinese-translated BabyRoBERTa achieved the lowest pseudo-perplexity on the Chinese ICNALE corpus, suggesting that training on translated Chinese-like English improves the model’s fit to English produced by Mandarin learners.

# Pseudo-Perplexity Evaluation

## What is Perplexity?

Perplexity tells us how well a language model understands a piece of text.

- **Lower perplexity = the model is more confident and understands the text better.**
- **Higher perplexity = the model is more confused by the text.**

For example:

> I drink coffee ___ the morning.

A good model will confidently predict **"in"**, giving a lower perplexity.

---

## Why do we use Pseudo-Perplexity?

Our models (BabyRoBERTa and BERT) are **Masked Language Models (MLMs)**.

They predict missing words instead of the next word, so they cannot use the normal perplexity calculation.

Instead, we use **Pseudo-Perplexity**, which is the standard way to evaluate masked language models.

---

## How is Pseudo-Perplexity Calculated?

For each sentence:

1. Hide (mask) one word.
2. Ask the model to predict the missing word.
3. Measure how confident the model is.
4. Repeat this for every word in the sentence.
5. Combine all the predictions to get one pseudo-perplexity score for the sentence.

If the model predicts the missing words confidently, the pseudo-perplexity will be **low**.

---

## How to Read the Results

**Lower pseudo-perplexity is better.**

A lower value means the model is more familiar with that type of text and can predict its words more accurately.

---

## What Did We Find?

- **English BabyRoBERTa** achieved lower pseudo-perplexity on **Native English**, showing that it understands natural English better.

- **Chinese BabyRoBERTa** achieved lower pseudo-perplexity on **Chinese ICNALE**, showing that it better models English written by Mandarin learners.

- For the Japanese, Korean, Thai, Filipino, and Urdu learner datasets, **English BabyRoBERTa** performed better. This is expected because our Chinese BabyRoBERTa was trained using **Chinese-to-English translated data**, so it is specifically adapted to Chinese learner English.

Overall, these results support our hypothesis that training on Chinese-influenced English helps the model better represent English written by Mandarin learners.

In [ ]:
import os
import pandas as pd

PROJECT_DIR = "/content/drive/MyDrive/BabyLM_Chinese_English_Translation"

ppl_file = f"{PROJECT_DIR}/pseudo_perplexity_three_model_comparison.csv"

df = pd.read_csv(ppl_file, index_col=0)

# Keep only same-architecture BabyRoBERTa models
baby_df = df[[
    "English_BabyRoBERTa_108M",
    "Chinese_BabyRoBERTa_108M"
]]

# Add which model has lower pseudo-perplexity
baby_df["Better_Model"] = baby_df.idxmin(axis=1)
baby_df["Difference"] = (
    baby_df["English_BabyRoBERTa_108M"] -
    baby_df["Chinese_BabyRoBERTa_108M"]
)

baby_df["Chinese_Model_Lower"] = (
    baby_df["Chinese_BabyRoBERTa_108M"] <
    baby_df["English_BabyRoBERTa_108M"]
)

print("=" * 80)
print("BABYROBERTA PSEUDO-PERPLEXITY COMPARISON")
print("=" * 80)
print(baby_df)

output_file = f"{PROJECT_DIR}/babyroberta_pseudo_perplexity_comparison_only.csv"
baby_df.to_csv(output_file)

print("\nSaved to:")
print(output_file)

BABYROBERTA PSEUDO-PERPLEXITY COMPARISON
                       English_BabyRoBERTa_108M  Chinese_BabyRoBERTa_108M  \
Native_English                       481.973946                717.502311   
Native_English_ICNALE                148.084206                168.742460   
Chinese_ICNALE                       127.343711                 82.213764   
Japanese_ICNALE                      114.487184                140.183140   
Korean_ICNALE                        119.390650                125.348016   
Thai_ICNALE                          141.925800                518.486829   
Filipino_ICNALE                      128.347562                309.541266   
Urdu_ICNALE                          140.881237                224.533477   

                                   Better_Model  Difference  \
Native_English         English_BabyRoBERTa_108M -235.528365   
Native_English_ICNALE  English_BabyRoBERTa_108M  -20.658254   
Chinese_ICNALE         Chinese_BabyRoBERTa_108M   45.129947   
Japanese_ICNA

Setup + load official English BabyLM dataset

In [ ]:
!pip install -q datasets transformers tokenizers accelerate evaluate

import os
import torch
from google.colab import drive
from datasets import load_dataset

drive.mount("/content/drive")

PROJECT_DIR = "/content/drive/MyDrive/BabyLM_Chinese_English_Translation"
os.makedirs(PROJECT_DIR, exist_ok=True)

print("=" * 80)
print("SETUP CHECK")
print("=" * 80)

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

print("Project folder:", PROJECT_DIR)

print("\nLoading official English BabyLM 2026 Strict dataset...")

english_dataset = load_dataset(
    "BabyLM-community/BabyLM-2026-Strict",
    split="train"
)

print(english_dataset)
print("\nColumns:", english_dataset.column_names)
print("\nFirst example:")
print(english_dataset[0])

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
SETUP CHECK
CUDA available: True
GPU: NVIDIA A100-SXM4-80GB
Project folder: /content/drive/MyDrive/BabyLM_Chinese_English_Translation

Loading official English BabyLM 2026 Strict dataset...


README.md:   0%|          | 0.00/2.55k [00:00<?, ?B/s]

bnc_spoken.train.txt:   0%|          | 0.00/39.5M [00:00<?, ?B/s]

childes.train.txt:   0%|          | 0.00/152M [00:00<?, ?B/s]

gutenberg.train.txt:   0%|          | 0.00/141M [00:00<?, ?B/s]

open_subtitles.train.txt:   0%|          | 0.00/121M [00:00<?, ?B/s]

simple_wiki.train.txt:   0%|          | 0.00/87.9M [00:00<?, ?B/s]

switchboard.train.txt:   0%|          | 0.00/1.22M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/11601896 [00:00<?, ? examples/s]

Dataset({
    features: ['text'],
    num_rows: 11601896
})

Columns: ['text']

First example:
{'text': "Well it's just that, you know, a pound, or a hundred pounds today, is not the same as a hundred pounds in a year's time, or two, two years' time."}


Create matched English corpus with same number of lines

In [ ]:
import os
import random

MATCHED_TOTAL_LINES = 4_414_524

english_corpus_file = f"{PROJECT_DIR}/babylm_english_matched_4414524_lines.txt"

random.seed(42)

print("=" * 80)
print("CREATING MATCHED ENGLISH BABYLM CORPUS")
print("=" * 80)

if os.path.exists(english_corpus_file):
    print("Matched English corpus already exists. Skipping creation.")
else:
    total_available = len(english_dataset)
    print("Total available English BabyLM lines:", f"{total_available:,}")
    print("Sampling lines:", f"{MATCHED_TOTAL_LINES:,}")

    selected_indices = random.sample(range(total_available), MATCHED_TOTAL_LINES)
    selected_indices = sorted(selected_indices)

    with open(english_corpus_file, "w", encoding="utf-8") as f:
        for idx in selected_indices:
            text = english_dataset[idx]["text"].strip()
            if text:
                f.write(text + "\n")

    print("Matched English corpus created.")

print("Saved file:", english_corpus_file)
print("Exists:", os.path.exists(english_corpus_file))
print("Size MB:", round(os.path.getsize(english_corpus_file) / 1024**2, 2))

CREATING MATCHED ENGLISH BABYLM CORPUS
Total available English BabyLM lines: 11,601,896
Sampling lines: 4,414,524
Matched English corpus created.
Saved file: /content/drive/MyDrive/BabyLM_Chinese_English_Translation/babylm_english_matched_4414524_lines.txt
Exists: True
Size MB: 197.07


Inspect English corpus stats

In [ ]:
import os
import statistics

print("=" * 80)
print("INSPECT MATCHED ENGLISH CORPUS")
print("=" * 80)

line_count = 0
word_counts = []
char_counts = []

with open(english_corpus_file, "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        line_count += 1
        word_counts.append(len(line.split()))
        char_counts.append(len(line))

print("Total lines       :", f"{line_count:,}")
print("Total words       :", f"{sum(word_counts):,}")
print("Total characters  :", f"{sum(char_counts):,}")
print("Avg words/line    :", round(statistics.mean(word_counts), 2))
print("Median words/line :", statistics.median(word_counts))
print("Min words/line    :", min(word_counts))
print("Max words/line    :", max(word_counts))
print("File size MB      :", round(os.path.getsize(english_corpus_file) / 1024**2, 2))

print("\nFirst 5 lines:")
with open(english_corpus_file, "r", encoding="utf-8") as f:
    for i in range(5):
        print(f"{i+1}: {f.readline().strip()[:250]}")

INSPECT MATCHED ENGLISH CORPUS
Total lines       : 4,414,524
Total words       : 38,032,183
Total characters  : 201,665,585
Avg words/line    : 8.62
Median words/line : 5.0
Min words/line    : 1
Max words/line    : 2804
File size MB      : 197.07

First 5 lines:
1: That's right, yes, that's right, so, in actual fact, this area here, right, is going to be the discounted sum of all future rural incomes.
2: Well it's, well, er, there's no need why that should be the case, it could be, you know, it could look something like that.
3: But, up in the, there the migrant's decision making process will be, is this area here  right, is that area there, greater  than that area there.
4: Alright, so this is the discounted sum in er, in non-agriculture, in the urban area, and this is the value, the discounted sum in agriculture.
5: Now, clearly on the, on the way I've drawn this diagram it is.


In [ ]:
import os

TARGET_WORDS = 108_042_603

english_wordmatched_file = f"{PROJECT_DIR}/babylm_english_matched_108M_words.txt"

print("=" * 80)
print("CREATING WORD-MATCHED ENGLISH BABYLM CORPUS")
print("=" * 80)

if os.path.exists(english_wordmatched_file):
    print("Word-matched English corpus already exists. Skipping creation.")
else:
    total_words = 0
    total_lines = 0

    with open(english_wordmatched_file, "w", encoding="utf-8") as f:
        for ex in english_dataset:
            text = ex["text"].strip()

            if not text:
                continue

            wc = len(text.split())

            f.write(text + "\n")

            total_words += wc
            total_lines += 1

            if total_words >= TARGET_WORDS:
                break

    print("Word-matched English corpus created.")
    print("Lines written:", f"{total_lines:,}")
    print("Words written:", f"{total_words:,}")

print("Saved file:", english_wordmatched_file)
print("Exists:", os.path.exists(english_wordmatched_file))
print("Size MB:", round(os.path.getsize(english_wordmatched_file) / 1024**2, 2))

CREATING WORD-MATCHED ENGLISH BABYLM CORPUS
Word-matched English corpus created.
Lines written: 11,601,896
Words written: 100,000,000
Saved file: /content/drive/MyDrive/BabyLM_Chinese_English_Translation/babylm_english_matched_108M_words.txt
Exists: True
Size MB: 518.13


Create 95/5 split for English BabyRoBERTa baseline

In [ ]:
import os
import random

english_train_file = f"{PROJECT_DIR}/babyroberta_english_train_95.txt"
english_valid_file = f"{PROJECT_DIR}/babyroberta_english_valid_5.txt"

random.seed(42)

print("=" * 80)
print("CREATING 95/5 SPLIT FOR ENGLISH BABYROBERTA")
print("=" * 80)

if os.path.exists(english_train_file) and os.path.exists(english_valid_file):
    print("English train/validation files already exist. Skipping split.")
else:
    train_count = 0
    valid_count = 0

    with open(english_wordmatched_file, "r", encoding="utf-8") as inp, \
         open(english_train_file, "w", encoding="utf-8") as train_out, \
         open(english_valid_file, "w", encoding="utf-8") as valid_out:

        for line in inp:
            line = line.strip()
            if not line:
                continue

            if random.random() < 0.05:
                valid_out.write(line + "\n")
                valid_count += 1
            else:
                train_out.write(line + "\n")
                train_count += 1

    print("Split complete.")
    print("Train lines:", f"{train_count:,}")
    print("Valid lines:", f"{valid_count:,}")

print("Train file:", english_train_file)
print("Valid file:", english_valid_file)
print("Train exists:", os.path.exists(english_train_file))
print("Valid exists:", os.path.exists(english_valid_file))

if os.path.exists(english_train_file):
    print("Train size MB:", round(os.path.getsize(english_train_file) / 1024**2, 2))

if os.path.exists(english_valid_file):
    print("Valid size MB:", round(os.path.getsize(english_valid_file) / 1024**2, 2))

CREATING 95/5 SPLIT FOR ENGLISH BABYROBERTA
Split complete.
Train lines: 11,021,932
Valid lines: 579,964
Train file: /content/drive/MyDrive/BabyLM_Chinese_English_Translation/babyroberta_english_train_95.txt
Valid file: /content/drive/MyDrive/BabyLM_Chinese_English_Translation/babyroberta_english_valid_5.txt
Train exists: True
Valid exists: True
Train size MB: 492.32
Valid size MB: 25.82


In [ ]:
from transformers import RobertaTokenizerFast
import os

print("=" * 80)
print("LOADING EXISTING BABYROBERTA TOKENIZER")
print("=" * 80)

tokenizer_dir = f"{PROJECT_DIR}/babyroberta_tokenizer_32k"

assert os.path.exists(tokenizer_dir), "Tokenizer directory not found."

tokenizer = RobertaTokenizerFast.from_pretrained(tokenizer_dir)

print("Tokenizer loaded successfully.")
print("Tokenizer directory:", tokenizer_dir)
print("Vocabulary size:", tokenizer.vocab_size)
print("Mask token:", tokenizer.mask_token, tokenizer.mask_token_id)
print("Pad token :", tokenizer.pad_token, tokenizer.pad_token_id)
print("CLS token :", tokenizer.cls_token)
print("SEP token :", tokenizer.sep_token)

LOADING EXISTING BABYROBERTA TOKENIZER
Tokenizer loaded successfully.
Tokenizer directory: /content/drive/MyDrive/BabyLM_Chinese_English_Translation/babyroberta_tokenizer_32k
Vocabulary size: 32000
Mask token: <mask> 4
Pad token : <pad> 1
CLS token : <s>
SEP token : </s>


 Load and tokenize English train/validation data

In [ ]:
from datasets import load_dataset

block_size = 128

print("=" * 80)
print("LOADING AND TOKENIZING ENGLISH BABYLM DATA")
print("=" * 80)

raw_english_datasets = load_dataset(
    "text",
    data_files={
        "train": english_train_file,
        "validation": english_valid_file,
    }
)

print(raw_english_datasets)

def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        max_length=block_size,
        padding="max_length",
        return_special_tokens_mask=True,
    )

tokenized_english_datasets = raw_english_datasets.map(
    tokenize_function,
    batched=True,
    batch_size=1000,
    num_proc=2,
    remove_columns=["text"],
)

print(tokenized_english_datasets)
print("English tokenization complete.")

LOADING AND TOKENIZING ENGLISH BABYLM DATA


Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text'],
        num_rows: 11021932
    })
    validation: Dataset({
        features: ['text'],
        num_rows: 579964
    })
})


Map (num_proc=2):   0%|          | 0/11021932 [00:00<?, ? examples/s]

Map (num_proc=2):   0%|          | 0/579964 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask', 'special_tokens_mask'],
        num_rows: 11021932
    })
    validation: Dataset({
        features: ['input_ids', 'attention_mask', 'special_tokens_mask'],
        num_rows: 579964
    })
})
English tokenization complete.


Create English BabyRoBERTa model with same architecture

In [ ]:
from transformers import RobertaConfig, RobertaForMaskedLM

print("=" * 80)
print("CREATING ENGLISH BABYROBERTA BASELINE - SAME ARCHITECTURE")
print("=" * 80)

english_config = RobertaConfig(
    vocab_size=tokenizer.vocab_size,
    max_position_embeddings=130,
    num_attention_heads=4,
    num_hidden_layers=4,
    hidden_size=256,
    intermediate_size=1024,
    type_vocab_size=1,
    pad_token_id=tokenizer.pad_token_id,
    bos_token_id=tokenizer.bos_token_id,
    eos_token_id=tokenizer.eos_token_id,
)

english_model = RobertaForMaskedLM(english_config)

total_params = sum(p.numel() for p in english_model.parameters())

print("English BabyRoBERTa baseline created.")
print("Total parameters:", f"{total_params:,}")
print("Total parameters in M:", round(total_params / 1e6, 2))
print(english_config)

CREATING ENGLISH BABYROBERTA BASELINE - SAME ARCHITECTURE
English BabyRoBERTa baseline created.
Total parameters: 11,483,392
Total parameters in M: 11.48
RobertaConfig {
  "add_cross_attention": false,
  "attention_probs_dropout_prob": 0.1,
  "bos_token_id": 0,
  "classifier_dropout": null,
  "eos_token_id": 2,
  "hidden_act": "gelu",
  "hidden_dropout_prob": 0.1,
  "hidden_size": 256,
  "initializer_range": 0.02,
  "intermediate_size": 1024,
  "is_decoder": false,
  "layer_norm_eps": 1e-12,
  "max_position_embeddings": 130,
  "model_type": "roberta",
  "num_attention_heads": 4,
  "num_hidden_layers": 4,
  "pad_token_id": 1,
  "tie_word_embeddings": true,
  "transformers_version": "5.12.1",
  "type_vocab_size": 1,
  "use_cache": true,
  "vocab_size": 32000
}



Training setup (identical to the translated BabyRoBERTa)

In [ ]:
from transformers import DataCollatorForLanguageModeling, TrainingArguments, Trainer

output_dir = f"{PROJECT_DIR}/babyroberta_english_same_arch_95_5_checkpoints"
final_model_dir = f"{PROJECT_DIR}/babyroberta_english_same_arch_95_5_final"

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=True,
    mlm_probability=0.15
)

training_args = TrainingArguments(
    output_dir=output_dir,

    max_steps=98295,

    per_device_train_batch_size=128,
    per_device_eval_batch_size=128,
    gradient_accumulation_steps=1,

    learning_rate=5e-4,
    weight_decay=0.01,
    warmup_ratio=0.1,

    logging_steps=500,
    eval_steps=5000,
    save_steps=5000,

    eval_strategy="steps",
    save_strategy="steps",
    save_total_limit=5,

    bf16=True,
    fp16=False,

    dataloader_num_workers=2,
    report_to="none",
    load_best_model_at_end=False,
)

trainer = Trainer(
    model=english_model,
    args=training_args,
    train_dataset=tokenized_english_datasets["train"],
    eval_dataset=tokenized_english_datasets["validation"],
    data_collator=data_collator,
)

print("Checkpoint folder :", output_dir)
print("Final model folder:", final_model_dir)
print("Train examples    :", len(tokenized_english_datasets["train"]))
print("Validation        :", len(tokenized_english_datasets["validation"]))
print("Batch size        :", training_args.per_device_train_batch_size)
print("Max steps         :", training_args.max_steps)

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Checkpoint folder : /content/drive/MyDrive/BabyLM_Chinese_English_Translation/babyroberta_english_same_arch_95_5_checkpoints
Final model folder: /content/drive/MyDrive/BabyLM_Chinese_English_Translation/babyroberta_english_same_arch_95_5_final
Train examples    : 11021932
Validation        : 579964
Batch size        : 128
Max steps         : 98295


In [ ]:
trainer.train()

trainer.save_model(final_model_dir)
tokenizer.save_pretrained(final_model_dir)

print("Training complete.")
print("Final English BabyRoBERTa saved to:", final_model_dir)

Step,Training Loss,Validation Loss
5000,1.852459,1.583568
10000,1.360004,1.136381
15000,1.158850,0.935568
20000,1.059967,0.855600
25000,1.001068,0.801266
30000,0.970914,0.766645
35000,0.925413,0.734587
40000,0.899059,0.708619
45000,0.874942,0.686001
50000,0.860374,0.668542


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Training complete.
Final English BabyRoBERTa saved to: /content/drive/MyDrive/BabyLM_Chinese_English_Translation/babyroberta_english_same_arch_95_5_final


In [ ]:
import os
import torch
import torch.nn.functional as F
import pandas as pd
from transformers import RobertaTokenizerFast, RobertaForMaskedLM

PROJECT_DIR = "/content/drive/MyDrive/BabyLM_Chinese_English_Translation"

INPUT_FILE = f"{PROJECT_DIR}/Mandarin_Question_MASK_CLEAN.txt"
OPTIONS_FILE = f"{PROJECT_DIR}/Mandarin_answers.txt"

MODEL_DIR = f"{PROJECT_DIR}/babyroberta_english_same_arch_95_5_final"

OUTPUT_DIR = f"{PROJECT_DIR}/babyroberta_english_preposition_eval"
os.makedirs(OUTPUT_DIR, exist_ok=True)

OUTPUT_FILE = f"{OUTPUT_DIR}/babyroberta_english_predictions.csv"
NORMALIZED_FILE = f"{OUTPUT_DIR}/babyroberta_english_normalized_scores.csv"

OPTIONS = ["on", "at", "like", "as", "with", "inside", "of", "among", "in",
           "by", "from", "during", "about", "near", "out", "round", "until",
           "along", "for", "against", "across", "to", "off", "upon", "towards",
           "under", "around", "behind", "besides", "within", "beside", "into",
           "between", "up", "over", "before", "above", "down", "after", "till",
           "toward", "without"]

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Question exists:", os.path.exists(INPUT_FILE))
print("Answers exists :", os.path.exists(OPTIONS_FILE))
print("Model exists   :", os.path.exists(MODEL_DIR))

tokenizer = RobertaTokenizerFast.from_pretrained(MODEL_DIR)
model = RobertaForMaskedLM.from_pretrained(MODEL_DIR)

model.to(device)
model.eval()

print("English BabyRoBERTa model loaded.")
print("Using device:", device)
print("Tokenizer vocab size:", tokenizer.vocab_size)
print("Model vocab size:", model.config.vocab_size)
print("Mask token:", tokenizer.mask_token)

def predict_masked_word(sentence, options):
    inputs = tokenizer(
        sentence,
        return_tensors="pt",
        truncation=True,
        max_length=128
    )

    inputs = {k: v.to(device) for k, v in inputs.items()}
    input_ids = inputs["input_ids"]

    mask_positions = torch.where(input_ids == tokenizer.mask_token_id)[1]

    if mask_positions.numel() == 0:
        return {word: 0.0 for word in options}

    with torch.no_grad():
        logits = model(**inputs).logits

    mask_logits = logits[0, mask_positions[0], :]
    probs = F.softmax(mask_logits, dim=-1)

    word_probs = {}

    for word in options:
        token_ids = tokenizer.encode(word, add_special_tokens=False)

        if len(token_ids) == 1:
            word_probs[word] = probs[token_ids[0]].item()
        else:
            prob = 1.0
            for tid in token_ids:
                prob *= probs[tid].item()
            word_probs[word] = prob

    return word_probs

sentences = []

with open(INPUT_FILE, "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if not line:
            continue

        line = line.replace("[MASK]", "<mask>")
        line = line.replace("____", "<mask>")
        line = line.replace(" ___ ", " <mask> ")
        line = line.replace("___", "<mask>")
        line = line.replace("__", "<mask>")

        if "<mask>" not in line:
            parts = line.split()
            if len(parts) > 1:
                parts.insert(-1, "<mask>")
                line = " ".join(parts)
            else:
                line = line + " <mask>"

        line = line.replace("<mask> <mask>", "<mask>")
        sentences.append(line)

print("Loaded", len(sentences), "sentences.")

rows = []

for sentence in sentences:
    probs = predict_masked_word(sentence, OPTIONS)
    row = {"Question": sentence}
    row.update(probs)
    rows.append(row)

df = pd.DataFrame(rows)
df.to_csv(OUTPUT_FILE, index=False, encoding="utf-8-sig")

print("Raw predictions saved to:", OUTPUT_FILE)

with open(OPTIONS_FILE, "r", encoding="utf-8") as f:
    option_lines = [line.strip() for line in f if line.strip()]

if len(option_lines) != len(sentences):
    raise ValueError("OPTIONS_FILE line count must match INPUT_FILE line count.")

normalized_rows = []

for i, allowed in enumerate(option_lines):
    allowed_list = [w.strip() for w in allowed.split(",") if w.strip()]
    sentence = sentences[i]

    row = {"sentence": sentence}
    total = sum(df.loc[i, w] for w in allowed_list if w in df.columns)

    for w in OPTIONS:
        if w in allowed_list and total > 0:
            row[w] = df.loc[i, w] / total
        else:
            row[w] = 0.0

    normalized_rows.append(row)

df_norm = pd.DataFrame(normalized_rows)
df_norm.to_csv(NORMALIZED_FILE, index=False, encoding="utf-8-sig")

print("Normalized scores saved to:", NORMALIZED_FILE)
print("DONE.")

Question exists: True
Answers exists : True
Model exists   : True


Loading weights:   0%|          | 0/74 [00:00<?, ?it/s]

English BabyRoBERTa model loaded.
Using device: cuda
Tokenizer vocab size: 32000
Model vocab size: 32000
Mask token: <mask>
Loaded 100 sentences.
Raw predictions saved to: /content/drive/MyDrive/BabyLM_Chinese_English_Translation/babyroberta_english_preposition_eval/babyroberta_english_predictions.csv
Normalized scores saved to: /content/drive/MyDrive/BabyLM_Chinese_English_Translation/babyroberta_english_preposition_eval/babyroberta_english_normalized_scores.csv
DONE.


Check MUSE files + English BabyRoBERTa model

In [ ]:
import os
import torch
from google.colab import drive

drive.mount("/content/drive", force_remount=False)

# =============================================================================
# FINAL EXPERIMENT CONFIGURATION
# =============================================================================

PROJECT_DIR = "/content/drive/MyDrive/BabyLM_Chinese_English_Translation"

# Both branches start from this exact same trained model
BASE_ENGLISH_MODEL = (
    f"{PROJECT_DIR}/babyroberta_english_same_arch_95_5_final"
)

# Same English training and validation data for both branches
TRAIN_FILE = f"{PROJECT_DIR}/babyroberta_english_train_95.txt"
VALID_FILE = f"{PROJECT_DIR}/babyroberta_english_valid_5.txt"

# Previously created MUSE resources
MUSE_DIR = f"{PROJECT_DIR}/babyroberta_muse_alignment"

ALIGNMENT_MATRIX = (
    f"{MUSE_DIR}/muse_en_to_zh_alignment_matrix.npy"
)

PROJECTION_MATRIX = (
    f"{MUSE_DIR}/muse300_to_babyroberta256_projection.npy"
)

MUSE_ENGLISH_VECTORS = "/content/drive/MyDrive/muse_data/wiki.en.vec"
MUSE_DICTIONARY = "/content/drive/MyDrive/muse_data_csee/en-zh.txt"

# Separate outputs so neither branch overwrites the other
CONTROL_OUTPUT_DIR = (
    f"{PROJECT_DIR}/babyroberta_english_continued_mlm_control"
)

MUSE_OUTPUT_DIR = (
    f"{PROJECT_DIR}/babyroberta_english_continued_mlm_muse"
)

# Reproducibility
SEED = 42

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("=" * 90)
print("FINAL CONTROL VS MUSE EXPERIMENT — SETUP CHECK")
print("=" * 90)

print("Device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(
        "GPU memory:",
        round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2),
        "GB"
    )

required_paths = {
    "Base English BabyRoBERTa": BASE_ENGLISH_MODEL,
    "Training file": TRAIN_FILE,
    "Validation file": VALID_FILE,
    "MUSE English vectors": MUSE_ENGLISH_VECTORS,
    "MUSE dictionary": MUSE_DICTIONARY,
    "English-to-Chinese alignment matrix": ALIGNMENT_MATRIX,
    "MUSE-to-BabyRoBERTa projection": PROJECTION_MATRIX,
}

print("\nRequired files:")

all_found = True

for name, path in required_paths.items():
    exists = os.path.exists(path)
    all_found = all_found and exists
    print(f"{name:40s}: {exists}")
    print(f"  {path}")

print("\nControl output:")
print(CONTROL_OUTPUT_DIR)

print("\nMUSE output:")
print(MUSE_OUTPUT_DIR)

print("\nAll required resources found:", all_found)

if not all_found:
    raise FileNotFoundError(
        "One or more required files are missing. Check the paths printed above."
    )

print("\nSetup complete. Both branches will start from the same model and use identical training settings.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
FINAL CONTROL VS MUSE EXPERIMENT — SETUP CHECK
Device: cuda
GPU: NVIDIA A100-SXM4-80GB
GPU memory: 79.25 GB

Required files:
Base English BabyRoBERTa                : True
  /content/drive/MyDrive/BabyLM_Chinese_English_Translation/babyroberta_english_same_arch_95_5_final
Training file                           : True
  /content/drive/MyDrive/BabyLM_Chinese_English_Translation/babyroberta_english_train_95.txt
Validation file                         : True
  /content/drive/MyDrive/BabyLM_Chinese_English_Translation/babyroberta_english_valid_5.txt
MUSE English vectors                    : True
  /content/drive/MyDrive/muse_data/wiki.en.vec
MUSE dictionary                         : True
  /content/drive/MyDrive/muse_data_csee/en-zh.txt
English-to-Chinese alignment matrix     : True
  /content/drive/MyDrive/BabyLM_Chinese_English_Translation/babyroberta_muse_alig

 Load and tokenize the shared continued-training dataset

In [ ]:
from datasets import load_dataset
from transformers import RobertaTokenizerFast

print("=" * 90)
print("LOADING SHARED DATA FOR CONTROL AND MUSE BRANCHES")
print("=" * 90)

# Fixed experiment sizes
MAX_TRAIN_EXAMPLES = 2_000_000
MAX_VALID_EXAMPLES = 100_000
BLOCK_SIZE = 128

# Load the exact tokenizer used by the English BabyRoBERTa
tokenizer = RobertaTokenizerFast.from_pretrained(BASE_ENGLISH_MODEL)

# Load full text files
raw_datasets = load_dataset(
    "text",
    data_files={
        "train": TRAIN_FILE,
        "validation": VALID_FILE,
    }
)

# Deterministic shuffle and selection
shared_train_raw = (
    raw_datasets["train"]
    .shuffle(seed=SEED)
    .select(range(min(MAX_TRAIN_EXAMPLES, len(raw_datasets["train"]))))
)

shared_valid_raw = (
    raw_datasets["validation"]
    .shuffle(seed=SEED)
    .select(range(min(MAX_VALID_EXAMPLES, len(raw_datasets["validation"]))))
)

print("Selected training examples  :", f"{len(shared_train_raw):,}")
print("Selected validation examples:", f"{len(shared_valid_raw):,}")

def tokenize_batch(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        max_length=BLOCK_SIZE,
        padding="max_length",
        return_special_tokens_mask=True,
    )

shared_train_dataset = shared_train_raw.map(
    tokenize_batch,
    batched=True,
    batch_size=1000,
    num_proc=2,
    remove_columns=["text"],
    desc="Tokenizing shared training data",
)

shared_valid_dataset = shared_valid_raw.map(
    tokenize_batch,
    batched=True,
    batch_size=1000,
    num_proc=2,
    remove_columns=["text"],
    desc="Tokenizing shared validation data",
)

print("\nTokenization complete.")
print("Training dataset  :", shared_train_dataset)
print("Validation dataset:", shared_valid_dataset)

print("\nTokenizer vocabulary:", tokenizer.vocab_size)
print("Sequence length      :", BLOCK_SIZE)

LOADING SHARED DATA FOR CONTROL AND MUSE BRANCHES


Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

Selected training examples  : 2,000,000
Selected validation examples: 100,000


Tokenizing shared training data (num_proc=2):   0%|          | 0/2000000 [00:00<?, ? examples/s]

Tokenizing shared validation data (num_proc=2):   0%|          | 0/100000 [00:00<?, ? examples/s]


Tokenization complete.
Training dataset  : Dataset({
    features: ['input_ids', 'attention_mask', 'special_tokens_mask'],
    num_rows: 2000000
})
Validation dataset: Dataset({
    features: ['input_ids', 'attention_mask', 'special_tokens_mask'],
    num_rows: 100000
})

Tokenizer vocabulary: 32000
Sequence length      : 128


In [ ]:
import os
import numpy as np
import torch
from tqdm import tqdm

print("=" * 90)
print("BUILDING MUSE TARGET LOOKUP TABLE")
print("=" * 90)

MUSE_TARGET_FILE = f"{MUSE_DIR}/babyroberta_muse_target_lookup.npz"

# ---------------------------------------------------------------------------
# Load the previously learned alignment and projection matrices
# ---------------------------------------------------------------------------

W_en_to_zh = np.load(ALIGNMENT_MATRIX).astype(np.float32)   # 300 x 300
P_muse_to_roberta = np.load(PROJECTION_MATRIX).astype(np.float32)  # 301 x 256

print("Alignment matrix:", W_en_to_zh.shape)
print("Projection matrix:", P_muse_to_roberta.shape)

# ---------------------------------------------------------------------------
# Read English words appearing in the bilingual MUSE dictionary
# ---------------------------------------------------------------------------

dictionary_english_words = set()

with open(MUSE_DICTIONARY, "r", encoding="utf-8", errors="ignore") as f:
    for line in f:
        parts = line.strip().split()

        if len(parts) >= 2:
            dictionary_english_words.add(parts[0].lower())

print("Unique English dictionary words:", f"{len(dictionary_english_words):,}")

# ---------------------------------------------------------------------------
# Identify BabyRoBERTa tokens that are complete dictionary words
# ---------------------------------------------------------------------------

vocab = tokenizer.get_vocab()
word_to_token_ids = {}

for token, token_id in vocab.items():
    clean_word = token.replace("Ġ", "").strip().lower()

    if not clean_word.isalpha():
        continue

    if clean_word not in dictionary_english_words:
        continue

    # Keep tokens only when the tokenizer can represent the word as one token.
    encoded_ids = tokenizer.encode(clean_word, add_special_tokens=False)

    is_standalone_token = (
        len(encoded_ids) == 1 and encoded_ids[0] == token_id
    )

    # Ġ marks a complete word following whitespace in byte-level BPE.
    is_word_start_token = token.startswith("Ġ")

    if is_standalone_token or is_word_start_token:
        word_to_token_ids.setdefault(clean_word, []).append(token_id)

wanted_words = set(word_to_token_ids)

print("Dictionary words matched to tokenizer:", f"{len(wanted_words):,}")

# ---------------------------------------------------------------------------
# Read only the required vectors from the large English MUSE file
# ---------------------------------------------------------------------------

english_vectors = {}
bad_lines = 0

with open(MUSE_ENGLISH_VECTORS, "r", encoding="utf-8", errors="ignore") as f:
    header = f.readline().strip()
    print("MUSE vector header:", header)

    for line in tqdm(f, desc="Reading selected MUSE vectors"):
        parts = line.rstrip().split()

        if len(parts) != 301:
            bad_lines += 1
            continue

        word = parts[0].lower()

        if word not in wanted_words:
            continue

        try:
            english_vectors[word] = np.asarray(
                parts[1:],
                dtype=np.float32
            )
        except ValueError:
            bad_lines += 1

print("Required vectors loaded:", f"{len(english_vectors):,}")
print("Bad/skipped vector lines:", f"{bad_lines:,}")

# ---------------------------------------------------------------------------
# Build a vocabulary-sized lookup:
# target_lookup[token_id] = projected MUSE target
# target_available[token_id] = True when a target exists
# ---------------------------------------------------------------------------

vocab_size = tokenizer.vocab_size
hidden_size = 256

target_lookup = np.zeros(
    (vocab_size, hidden_size),
    dtype=np.float32
)

target_available = np.zeros(
    vocab_size,
    dtype=np.bool_
)

matched_token_count = 0

for word, token_ids in word_to_token_ids.items():
    if word not in english_vectors:
        continue

    # English MUSE vector -> aligned Chinese MUSE space
    aligned_300 = english_vectors[word] @ W_en_to_zh
    aligned_300 /= np.linalg.norm(aligned_300) + 1e-8

    # Add bias term, then project 300D -> BabyRoBERTa 256D
    aligned_with_bias = np.concatenate(
        [aligned_300, np.array([1.0], dtype=np.float32)]
    )

    target_256 = aligned_with_bias @ P_muse_to_roberta
    target_256 /= np.linalg.norm(target_256) + 1e-8

    for token_id in token_ids:
        target_lookup[token_id] = target_256
        target_available[token_id] = True
        matched_token_count += 1

# Save for reuse
np.savez_compressed(
    MUSE_TARGET_FILE,
    target_lookup=target_lookup,
    target_available=target_available
)

# Convert to tensors for later training
muse_target_lookup = torch.tensor(
    target_lookup,
    dtype=torch.float32,
    device=device
)

muse_target_available = torch.tensor(
    target_available,
    dtype=torch.bool,
    device=device
)

print("\nMUSE lookup complete.")
print("Matched vocabulary tokens:", f"{matched_token_count:,}")
print("Lookup shape             :", muse_target_lookup.shape)
print("Available-target count   :", int(muse_target_available.sum().item()))
print("Saved to                 :", MUSE_TARGET_FILE)

if matched_token_count == 0:
    raise RuntimeError("No tokenizer tokens were matched to MUSE targets.")

BUILDING MUSE TARGET LOOKUP TABLE
Alignment matrix: (300, 300)
Projection matrix: (301, 256)
Unique English dictionary words: 32,495
Dictionary words matched to tokenizer: 8,845
MUSE vector header: 2519370 300


Reading selected MUSE vectors: 2519370it [01:56, 21631.86it/s]


Required vectors loaded: 8,845
Bad/skipped vector lines: 192

MUSE lookup complete.
Matched vocabulary tokens: 10,451
Lookup shape             : torch.Size([32000, 256])
Available-target count   : 10451
Saved to                 : /content/drive/MyDrive/BabyLM_Chinese_English_Translation/babyroberta_muse_alignment/babyroberta_muse_target_lookup.npz


Define the MUSE-loss Trainer

In [ ]:
import torch
import torch.nn.functional as F
from transformers import Trainer

print("=" * 90)
print("DEFINING MUSE-ALIGNMENT TRAINER")
print("=" * 90)

# Fixed alignment strength for the experiment
LAMBDA_MUSE = 0.05


class MuseAlignmentTrainer(Trainer):
    """
    Continued MLM training with an additional MUSE alignment objective.

    Total loss:
        MLM loss + LAMBDA_MUSE * MUSE alignment loss
    """

    def __init__(
        self,
        *args,
        muse_target_lookup=None,
        muse_target_available=None,
        lambda_muse=0.05,
        **kwargs
    ):
        super().__init__(*args, **kwargs)

        if muse_target_lookup is None or muse_target_available is None:
            raise ValueError("MUSE target lookup tensors must be provided.")

        self.muse_target_lookup = muse_target_lookup
        self.muse_target_available = muse_target_available
        self.lambda_muse = lambda_muse

        self.latest_mlm_loss = None
        self.latest_muse_loss = None
        self.latest_matched_tokens = 0

    def compute_loss(
        self,
        model,
        inputs,
        return_outputs=False,
        num_items_in_batch=None
    ):
        # -------------------------------------------------------------
        # 1. Normal masked-language-model loss
        # -------------------------------------------------------------
        outputs = model(**inputs)
        mlm_loss = outputs.loss

        input_ids = inputs["input_ids"]
        labels = inputs.get("labels")

        # Reconstruct original token IDs.
        # At masked locations, labels contain the original token.
        original_ids = input_ids.clone()

        if labels is not None:
            masked_positions = labels != -100
            original_ids[masked_positions] = labels[masked_positions]

        # -------------------------------------------------------------
        # 2. Find MUSE-covered tokens appearing in this batch
        # -------------------------------------------------------------
        flat_ids = original_ids.reshape(-1)

        valid_vocab_ids = (
            (flat_ids >= 0) &
            (flat_ids < self.muse_target_available.shape[0])
        )

        candidate_ids = flat_ids[valid_vocab_ids]

        if candidate_ids.numel() > 0:
            covered = self.muse_target_available[candidate_ids]
            matched_ids = candidate_ids[covered]
        else:
            matched_ids = candidate_ids

        # Use each vocabulary token once per batch so common words do not
        # dominate the alignment loss.
        matched_ids = torch.unique(matched_ids)

        # -------------------------------------------------------------
        # 3. MUSE cosine-alignment loss
        # -------------------------------------------------------------
        if matched_ids.numel() > 0:
            current_embeddings = (
                model.roberta.embeddings.word_embeddings(matched_ids)
            )

            target_embeddings = self.muse_target_lookup[matched_ids]

            current_embeddings = F.normalize(
                current_embeddings.float(),
                p=2,
                dim=-1
            )

            target_embeddings = F.normalize(
                target_embeddings.float(),
                p=2,
                dim=-1
            )

            cosine_similarity = (
                current_embeddings * target_embeddings
            ).sum(dim=-1)

            muse_loss = (1.0 - cosine_similarity).mean()

        else:
            # Keep graph/device compatibility when no matched token occurs.
            muse_loss = mlm_loss.new_zeros(())

        # -------------------------------------------------------------
        # 4. Combined loss
        # -------------------------------------------------------------
        total_loss = mlm_loss + self.lambda_muse * muse_loss

        self.latest_mlm_loss = float(mlm_loss.detach().cpu())
        self.latest_muse_loss = float(muse_loss.detach().cpu())
        self.latest_matched_tokens = int(matched_ids.numel())

        if return_outputs:
            return total_loss, outputs

        return total_loss


print("MUSE Trainer defined successfully.")
print("Total loss = MLM loss +", LAMBDA_MUSE, "× MUSE alignment loss")
print("Alignment metric: cosine distance")
print("MUSE targets available:", int(muse_target_available.sum().item()))

DEFINING MUSE-ALIGNMENT TRAINER
MUSE Trainer defined successfully.
Total loss = MLM loss + 0.05 × MUSE alignment loss
Alignment metric: cosine distance
MUSE targets available: 10451


Fix identical continued-training settings for both branches

In [ ]:
import os
import random
import numpy as np
import torch

from transformers import (
    DataCollatorForLanguageModeling,
    TrainingArguments,
    set_seed,
)

print("=" * 90)
print("FIXED CONTROL VS MUSE TRAINING SETTINGS")
print("=" * 90)

# =============================================================================
# IDENTICAL SETTINGS FOR BOTH BRANCHES
# =============================================================================

CONTINUED_TRAINING_STEPS = 10_000
TRAIN_BATCH_SIZE = 128
EVAL_BATCH_SIZE = 128

# Lower than from-scratch training because the base model is already trained.
CONTINUED_LEARNING_RATE = 5e-5

WEIGHT_DECAY = 0.01
WARMUP_STEPS = 500

LOGGING_STEPS = 250
EVAL_STEPS = 2_000
SAVE_STEPS = 2_000

MLM_PROBABILITY = 0.15

# Reset all random generators before each branch later.
def reset_experiment_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    set_seed(seed)

reset_experiment_seed()

# Same dynamic masking method for both branches
shared_data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=True,
    mlm_probability=MLM_PROBABILITY,
)

print("Continued-training steps :", f"{CONTINUED_TRAINING_STEPS:,}")
print("Training batch size      :", TRAIN_BATCH_SIZE)
print("Evaluation batch size    :", EVAL_BATCH_SIZE)
print("Learning rate            :", CONTINUED_LEARNING_RATE)
print("Weight decay             :", WEIGHT_DECAY)
print("Warmup steps             :", WARMUP_STEPS)
print("MLM probability          :", MLM_PROBABILITY)
print("Seed                     :", SEED)
print("MUSE loss weight         :", LAMBDA_MUSE)

print("\nBoth branches will:")
print("- start from the same English BabyRoBERTa checkpoint")
print("- use the same tokenized dataset")
print("- run for the same number of optimizer steps")
print("- use the same training hyperparameters")
print("- differ only by the additional MUSE alignment loss")

FIXED CONTROL VS MUSE TRAINING SETTINGS
Continued-training steps : 10,000
Training batch size      : 128
Evaluation batch size    : 128
Learning rate            : 5e-05
Weight decay             : 0.01
Warmup steps             : 500
MLM probability          : 0.15
Seed                     : 42
MUSE loss weight         : 0.05

Both branches will:
- start from the same English BabyRoBERTa checkpoint
- use the same tokenized dataset
- run for the same number of optimizer steps
- use the same training hyperparameters
- differ only by the additional MUSE alignment loss


Create the MLM-only control branch

In [ ]:
import os
from transformers import (
    RobertaForMaskedLM,
    TrainingArguments,
    Trainer,
)

print("=" * 90)
print("CREATING MLM-ONLY CONTROL BRANCH")
print("=" * 90)

# Reset randomness before constructing this branch
reset_experiment_seed(SEED)

# Load a fresh copy of the exact starting checkpoint
control_model = RobertaForMaskedLM.from_pretrained(
    BASE_ENGLISH_MODEL
)

control_model.to(device)

control_checkpoints_dir = (
    f"{CONTROL_OUTPUT_DIR}/checkpoints"
)

control_final_dir = (
    f"{CONTROL_OUTPUT_DIR}/final_model"
)

os.makedirs(CONTROL_OUTPUT_DIR, exist_ok=True)

control_training_args = TrainingArguments(
    output_dir=control_checkpoints_dir,

    do_train=True,
    do_eval=True,

    max_steps=CONTINUED_TRAINING_STEPS,

    per_device_train_batch_size=TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=EVAL_BATCH_SIZE,
    gradient_accumulation_steps=1,

    learning_rate=CONTINUED_LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    warmup_steps=WARMUP_STEPS,

    logging_strategy="steps",
    logging_steps=LOGGING_STEPS,

    eval_strategy="steps",
    eval_steps=EVAL_STEPS,

    save_strategy="steps",
    save_steps=SAVE_STEPS,
    save_total_limit=3,

    bf16=True,
    fp16=False,

    dataloader_num_workers=2,
    report_to="none",

    load_best_model_at_end=False,
    seed=SEED,
    data_seed=SEED,
)

control_trainer = Trainer(
    model=control_model,
    args=control_training_args,
    train_dataset=shared_train_dataset,
    eval_dataset=shared_valid_dataset,
    data_collator=shared_data_collator,
)

print("Control model loaded from:")
print(BASE_ENGLISH_MODEL)

print("\nControl checkpoint directory:")
print(control_checkpoints_dir)

print("\nControl final model directory:")
print(control_final_dir)

print("\nTraining steps :", control_training_args.max_steps)
print("Batch size     :", control_training_args.per_device_train_batch_size)
print("Learning rate  :", control_training_args.learning_rate)
print("Warmup steps   :", control_training_args.warmup_steps)
print("Seed           :", control_training_args.seed)

print("\nMLM-only control branch is ready.")

CREATING MLM-ONLY CONTROL BRANCH


Loading weights:   0%|          | 0/74 [00:00<?, ?it/s]

Control model loaded from:
/content/drive/MyDrive/BabyLM_Chinese_English_Translation/babyroberta_english_same_arch_95_5_final

Control checkpoint directory:
/content/drive/MyDrive/BabyLM_Chinese_English_Translation/babyroberta_english_continued_mlm_control/checkpoints

Control final model directory:
/content/drive/MyDrive/BabyLM_Chinese_English_Translation/babyroberta_english_continued_mlm_control/final_model

Training steps : 10000
Batch size     : 128
Learning rate  : 5e-05
Warmup steps   : 500
Seed           : 42

MLM-only control branch is ready.


In [ ]:
print("=" * 90)
print("TRAINING MLM-ONLY CONTROL BRANCH")
print("=" * 90)

reset_experiment_seed(SEED)

control_train_result = control_trainer.train()

control_trainer.save_model(control_final_dir)
tokenizer.save_pretrained(control_final_dir)

control_metrics = control_train_result.metrics
control_trainer.log_metrics("control_train", control_metrics)
control_trainer.save_metrics("control_train", control_metrics)
control_trainer.save_state()

print("\nControl training complete.")
print("Final control model saved to:")
print(control_final_dir)

TRAINING MLM-ONLY CONTROL BRANCH


Step,Training Loss,Validation Loss
2000,0.718272,0.635449
4000,0.718637,0.634432
6000,0.720578,0.630976
8000,0.716042,0.627980
10000,0.713301,0.628185


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

***** control_train metrics *****
  epoch                    =       0.64
  total_flos               =  2982656GF
  train_loss               =     0.7211
  train_runtime            = 0:07:07.47
  train_samples_per_second =   2994.316
  train_steps_per_second   =     23.393

Control training complete.
Final control model saved to:
/content/drive/MyDrive/BabyLM_Chinese_English_Translation/babyroberta_english_continued_mlm_control/final_model


In [ ]:
import os
from transformers import RobertaForMaskedLM, TrainingArguments

print("=" * 90)
print("CREATING MLM + MUSE BRANCH")
print("=" * 90)

# Reset randomness so this branch starts under the same conditions
reset_experiment_seed(SEED)

# Load a fresh copy of the ORIGINAL English model,
# not the continued-training control model
muse_model = RobertaForMaskedLM.from_pretrained(
    BASE_ENGLISH_MODEL
)

muse_model.to(device)

muse_checkpoints_dir = f"{MUSE_OUTPUT_DIR}/checkpoints"
muse_final_dir = f"{MUSE_OUTPUT_DIR}/final_model"

os.makedirs(MUSE_OUTPUT_DIR, exist_ok=True)

muse_training_args = TrainingArguments(
    output_dir=muse_checkpoints_dir,

    do_train=True,
    do_eval=True,

    max_steps=CONTINUED_TRAINING_STEPS,

    per_device_train_batch_size=TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=EVAL_BATCH_SIZE,
    gradient_accumulation_steps=1,

    learning_rate=CONTINUED_LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    warmup_steps=WARMUP_STEPS,

    logging_strategy="steps",
    logging_steps=LOGGING_STEPS,

    eval_strategy="steps",
    eval_steps=EVAL_STEPS,

    save_strategy="steps",
    save_steps=SAVE_STEPS,
    save_total_limit=3,

    bf16=True,
    fp16=False,

    dataloader_num_workers=2,
    report_to="none",

    load_best_model_at_end=False,
    seed=SEED,
    data_seed=SEED,
)

muse_trainer = MuseAlignmentTrainer(
    model=muse_model,
    args=muse_training_args,
    train_dataset=shared_train_dataset,
    eval_dataset=shared_valid_dataset,
    data_collator=shared_data_collator,

    muse_target_lookup=muse_target_lookup,
    muse_target_available=muse_target_available,
    lambda_muse=LAMBDA_MUSE,
)

print("MUSE model loaded from:")
print(BASE_ENGLISH_MODEL)

print("\nMUSE checkpoint directory:")
print(muse_checkpoints_dir)

print("\nMUSE final model directory:")
print(muse_final_dir)

print("\nTraining steps :", muse_training_args.max_steps)
print("Batch size     :", muse_training_args.per_device_train_batch_size)
print("Learning rate  :", muse_training_args.learning_rate)
print("Warmup steps   :", muse_training_args.warmup_steps)
print("MUSE lambda    :", LAMBDA_MUSE)
print("Seed           :", muse_training_args.seed)

print("\nMLM + MUSE branch is ready.")

CREATING MLM + MUSE BRANCH


Loading weights:   0%|          | 0/74 [00:00<?, ?it/s]

MUSE model loaded from:
/content/drive/MyDrive/BabyLM_Chinese_English_Translation/babyroberta_english_same_arch_95_5_final

MUSE checkpoint directory:
/content/drive/MyDrive/BabyLM_Chinese_English_Translation/babyroberta_english_continued_mlm_muse/checkpoints

MUSE final model directory:
/content/drive/MyDrive/BabyLM_Chinese_English_Translation/babyroberta_english_continued_mlm_muse/final_model

Training steps : 10000
Batch size     : 128
Learning rate  : 5e-05
Warmup steps   : 500
MUSE lambda    : 0.05
Seed           : 42

MLM + MUSE branch is ready.


In [ ]:
print("=" * 90)
print("TRAINING MLM + MUSE BRANCH")
print("=" * 90)

# Reset seed immediately before training
reset_experiment_seed(SEED)

muse_train_result = muse_trainer.train()

# Save final MUSE model and tokenizer
muse_trainer.save_model(muse_final_dir)
tokenizer.save_pretrained(muse_final_dir)

# Save training metrics and trainer state
muse_metrics = muse_train_result.metrics
muse_trainer.log_metrics("muse_train", muse_metrics)
muse_trainer.save_metrics("muse_train", muse_metrics)
muse_trainer.save_state()

print("\nMUSE training complete.")
print("Final MUSE model saved to:")
print(muse_final_dir)

print("\nLast recorded losses:")
print("MLM loss          :", muse_trainer.latest_mlm_loss)
print("MUSE alignment loss:", muse_trainer.latest_muse_loss)
print("Matched MUSE tokens:", muse_trainer.latest_matched_tokens)

TRAINING MLM + MUSE BRANCH


Step,Training Loss,Validation Loss
2000,0.718265,0.635414
4000,0.718591,0.634451
6000,0.720562,0.630953
8000,0.716051,0.627950
10000,0.713316,0.628158


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

***** muse_train metrics *****
  epoch                    =       0.64
  total_flos               =  2982656GF
  train_loss               =     0.7211
  train_runtime            = 0:07:24.86
  train_samples_per_second =   2877.266
  train_steps_per_second   =     22.479

MUSE training complete.
Final MUSE model saved to:
/content/drive/MyDrive/BabyLM_Chinese_English_Translation/babyroberta_english_continued_mlm_muse/final_model

Last recorded losses:
MLM loss          : 0.5692014098167419
MUSE alignment loss: 0.0
Matched MUSE tokens: 0


In [ ]:
import torch

print("=" * 90)
print("DIAGNOSING MUSE TOKEN MATCHING")
print("=" * 90)

# Get one real batch from the trainer
train_loader = muse_trainer.get_train_dataloader()
batch = next(iter(train_loader))

batch = {
    key: value.to(device) if torch.is_tensor(value) else value
    for key, value in batch.items()
}

input_ids = batch["input_ids"]
labels = batch["labels"]

# Reconstruct the original unmasked tokens
original_ids = input_ids.clone()
masked_positions = labels != -100
original_ids[masked_positions] = labels[masked_positions]

flat_ids = original_ids.reshape(-1)

valid_ids = flat_ids[
    (flat_ids >= 0) &
    (flat_ids < muse_target_available.shape[0])
]

covered_mask = muse_target_available[valid_ids]
matched_ids = valid_ids[covered_mask]
unique_matched_ids = torch.unique(matched_ids)

print("Batch shape                    :", input_ids.shape)
print("Total tokens in batch          :", flat_ids.numel())
print("Valid vocabulary tokens        :", valid_ids.numel())
print("Matched MUSE token occurrences :", matched_ids.numel())
print("Unique matched MUSE tokens     :", unique_matched_ids.numel())

print("\nFirst matched tokens:")

for token_id in unique_matched_ids[:30].tolist():
    print(
        token_id,
        repr(tokenizer.convert_ids_to_tokens(token_id))
    )

if unique_matched_ids.numel() == 0:
    print("\nERROR: The lookup table does not match tokens in the training batches.")
else:
    print("\nToken matching works. The problem is inside the custom Trainer logic.")

DIAGNOSING MUSE TOKEN MATCHING
Batch shape                    : torch.Size([128, 128])
Total tokens in batch          : 16384
Valid vocabulary tokens        : 16384
Matched MUSE token occurrences : 0
Unique matched MUSE tokens     : 0

First matched tokens:

ERROR: The lookup table does not match tokens in the training batches.


In [ ]:
import torch

print("=" * 90)
print("CHECKING TOKEN-ID COMPATIBILITY")
print("=" * 90)

print("Available MUSE targets:", int(muse_target_available.sum().item()))
print("Lookup length:", len(muse_target_available))
print("Tokenizer vocab size:", tokenizer.vocab_size)

# Get one real batch
train_loader = muse_trainer.get_train_dataloader()
batch = next(iter(train_loader))

input_ids = batch["input_ids"].to(device)
labels = batch["labels"].to(device)

# Recover original tokens before MLM masking
original_ids = input_ids.clone()
masked_positions = labels != -100
original_ids[masked_positions] = labels[masked_positions]

batch_unique_ids = torch.unique(original_ids.reshape(-1))
available_ids = torch.where(muse_target_available)[0]

intersection = batch_unique_ids[
    muse_target_available[batch_unique_ids]
]

print("\nUnique token IDs in batch :", batch_unique_ids.numel())
print("Available MUSE token IDs :", available_ids.numel())
print("Intersection             :", intersection.numel())

print("\nFirst 40 actual batch tokens:")
for token_id in batch_unique_ids[:40].tolist():
    print(
        token_id,
        repr(tokenizer.convert_ids_to_tokens(token_id))
    )

print("\nFirst 40 MUSE-available tokens:")
for token_id in available_ids[:40].tolist():
    print(
        token_id,
        repr(tokenizer.convert_ids_to_tokens(token_id))
    )

# Also inspect frequent non-padding tokens from the batch
flat_ids = original_ids.reshape(-1)
non_special = flat_ids[
    ~torch.isin(
        flat_ids,
        torch.tensor(
            [
                tokenizer.pad_token_id,
                tokenizer.bos_token_id,
                tokenizer.eos_token_id,
                tokenizer.mask_token_id,
            ],
            device=device,
        )
    )
]

values, counts = torch.unique(non_special, return_counts=True)
top_order = torch.argsort(counts, descending=True)[:40]

print("\nMost frequent real batch tokens:")
for idx in top_order.tolist():
    token_id = values[idx].item()
    print(
        token_id,
        repr(tokenizer.convert_ids_to_tokens(token_id)),
        "count =", counts[idx].item(),
        "MUSE =", bool(muse_target_available[token_id].item()),
    )

CHECKING TOKEN-ID COMPATIBILITY
Available MUSE targets: 10451
Lookup length: 32000
Tokenizer vocab size: 32000

Unique token IDs in batch : 69
Available MUSE token IDs : 10451
Intersection             : 0

First 40 actual batch tokens:
1 '<pad>'
5 '!'
6 '"'
9 '%'
11 "'"
14 '*'
16 ','
17 '-'
18 '.'
20 '0'
21 '1'
22 '2'
25 '5'
26 '6'
27 '7'
30 ':'
33 '='
35 '?'
37 'A'
38 'B'
39 'C'
40 'D'
41 'E'
42 'F'
43 'G'
44 'H'
45 'I'
47 'K'
48 'L'
49 'M'
50 'N'
51 'O'
52 'P'
54 'R'
55 'S'
56 'T'
58 'V'
59 'W'
61 'Y'
63 '['

First 40 MUSE-available tokens:
288 'Ġyou'
306 'Ġthis'
322 'Ġone'
336 'Ġjust'
351 'Ġall'
358 'Ġalso'
359 'Ġsay'
368 'Ġthen'
376 'Ġand'
380 'Ġthat'
386 'Ġtime'
396 'Ġwant'
408 'Ġstill'
416 'Ġvery'
420 'Ġhow'
423 'Ġwill'
435 'Ġsee'
441 'Ġcon'
442 'Ġbut'
450 'Ġcom'
456 'Ġlike'
464 'Ġout'
473 'Ġbig'
480 'Ġmat'
486 'Ġpeople'
488 'Ġman'
489 'Ġagain'
493 'Ġlittle'
503 'Ġtwo'
504 'Ġnow'
505 'Ġlet'
506 'Ġonly'
511 'Ġmust'
512 'Ġmore'
519 'Ġbec'
523 'Ġalready'
528 'Ġday'
535 'Ġyour'
536 '

In [ ]:
import numpy as np
import torch

print("=" * 90)
print("REBUILDING MUSE LOOKUP WITH ACTUAL TOKEN IDS")
print("=" * 90)

vocab_size = tokenizer.vocab_size
hidden_size = 256

target_lookup_fixed = np.zeros(
    (vocab_size, hidden_size),
    dtype=np.float32
)

target_available_fixed = np.zeros(
    vocab_size,
    dtype=np.bool_
)

matched_words = 0
matched_token_ids = 0

# Iterate through real token IDs instead of tokenizer.get_vocab()
for token_id in range(vocab_size):

    token = tokenizer.convert_ids_to_tokens(token_id)

    if token is None:
        continue

    # RoBERTa Ġ means the token begins a new word
    clean_word = token.replace("Ġ", "").strip().lower()

    if not clean_word.isalpha():
        continue

    if clean_word not in english_vectors:
        continue

    # Only use full-word tokens
    encoded_with_space = tokenizer.encode(
        " " + clean_word,
        add_special_tokens=False
    )

    encoded_without_space = tokenizer.encode(
        clean_word,
        add_special_tokens=False
    )

    is_full_word = (
        (len(encoded_with_space) == 1 and encoded_with_space[0] == token_id)
        or
        (len(encoded_without_space) == 1 and encoded_without_space[0] == token_id)
    )

    if not is_full_word:
        continue

    # English MUSE -> Chinese-aligned MUSE
    aligned_300 = english_vectors[clean_word] @ W_en_to_zh
    aligned_300 /= np.linalg.norm(aligned_300) + 1e-8

    # 300D MUSE -> 256D BabyRoBERTa
    aligned_with_bias = np.concatenate(
        [aligned_300, np.array([1.0], dtype=np.float32)]
    )

    target_256 = aligned_with_bias @ P_muse_to_roberta
    target_256 /= np.linalg.norm(target_256) + 1e-8

    target_lookup_fixed[token_id] = target_256
    target_available_fixed[token_id] = True

    matched_words += 1
    matched_token_ids += 1

# Replace the incorrect lookup tensors
muse_target_lookup = torch.tensor(
    target_lookup_fixed,
    dtype=torch.float32,
    device=device
)

muse_target_available = torch.tensor(
    target_available_fixed,
    dtype=torch.bool,
    device=device
)

FIXED_MUSE_TARGET_FILE = (
    f"{MUSE_DIR}/babyroberta_muse_target_lookup_fixed.npz"
)

np.savez_compressed(
    FIXED_MUSE_TARGET_FILE,
    target_lookup=target_lookup_fixed,
    target_available=target_available_fixed
)

print("Matched token IDs:", matched_token_ids)
print("Available targets:", int(muse_target_available.sum().item()))
print("Saved fixed lookup:", FIXED_MUSE_TARGET_FILE)

print("\nFirst 30 available tokens:")
available_ids = torch.where(muse_target_available)[0]

for token_id in available_ids[:30].tolist():
    print(
        token_id,
        repr(tokenizer.convert_ids_to_tokens(token_id))
    )

REBUILDING MUSE LOOKUP WITH ACTUAL TOKEN IDS
Matched token IDs: 0
Available targets: 0
Saved fixed lookup: /content/drive/MyDrive/BabyLM_Chinese_English_Translation/babyroberta_muse_alignment/babyroberta_muse_target_lookup_fixed.npz

First 30 available tokens:


In [ ]:
import torch

print("=" * 90)
print("CHECKING FAST-TOKENIZER BACKEND ID MAPPING")
print("=" * 90)

backend = tokenizer.backend_tokenizer

# Get one real batch
batch = next(iter(muse_trainer.get_train_dataloader()))

input_ids = batch["input_ids"]
labels = batch["labels"]

# Recover original unmasked IDs
original_ids = input_ids.clone()
masked_positions = labels != -100
original_ids[masked_positions] = labels[masked_positions]

unique_batch_ids = torch.unique(original_ids.reshape(-1))

print("Tokenizer vocab size:", tokenizer.vocab_size)
print("Unique IDs in batch:", unique_batch_ids.numel())

print("\nComparing token mappings for real batch IDs:")
print(f"{'ID':>6} | {'Transformers mapping':<25} | {'Backend mapping':<25}")
print("-" * 70)

for token_id in unique_batch_ids[:60].tolist():
    transformers_token = tokenizer.convert_ids_to_tokens(token_id)
    backend_token = backend.id_to_token(token_id)

    print(
        f"{token_id:6d} | "
        f"{repr(transformers_token):<25} | "
        f"{repr(backend_token):<25}"
    )

print("\nEncoding checks:")

test_words = ["people", "man", "again", "little", "inside", "year"]

for word in test_words:
    encoded = tokenizer(
        " " + word,
        add_special_tokens=False
    )["input_ids"]

    print(f"\nWord: {word}")
    print("IDs:", encoded)

    for token_id in encoded:
        print(
            " ",
            token_id,
            "Transformers:",
            repr(tokenizer.convert_ids_to_tokens(token_id)),
            "| Backend:",
            repr(backend.id_to_token(token_id))
        )

CHECKING FAST-TOKENIZER BACKEND ID MAPPING
Tokenizer vocab size: 32000
Unique IDs in batch: 69

Comparing token mappings for real batch IDs:
    ID | Transformers mapping      | Backend mapping          
----------------------------------------------------------------------
     1 | '<pad>'                   | '<pad>'                  
     5 | '!'                       | '!'                      
     6 | '"'                       | '"'                      
     9 | '%'                       | '%'                      
    11 | "'"                       | "'"                      
    14 | '*'                       | '*'                      
    16 | ','                       | ','                      
    17 | '-'                       | '-'                      
    18 | '.'                       | '.'                      
    20 | '0'                       | '0'                      
    21 | '1'                       | '1'                      
    22 | '2'                    

In [ ]:
import os
import numpy as np
import torch

print("=" * 90)
print("BUILDING WORD-LEVEL MUSE TRAINING BANK")
print("=" * 90)

# We no longer use the old 300D -> 256D projection.
# A new trainable projection will be learned during MUSE training.

muse_words = []
muse_aligned_targets = []

seen_words = set()

for word in sorted(dictionary_english_words):

    word = word.strip().lower()

    if word in seen_words:
        continue

    if not word.isalpha():
        continue

    if word not in english_vectors:
        continue

    # Avoid unusually long dictionary entries
    token_ids = tokenizer.encode(
        word,
        add_special_tokens=False
    )

    if len(token_ids) == 0 or len(token_ids) > 24:
        continue

    # English MUSE vector mapped toward Chinese MUSE space
    aligned_vector = english_vectors[word] @ W_en_to_zh
    aligned_vector = aligned_vector / (
        np.linalg.norm(aligned_vector) + 1e-8
    )

    muse_words.append(word)
    muse_aligned_targets.append(
        aligned_vector.astype(np.float32)
    )

    seen_words.add(word)

muse_aligned_targets = np.stack(
    muse_aligned_targets
).astype(np.float32)

WORD_MUSE_BANK_FILE = (
    f"{MUSE_DIR}/babyroberta_word_level_muse_bank.npz"
)

np.savez_compressed(
    WORD_MUSE_BANK_FILE,
    words=np.asarray(muse_words),
    targets=muse_aligned_targets
)

print("Usable MUSE words :", f"{len(muse_words):,}")
print("Target shape      :", muse_aligned_targets.shape)
print("Saved to          :", WORD_MUSE_BANK_FILE)

print("\nEncoding examples:")

for word in muse_words[:10]:
    ids = tokenizer.encode(
        word,
        add_special_tokens=False
    )

    tokens = tokenizer.convert_ids_to_tokens(ids)

    print(
        f"{word:15s}",
        "→",
        tokens
    )

if len(muse_words) == 0:
    raise RuntimeError("No usable MUSE words were created.")

BUILDING WORD-LEVEL MUSE TRAINING BANK
Usable MUSE words : 8,845
Target shape      : (8845, 300)
Saved to          : /content/drive/MyDrive/BabyLM_Chinese_English_Translation/babyroberta_muse_alignment/babyroberta_word_level_muse_bank.npz

Encoding examples:
abacus          → ['a', 'b', 'a', 'c', 'u', 's']
abalone         → ['a', 'b', 'a', 'l', 'o', 'n', 'e']
abandoned       → ['a', 'b', 'a', 'n', 'd', 'o', 'n', 'e', 'd']
abbot           → ['a', 'b', 'b', 'o', 't']
abbreviation    → ['a', 'b', 'b', 'r', 'e', 'v', 'i', 'a', 't', 'i', 'o', 'n']
abby            → ['a', 'b', 'b', 'y']
abc             → ['a', 'b', 'c']
abdomen         → ['a', 'b', 'd', 'o', 'm', 'e', 'n']
abdominal       → ['a', 'b', 'd', 'o', 'm', 'i', 'n', 'a', 'l']
abe             → ['a', 'b', 'e']


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import Trainer

print("=" * 90)
print("PREPARING CORRECTED WORD-LEVEL MUSE TRAINER")
print("=" * 90)

# =============================================================================
# FIXED MUSE SETTINGS
# =============================================================================

LAMBDA_MUSE = 0.05
MUSE_WORD_BATCH_SIZE = 128
MUSE_WORD_MAX_LENGTH = 26  # up to 24 characters + special tokens

# =============================================================================
# PRE-TOKENIZE ALL MUSE WORDS ONCE
# =============================================================================

muse_word_encodings = tokenizer(
    muse_words,
    add_special_tokens=True,
    truncation=True,
    max_length=MUSE_WORD_MAX_LENGTH,
    padding="max_length",
    return_tensors="pt",
)

muse_word_input_ids = muse_word_encodings["input_ids"]
muse_word_attention_mask = muse_word_encodings["attention_mask"]

muse_word_targets = torch.tensor(
    muse_aligned_targets,
    dtype=torch.float32
)

print("Word input IDs shape :", muse_word_input_ids.shape)
print("Word attention shape :", muse_word_attention_mask.shape)
print("MUSE targets shape   :", muse_word_targets.shape)

# =============================================================================
# CORRECTED TRAINER
# =============================================================================

class WordLevelMuseTrainer(Trainer):
    """
    Continued MLM training with a word-level MUSE alignment objective.

    Total loss:
        MLM loss + lambda_muse * word-level MUSE cosine loss
    """

    def __init__(
        self,
        *args,
        muse_word_input_ids=None,
        muse_word_attention_mask=None,
        muse_word_targets=None,
        lambda_muse=0.05,
        muse_word_batch_size=128,
        **kwargs
    ):
        super().__init__(*args, **kwargs)

        self.muse_word_input_ids = muse_word_input_ids
        self.muse_word_attention_mask = muse_word_attention_mask
        self.muse_word_targets = muse_word_targets

        self.lambda_muse = lambda_muse
        self.muse_word_batch_size = muse_word_batch_size

        self.latest_mlm_loss = None
        self.latest_muse_loss = None

    def compute_loss(
        self,
        model,
        inputs,
        return_outputs=False,
        num_items_in_batch=None
    ):
        # ---------------------------------------------------------------------
        # 1. Normal MLM loss on English BabyLM text
        # ---------------------------------------------------------------------
        outputs = model(**inputs)
        mlm_loss = outputs.loss

        model_device = next(model.parameters()).device

        # ---------------------------------------------------------------------
        # 2. Sample a batch of MUSE dictionary words
        # ---------------------------------------------------------------------
        total_words = self.muse_word_input_ids.shape[0]

        sampled_indices = torch.randint(
            low=0,
            high=total_words,
            size=(self.muse_word_batch_size,),
        )

        word_input_ids = self.muse_word_input_ids[
            sampled_indices
        ].to(model_device)

        word_attention_mask = self.muse_word_attention_mask[
            sampled_indices
        ].to(model_device)

        target_vectors = self.muse_word_targets[
            sampled_indices
        ].to(model_device)

        # ---------------------------------------------------------------------
        # 3. Produce contextual word representations using BabyRoBERTa
        # ---------------------------------------------------------------------
        word_outputs = model.roberta(
            input_ids=word_input_ids,
            attention_mask=word_attention_mask,
            return_dict=True,
        )

        hidden_states = word_outputs.last_hidden_state

        # Exclude padding and special tokens from the word average
        valid_token_mask = word_attention_mask.bool()

        for special_id in [
            tokenizer.pad_token_id,
            tokenizer.bos_token_id,
            tokenizer.eos_token_id,
            tokenizer.mask_token_id,
        ]:
            if special_id is not None:
                valid_token_mask &= word_input_ids.ne(special_id)

        valid_token_mask_float = valid_token_mask.unsqueeze(-1).float()

        summed_hidden = (
            hidden_states.float() * valid_token_mask_float
        ).sum(dim=1)

        token_counts = valid_token_mask_float.sum(dim=1).clamp(min=1.0)

        word_representations = summed_hidden / token_counts

        # ---------------------------------------------------------------------
        # 4. Learnable projection: BabyRoBERTa 256D -> MUSE 300D
        # ---------------------------------------------------------------------
        projected_words = model.muse_projection(word_representations)

        projected_words = F.normalize(
            projected_words,
            p=2,
            dim=-1
        )

        target_vectors = F.normalize(
            target_vectors.float(),
            p=2,
            dim=-1
        )

        cosine_similarity = (
            projected_words * target_vectors
        ).sum(dim=-1)

        muse_loss = (1.0 - cosine_similarity).mean()

        # ---------------------------------------------------------------------
        # 5. Combined training objective
        # ---------------------------------------------------------------------
        total_loss = mlm_loss + self.lambda_muse * muse_loss

        self.latest_mlm_loss = float(mlm_loss.detach().cpu())
        self.latest_muse_loss = float(muse_loss.detach().cpu())

        if return_outputs:
            return total_loss, outputs

        return total_loss


print("\nCorrected trainer defined successfully.")
print("MUSE unit            : complete English word")
print("Word representation  : average contextual hidden state")
print("Projection           : trainable 256D → 300D")
print("MUSE word batch size :", MUSE_WORD_BATCH_SIZE)
print("MUSE lambda          :", LAMBDA_MUSE)
print(
    "Total loss          : MLM loss + "
    f"{LAMBDA_MUSE} × word-level MUSE loss"
)

PREPARING CORRECTED WORD-LEVEL MUSE TRAINER
Word input IDs shape : torch.Size([8845, 26])
Word attention shape : torch.Size([8845, 26])
MUSE targets shape   : torch.Size([8845, 300])

Corrected trainer defined successfully.
MUSE unit            : complete English word
Word representation  : average contextual hidden state
Projection           : trainable 256D → 300D
MUSE word batch size : 128
MUSE lambda          : 0.05
Total loss          : MLM loss + 0.05 × word-level MUSE loss


In [ ]:
import os
import torch
import torch.nn as nn
from transformers import RobertaForMaskedLM, TrainingArguments

print("=" * 90)
print("CREATING CORRECTED WORD-LEVEL MUSE BRANCH")
print("=" * 90)

# Start again from the ORIGINAL English BabyRoBERTa checkpoint
reset_experiment_seed(SEED)

corrected_muse_model = RobertaForMaskedLM.from_pretrained(
    BASE_ENGLISH_MODEL
)

# Add the trainable BabyRoBERTa 256D -> MUSE 300D projection
corrected_muse_model.muse_projection = nn.Linear(
    corrected_muse_model.config.hidden_size,  # 256
    300,
    bias=False,
)

corrected_muse_model.to(device)

# Use a new output folder so the invalid MUSE run is never mixed with this one
CORRECTED_MUSE_OUTPUT_DIR = (
    f"{PROJECT_DIR}/babyroberta_english_continued_word_muse"
)

corrected_muse_checkpoints_dir = (
    f"{CORRECTED_MUSE_OUTPUT_DIR}/checkpoints"
)

corrected_muse_final_dir = (
    f"{CORRECTED_MUSE_OUTPUT_DIR}/final_model"
)

os.makedirs(CORRECTED_MUSE_OUTPUT_DIR, exist_ok=True)

corrected_muse_training_args = TrainingArguments(
    output_dir=corrected_muse_checkpoints_dir,

    do_train=True,
    do_eval=True,

    max_steps=CONTINUED_TRAINING_STEPS,

    per_device_train_batch_size=TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=EVAL_BATCH_SIZE,
    gradient_accumulation_steps=1,

    learning_rate=CONTINUED_LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    warmup_steps=WARMUP_STEPS,

    logging_strategy="steps",
    logging_steps=LOGGING_STEPS,

    eval_strategy="steps",
    eval_steps=EVAL_STEPS,

    save_strategy="steps",
    save_steps=SAVE_STEPS,
    save_total_limit=3,

    bf16=True,
    fp16=False,

    dataloader_num_workers=2,
    report_to="none",

    load_best_model_at_end=False,
    seed=SEED,
    data_seed=SEED,
)

corrected_muse_trainer = WordLevelMuseTrainer(
    model=corrected_muse_model,
    args=corrected_muse_training_args,

    train_dataset=shared_train_dataset,
    eval_dataset=shared_valid_dataset,
    data_collator=shared_data_collator,

    muse_word_input_ids=muse_word_input_ids,
    muse_word_attention_mask=muse_word_attention_mask,
    muse_word_targets=muse_word_targets,

    lambda_muse=LAMBDA_MUSE,
    muse_word_batch_size=MUSE_WORD_BATCH_SIZE,
)

# =============================================================================
# SANITY CHECK — calculate one batch loss before starting the full run
# =============================================================================

test_batch = next(iter(corrected_muse_trainer.get_train_dataloader()))

test_batch = {
    key: value.to(device) if torch.is_tensor(value) else value
    for key, value in test_batch.items()
}

corrected_muse_model.train()

with torch.no_grad():
    test_total_loss = corrected_muse_trainer.compute_loss(
        corrected_muse_model,
        test_batch
    )

print("Starting checkpoint:")
print(BASE_ENGLISH_MODEL)

print("\nProjection layer:")
print(corrected_muse_model.muse_projection)

print("\nSanity-check losses:")
print("Total loss         :", float(test_total_loss.cpu()))
print("MLM loss           :", corrected_muse_trainer.latest_mlm_loss)
print("MUSE alignment loss:", corrected_muse_trainer.latest_muse_loss)

print("\nTraining steps :", corrected_muse_training_args.max_steps)
print("Batch size     :", corrected_muse_training_args.per_device_train_batch_size)
print("Learning rate  :", corrected_muse_training_args.learning_rate)
print("MUSE lambda    :", LAMBDA_MUSE)

print("\nCorrected final model will be saved to:")
print(corrected_muse_final_dir)

if corrected_muse_trainer.latest_muse_loss is None:
    raise RuntimeError("MUSE loss was not calculated.")

if corrected_muse_trainer.latest_muse_loss <= 0:
    raise RuntimeError(
        "MUSE loss is zero. Do not begin training until this is fixed."
    )

print("\nSUCCESS: Non-zero word-level MUSE loss confirmed.")
print("The corrected MUSE branch is ready for training.")

CREATING CORRECTED WORD-LEVEL MUSE BRANCH


Loading weights:   0%|          | 0/74 [00:00<?, ?it/s]

Starting checkpoint:
/content/drive/MyDrive/BabyLM_Chinese_English_Translation/babyroberta_english_same_arch_95_5_final

Projection layer:
Linear(in_features=256, out_features=300, bias=False)

Sanity-check losses:
Total loss         : 0.6180474162101746
MLM loss           : 0.5680830478668213
MUSE alignment loss: 0.9992876648902893

Training steps : 10000
Batch size     : 128
Learning rate  : 5e-05
MUSE lambda    : 0.05

Corrected final model will be saved to:
/content/drive/MyDrive/BabyLM_Chinese_English_Translation/babyroberta_english_continued_word_muse/final_model

SUCCESS: Non-zero word-level MUSE loss confirmed.
The corrected MUSE branch is ready for training.


In [ ]:
print("=" * 90)
print("TRAINING CORRECTED WORD-LEVEL MUSE BRANCH")
print("=" * 90)

reset_experiment_seed(SEED)

corrected_muse_train_result = corrected_muse_trainer.train()

# Save final model and tokenizer
corrected_muse_trainer.save_model(corrected_muse_final_dir)
tokenizer.save_pretrained(corrected_muse_final_dir)

# Save the custom projection layer separately
projection_save_path = (
    f"{corrected_muse_final_dir}/muse_projection.pt"
)

torch.save(
    corrected_muse_model.muse_projection.state_dict(),
    projection_save_path
)

# Save training metrics and state
corrected_muse_metrics = corrected_muse_train_result.metrics

corrected_muse_trainer.log_metrics(
    "corrected_muse_train",
    corrected_muse_metrics
)

corrected_muse_trainer.save_metrics(
    "corrected_muse_train",
    corrected_muse_metrics
)

corrected_muse_trainer.save_state()

print("\nCorrected MUSE training complete.")

print("\nFinal model saved to:")
print(corrected_muse_final_dir)

print("\nMUSE projection saved to:")
print(projection_save_path)

print("\nLast recorded losses:")
print("MLM loss            :", corrected_muse_trainer.latest_mlm_loss)
print("MUSE alignment loss :", corrected_muse_trainer.latest_muse_loss)

TRAINING CORRECTED WORD-LEVEL MUSE BRANCH


Step,Training Loss,Validation Loss
2000,0.745864,0.669142
4000,0.746768,0.663265
6000,0.745728,0.660322
8000,0.744980,0.655605
10000,0.742928,0.656216


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

***** corrected_muse_train metrics *****
  epoch                    =       0.64
  total_flos               =  3052968GF
  train_loss               =     0.7507
  train_runtime            = 0:09:35.09
  train_samples_per_second =   2225.712
  train_steps_per_second   =     17.388

Corrected MUSE training complete.

Final model saved to:
/content/drive/MyDrive/BabyLM_Chinese_English_Translation/babyroberta_english_continued_word_muse/final_model

MUSE projection saved to:
/content/drive/MyDrive/BabyLM_Chinese_English_Translation/babyroberta_english_continued_word_muse/final_model/muse_projection.pt

Last recorded losses:
MLM loss            : 0.826576828956604
MUSE alignment loss : 0.560677170753479


In [ ]:
import math
import torch
from transformers import RobertaForMaskedLM, Trainer, TrainingArguments

print("=" * 90)
print("PURE MLM VALIDATION COMPARISON")
print("=" * 90)

CONTROL_MODEL_DIR = (
    f"{PROJECT_DIR}/babyroberta_english_continued_mlm_control/final_model"
)

MUSE_MODEL_DIR = (
    f"{PROJECT_DIR}/babyroberta_english_continued_word_muse/final_model"
)

evaluation_args = TrainingArguments(
    output_dir=f"{PROJECT_DIR}/temporary_mlm_evaluation",
    per_device_eval_batch_size=128,
    bf16=True,
    report_to="none",
    dataloader_num_workers=2,
)

results = {}

for model_name, model_path in {
    "MLM_Control": CONTROL_MODEL_DIR,
    "MLM_MUSE": MUSE_MODEL_DIR,
}.items():

    print("\nEvaluating:", model_name)

    eval_model = RobertaForMaskedLM.from_pretrained(model_path).to(device)

    eval_trainer = Trainer(
        model=eval_model,
        args=evaluation_args,
        eval_dataset=shared_valid_dataset,
        data_collator=shared_data_collator,
    )

    metrics = eval_trainer.evaluate()

    eval_loss = metrics["eval_loss"]
    approximate_mlm_ppl = math.exp(eval_loss)

    results[model_name] = {
        "MLM_validation_loss": eval_loss,
        "Approx_MLM_perplexity": approximate_mlm_ppl,
    }

    print("Pure MLM validation loss:", round(eval_loss, 6))
    print("Approximate MLM perplexity:", round(approximate_mlm_ppl, 4))

    del eval_model
    torch.cuda.empty_cache()

print("\n" + "=" * 90)
print("FINAL FAIR COMPARISON")
print("=" * 90)

for model_name, values in results.items():
    print(
        f"{model_name:15s} | "
        f"Loss: {values['MLM_validation_loss']:.6f} | "
        f"PPL: {values['Approx_MLM_perplexity']:.4f}"
    )

PURE MLM VALIDATION COMPARISON

Evaluating: MLM_Control


Loading weights:   0%|          | 0/74 [00:00<?, ?it/s]

Training Loss,Validation Loss,Step
No log,0.624643,0


Pure MLM validation loss: 0.624643
Approximate MLM perplexity: 1.8676

Evaluating: MLM_MUSE


Loading weights:   0%|          | 0/74 [00:00<?, ?it/s]

[transformers] RobertaForMaskedLM LOAD REPORT from: /content/drive/MyDrive/BabyLM_Chinese_English_Translation/babyroberta_english_continued_word_muse/final_model
Key                    | Status     |  | 
-----------------------+------------+--+-
muse_projection.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Training Loss,Validation Loss,Step
No log,0.625057,0


Pure MLM validation loss: 0.625057
Approximate MLM perplexity: 1.8684

FINAL FAIR COMPARISON
MLM_Control     | Loss: 0.624643 | PPL: 1.8676
MLM_MUSE        | Loss: 0.625057 | PPL: 1.8684


In [ ]:
import os
import re
import math
import torch
import numpy as np
import pandas as pd
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModelForMaskedLM

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

PROJECT_DIR = "/content/drive/MyDrive/BabyLM_Chinese_English_Translation"

# Evaluate only the corrected word-level MUSE model
models = {
    "English_BabyRoBERTa_Word_MUSE":
        f"{PROJECT_DIR}/babyroberta_english_continued_word_muse/final_model",
}

essay_files = {
    "Native_English": "/content/drive/MyDrive/SLA_Project/native_wikitext.txt",
    "Native_English_ICNALE": "/content/drive/MyDrive/SLA_Project/ens_icnale_clean.txt",
    "Chinese_ICNALE": "/content/drive/MyDrive/SLA_Project/chinese_icnale_clean.txt",
    "Japanese_ICNALE": "/content/drive/MyDrive/SLA_Project/japanese_icnale_clean.txt",
    "Korean_ICNALE": "/content/drive/MyDrive/SLA_Project/kor_icnale_clean.txt",
    "Thai_ICNALE": "/content/drive/MyDrive/SLA_Project/tha_icnale_clean.txt",
    "Filipino_ICNALE": "/content/drive/MyDrive/SLA_Project/phl_icnale_clean.txt",
    "Urdu_ICNALE": "/content/drive/MyDrive/SLA_Project/urdu_icnale_clean.txt",
}

def compute_pseudo_perplexity(text, tokenizer, model, max_length=128):
    model.eval()

    encodings = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=max_length
    )

    input_ids = encodings["input_ids"].to(device)
    attention_mask = encodings["attention_mask"].to(device)

    seq_len = input_ids.size(1)
    log_likelihood = 0.0
    token_count = 0

    with torch.no_grad():
        for i in range(1, seq_len - 1):
            masked_input = input_ids.clone()
            masked_input[0, i] = tokenizer.mask_token_id

            outputs = model(
                input_ids=masked_input,
                attention_mask=attention_mask
            )

            logits = outputs.logits
            log_probs = torch.log_softmax(logits[0, i], dim=-1)

            true_token = input_ids[0, i]

            log_likelihood += log_probs[true_token].item()
            token_count += 1

    if token_count == 0:
        return None

    return math.exp(-log_likelihood / token_count)

all_results = {}

for model_name, model_path in models.items():

    print("\n" + "=" * 80)
    print("Loading model:", model_name)
    print("Path:", model_path)
    print("=" * 80)

    if not os.path.exists(model_path):
        raise FileNotFoundError(f"Model path not found: {model_path}")

    tokenizer = AutoTokenizer.from_pretrained(model_path)
    model = AutoModelForMaskedLM.from_pretrained(model_path).to(device)
    model.eval()

    model_results = {}

    for group, path in essay_files.items():

        print(f"\nEvaluating {group}...")

        if not os.path.exists(path):
            print("Missing file:", path)
            continue

        with open(path, "r", encoding="utf-8", errors="ignore") as f:
            text = f.read()

        if group == "Chinese_ICNALE":
            essays = re.split(r"(?=I\s)", text)
            essays = [
                essay.strip()
                for essay in essays
                if len(essay.split()) > 50
            ]
        else:
            essays = [
                line.strip()
                for line in text.splitlines()
                if len(line.split()) > 5
            ]

        essays = essays[:200]
        ppl_scores = []

        for essay in tqdm(essays):
            ppl = compute_pseudo_perplexity(
                essay,
                tokenizer,
                model,
                max_length=128
            )

            if (
                ppl is not None
                and not math.isnan(ppl)
                and not math.isinf(ppl)
            ):
                ppl_scores.append(ppl)

        if not ppl_scores:
            avg_ppl = np.nan
        else:
            avg_ppl = float(np.mean(ppl_scores))

        model_results[group] = avg_ppl

        print(f"{group} Average Pseudo-PPL: {avg_ppl:.3f}")

    all_results[model_name] = model_results

print("\n" + "=" * 80)
print("FINAL WORD-LEVEL MUSE PSEUDO-PERPLEXITY RESULTS")
print("=" * 80)

results_df = pd.DataFrame(all_results)
print(results_df)

output_csv = (
    f"{PROJECT_DIR}/pseudo_perplexity_word_muse_only.csv"
)

results_df.to_csv(output_csv)

print("\nSaved results to:")
print(output_csv)


Loading model: English_BabyRoBERTa_Word_MUSE
Path: /content/drive/MyDrive/BabyLM_Chinese_English_Translation/babyroberta_english_continued_word_muse/final_model


Loading weights:   0%|          | 0/74 [00:00<?, ?it/s]

[transformers] RobertaForMaskedLM LOAD REPORT from: /content/drive/MyDrive/BabyLM_Chinese_English_Translation/babyroberta_english_continued_word_muse/final_model
Key                    | Status     |  | 
-----------------------+------------+--+-
muse_projection.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.



Evaluating Native_English...


100%|██████████| 200/200 [01:14<00:00,  2.69it/s]


...

Thai_ICNALE Average Pseudo-PPL: 143.454

Evaluating Filipino_ICNALE...


100%|██████████| 200/200 [01:09<00:00,  2.89it/s]


Filipino_ICNALE Average Pseudo-PPL: 129.903

Evaluating Urdu_ICNALE...


100%|██████████| 200/200 [01:28<00:00,  2.27it/s]

Urdu_ICNALE Average Pseudo-PPL: 142.284

FINAL WORD-LEVEL MUSE PSEUDO-PERPLEXITY RESULTS
                       English_BabyRoBERTa_Word_MUSE
Native_English                            466.675628
Native_English_ICNALE                     151.327828
Chinese_ICNALE                            129.452530
Japanese_ICNALE                           116.183094
Korean_ICNALE                             120.503612
Thai_ICNALE                               143.453656
Filipino_ICNALE                           129.903058
Urdu_ICNALE                               142.284282

Saved results to:
/content/drive/MyDrive/BabyLM_Chinese_English_Translation/pseudo_perplexity_word_muse_only.csv


In [ ]:
import os
import torch
import torch.nn.functional as F
import pandas as pd
from transformers import RobertaTokenizerFast, RobertaForMaskedLM

PROJECT_DIR = "/content/drive/MyDrive/BabyLM_Chinese_English_Translation"

INPUT_FILE = f"{PROJECT_DIR}/Mandarin_Question_MASK_CLEAN.txt"
OPTIONS_FILE = f"{PROJECT_DIR}/Mandarin_answers.txt"

MODEL_DIR = (
    f"{PROJECT_DIR}/babyroberta_english_continued_word_muse/final_model"
)

OUTPUT_DIR = (
    f"{PROJECT_DIR}/babyroberta_english_word_muse_preposition_eval"
)
os.makedirs(OUTPUT_DIR, exist_ok=True)

OUTPUT_FILE = (
    f"{OUTPUT_DIR}/babyroberta_english_word_muse_predictions.csv"
)

NORMALIZED_FILE = (
    f"{OUTPUT_DIR}/babyroberta_english_word_muse_normalized_scores.csv"
)

OPTIONS = [
    "on", "at", "like", "as", "with", "inside", "of", "among", "in",
    "by", "from", "during", "about", "near", "out", "round", "until",
    "along", "for", "against", "across", "to", "off", "upon", "towards",
    "under", "around", "behind", "besides", "within", "beside", "into",
    "between", "up", "over", "before", "above", "down", "after", "till",
    "toward", "without"
]

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Question exists:", os.path.exists(INPUT_FILE))
print("Answers exists :", os.path.exists(OPTIONS_FILE))
print("Model exists   :", os.path.exists(MODEL_DIR))

tokenizer = RobertaTokenizerFast.from_pretrained(MODEL_DIR)
model = RobertaForMaskedLM.from_pretrained(MODEL_DIR)

model.to(device)
model.eval()

print("Word-level MUSE BabyRoBERTa loaded.")
print("Using device:", device)
print("Tokenizer vocab size:", tokenizer.vocab_size)
print("Model vocab size:", model.config.vocab_size)
print("Mask token:", tokenizer.mask_token)

def predict_masked_word(sentence, options):
    inputs = tokenizer(
        sentence,
        return_tensors="pt",
        truncation=True,
        max_length=128
    )

    inputs = {k: v.to(device) for k, v in inputs.items()}
    input_ids = inputs["input_ids"]

    mask_positions = torch.where(
        input_ids == tokenizer.mask_token_id
    )[1]

    if mask_positions.numel() == 0:
        return {word: 0.0 for word in options}

    with torch.no_grad():
        logits = model(**inputs).logits

    mask_logits = logits[0, mask_positions[0], :]
    probs = F.softmax(mask_logits, dim=-1)

    word_probs = {}

    for word in options:
        token_ids = tokenizer.encode(
            word,
            add_special_tokens=False
        )

        if len(token_ids) == 1:
            word_probs[word] = probs[token_ids[0]].item()
        else:
            # Approximation for words split into multiple character tokens
            probability = 1.0

            for token_id in token_ids:
                probability *= probs[token_id].item()

            word_probs[word] = probability

    return word_probs

sentences = []

with open(INPUT_FILE, "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()

        if not line:
            continue

        line = line.replace("[MASK]", "<mask>")
        line = line.replace("____", "<mask>")
        line = line.replace(" ___ ", " <mask> ")
        line = line.replace("___", "<mask>")
        line = line.replace("__", "<mask>")

        if "<mask>" not in line:
            parts = line.split()

            if len(parts) > 1:
                parts.insert(-1, "<mask>")
                line = " ".join(parts)
            else:
                line = line + " <mask>"

        line = line.replace("<mask> <mask>", "<mask>")
        sentences.append(line)

print("Loaded", len(sentences), "sentences.")

rows = []

for sentence in sentences:
    probabilities = predict_masked_word(
        sentence,
        OPTIONS
    )

    row = {"Question": sentence}
    row.update(probabilities)
    rows.append(row)

df = pd.DataFrame(rows)

df.to_csv(
    OUTPUT_FILE,
    index=False,
    encoding="utf-8-sig"
)

print("Raw predictions saved to:", OUTPUT_FILE)

with open(OPTIONS_FILE, "r", encoding="utf-8") as f:
    option_lines = [
        line.strip()
        for line in f
        if line.strip()
    ]

if len(option_lines) != len(sentences):
    raise ValueError(
        "OPTIONS_FILE line count must match INPUT_FILE line count."
    )

normalized_rows = []

for i, allowed in enumerate(option_lines):
    allowed_list = [
        word.strip()
        for word in allowed.split(",")
        if word.strip()
    ]

    sentence = sentences[i]
    row = {"sentence": sentence}

    total = sum(
        df.loc[i, word]
        for word in allowed_list
        if word in df.columns
    )

    for word in OPTIONS:
        if word in allowed_list and total > 0:
            row[word] = df.loc[i, word] / total
        else:
            row[word] = 0.0

    normalized_rows.append(row)

df_norm = pd.DataFrame(normalized_rows)

df_norm.to_csv(
    NORMALIZED_FILE,
    index=False,
    encoding="utf-8-sig"
)

print("Normalized scores saved to:", NORMALIZED_FILE)
print("DONE.")

Question exists: True
Answers exists : True
Model exists   : True


Loading weights:   0%|          | 0/74 [00:00<?, ?it/s]

[transformers] RobertaForMaskedLM LOAD REPORT from: /content/drive/MyDrive/BabyLM_Chinese_English_Translation/babyroberta_english_continued_word_muse/final_model
Key                    | Status     |  | 
-----------------------+------------+--+-
muse_projection.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Word-level MUSE BabyRoBERTa loaded.
Using device: cuda
Tokenizer vocab size: 32000
Model vocab size: 32000
Mask token: <mask>
Loaded 100 sentences.
Raw predictions saved to: /content/drive/MyDrive/BabyLM_Chinese_English_Translation/babyroberta_english_word_muse_preposition_eval/babyroberta_english_word_muse_predictions.csv
Normalized scores saved to: /content/drive/MyDrive/BabyLM_Chinese_English_Translation/babyroberta_english_word_muse_preposition_eval/babyroberta_english_word_muse_normalized_scores.csv
DONE.


In [ ]:
import os
import re
import pandas as pd

# =============================================================================
# PATHS
# =============================================================================

PROJECT_DIR = "/content/drive/MyDrive/BabyLM_Chinese_English_Translation"

GOLD_CSV = "/content/drive/MyDrive/gold.csv"
QUESTION_EXCEL = "/content/drive/MyDrive/mandarin_Questions_Full.xlsx"

MODEL_PATHS = {
    "English BERT Base":
        f"{PROJECT_DIR}/bert_base_english_preposition_eval/"
        "bert_base_english_normalized_scores.csv",

    "English BabyRoBERTa":
        f"{PROJECT_DIR}/babyroberta_english_preposition_eval/"
        "babyroberta_english_normalized_scores.csv",

    "English BabyRoBERTa + MUSE":
        f"{PROJECT_DIR}/babyroberta_english_word_muse_preposition_eval/"
        "babyroberta_english_word_muse_normalized_scores.csv",

    "Chinese BabyRoBERTa":
        f"{PROJECT_DIR}/babyroberta_108M_preposition_eval/"
        "babyroberta_108M_normalized_scores.csv",
}

# =============================================================================
# NORMALIZE QUESTION TEXT
# =============================================================================

def normalize_question(text):
    text = str(text).lower()

    text = text.replace("，", ",")
    text = text.replace("’", "'")
    text = text.replace("`", "'")

    # Make all blank/mask forms identical
    text = text.replace("[mask]", " mask ")
    text = text.replace("<mask>", " mask ")
    text = text.replace("____", " mask ")
    text = text.replace("___", " mask ")
    text = text.replace("__", " mask ")

    text = re.sub(r"[^\w\s]", " ", text)
    text = re.sub(r"\s+", " ", text)

    return text.strip()


# =============================================================================
# LOAD HUMAN RESPONSE DATA
# =============================================================================

gold = pd.read_csv(GOLD_CSV)

gold["question_norm"] = gold["question"].apply(normalize_question)
gold["preposition"] = (
    gold["preposition"]
    .astype(str)
    .str.strip()
    .str.lower()
)
gold["Precent"] = pd.to_numeric(gold["Precent"], errors="coerce")

# Human top choice for each question
human_top = (
    gold.sort_values(
        ["question_norm", "Precent"],
        ascending=[True, False]
    )
    .groupby("question_norm", as_index=False)
    .first()
    [["question_norm", "preposition", "Precent"]]
    .rename(columns={"preposition": "human_top_choice"})
)

print("Human questions in gold.csv:", len(human_top))


# =============================================================================
# LOAD GRAMMATICALLY CORRECT ANSWERS
# Needed only to identify Table 5 wrong-dominant cases
# =============================================================================

correct_data = pd.read_excel(QUESTION_EXCEL)

question_col = "Question.1"
prep_col = "Preposition"
correct_col = "Correct"

correct_data["question_norm"] = (
    correct_data[question_col].apply(normalize_question)
)

correct_data[prep_col] = (
    correct_data[prep_col]
    .astype(str)
    .str.strip()
    .str.lower()
)

correct_data[correct_col] = pd.to_numeric(
    correct_data[correct_col],
    errors="coerce"
)

correct_answers = (
    correct_data[correct_data[correct_col] == 1]
    .drop_duplicates("question_norm")
    [["question_norm", prep_col]]
    .rename(columns={prep_col: "grammatical_answer"})
)

human_top = human_top.merge(
    correct_answers,
    on="question_norm",
    how="left"
)

human_top["human_top_is_incorrect"] = (
    human_top["human_top_choice"]
    != human_top["grammatical_answer"]
)


# =============================================================================
# EVALUATE ONE MODEL
# =============================================================================

def evaluate_model(model_name, model_path):
    model = pd.read_csv(model_path)

    sentence_col = (
        "sentence" if "sentence" in model.columns else "Question"
    )

    model["question_norm"] = model[sentence_col].apply(
        normalize_question
    )

    excluded = {
        "sentence",
        "Question",
        "question_norm",
    }

    probability_columns = [
        column
        for column in model.columns
        if column not in excluded
    ]

    # Convert all probability columns to numeric
    model[probability_columns] = model[
        probability_columns
    ].apply(pd.to_numeric, errors="coerce")

    model["model_choice"] = model[
        probability_columns
    ].idxmax(axis=1)

    model["model_choice"] = (
        model["model_choice"]
        .astype(str)
        .str.strip()
        .str.lower()
    )

    # Exact question matching
    comparison = human_top.merge(
        model[["question_norm", "model_choice"]],
        on="question_norm",
        how="inner"
    )

    # -------------------------------------------------------------------------
    # TABLE 4:
    # Agreement with the most common Mandarin-learner answer
    # -------------------------------------------------------------------------

    table4_correct = (
        comparison["model_choice"]
        == comparison["human_top_choice"]
    )

    table4_total = len(comparison)
    table4_count = int(table4_correct.sum())
    table4_accuracy = (
        table4_count / table4_total
        if table4_total else 0
    )

    # -------------------------------------------------------------------------
    # TABLE 5:
    # Keep only questions where the learners' top answer was incorrect.
    # Credit model when it chooses that same incorrect answer.
    # -------------------------------------------------------------------------

    wrong_cases = comparison[
        comparison["human_top_is_incorrect"]
    ].copy()

    table5_correct = (
        wrong_cases["model_choice"]
        == wrong_cases["human_top_choice"]
    )

    table5_total = len(wrong_cases)
    table5_count = int(table5_correct.sum())
    table5_accuracy = (
        table5_count / table5_total
        if table5_total else 0
    )

    return {
        "Model": model_name,
        "Matched questions": table4_total,
        "Table 4 correct": table4_count,
        "Table 4 accuracy": table4_accuracy,
        "Wrong-dominant questions": table5_total,
        "Table 5 correct": table5_count,
        "Table 5 accuracy": table5_accuracy,
    }


# =============================================================================
# RUN ALL MODELS
# =============================================================================

results = []

for model_name, model_path in MODEL_PATHS.items():
    print(f"Evaluating {model_name}...")
    results.append(
        evaluate_model(model_name, model_path)
    )

results_df = pd.DataFrame(results)

print("\n" + "=" * 90)
print("CORRECTED TABLE 4 — AGREEMENT WITH HUMAN TOP CHOICE")
print("=" * 90)

for _, row in results_df.iterrows():
    print(
        f"{row['Model']:35s}: "
        f"{row['Table 4 accuracy']:.4f} "
        f"({int(row['Table 4 correct'])}/"
        f"{int(row['Matched questions'])})"
    )

print("\n" + "=" * 90)
print("CORRECTED TABLE 5 — SAME INCORRECT CHOICE AS HUMANS")
print("=" * 90)

for _, row in results_df.iterrows():
    print(
        f"{row['Model']:35s}: "
        f"{row['Table 5 accuracy']:.4f} "
        f"({int(row['Table 5 correct'])}/"
        f"{int(row['Wrong-dominant questions'])})"
    )

OUTPUT = f"{PROJECT_DIR}/corrected_preposition_results_all_models.csv"
results_df.to_csv(OUTPUT, index=False)

print("\nSaved to:")
print(OUTPUT)

Human questions in gold.csv: 100
Evaluating English BERT Base...
Evaluating English BabyRoBERTa...
Evaluating English BabyRoBERTa + MUSE...
Evaluating Chinese BabyRoBERTa...

CORRECTED TABLE 4 — AGREEMENT WITH HUMAN TOP CHOICE
English BERT Base                  : 0.2500 (25/100)
English BabyRoBERTa                : 0.3000 (30/100)
English BabyRoBERTa + MUSE         : 0.3000 (30/100)
Chinese BabyRoBERTa                : 0.2000 (20/100)

CORRECTED TABLE 5 — SAME INCORRECT CHOICE AS HUMANS
English BERT Base                  : 0.2347 (23/98)
English BabyRoBERTa                : 0.3061 (30/98)
English BabyRoBERTa + MUSE         : 0.3061 (30/98)
Chinese BabyRoBERTa                : 0.1939 (19/98)

Saved to:
/content/drive/MyDrive/BabyLM_Chinese_English_Translation/corrected_preposition_results_all_models.csv


In [ ]:
import re
import pandas as pd

PROJECT_DIR = "/content/drive/MyDrive/BabyLM_Chinese_English_Translation"

GOLD_EXCEL = "/content/drive/MyDrive/mandarin_Questions_Full.xlsx"

MODEL_PATHS = {
    "English BERT Base":
        f"{PROJECT_DIR}/bert_base_english_preposition_eval/"
        "bert_base_english_normalized_scores.csv",

    "English BabyRoBERTa":
        f"{PROJECT_DIR}/babyroberta_english_preposition_eval/"
        "babyroberta_english_normalized_scores.csv",

    "English BabyRoBERTa + MUSE":
        f"{PROJECT_DIR}/babyroberta_english_word_muse_preposition_eval/"
        "babyroberta_english_word_muse_normalized_scores.csv",

    "Chinese BabyRoBERTa":
        f"{PROJECT_DIR}/babyroberta_108M_preposition_eval/"
        "babyroberta_108M_normalized_scores.csv",
}


def normalize_text(text):
    text = str(text).lower()

    text = text.replace("，", ",")
    text = text.replace("’", "'")
    text = text.replace("`", "'")

    text = text.replace("[mask]", " mask ")
    text = text.replace("<mask>", " mask ")
    text = text.replace("____", " mask ")
    text = text.replace("___", " mask ")
    text = text.replace("__", " mask ")

    text = re.sub(r"[^\w\s]", " ", text)
    text = re.sub(r"\s+", " ", text)

    return text.strip()


def find_best_match(question, model_df):
    q_words = set(question.split())

    best_row = None
    best_score = 0

    for _, row in model_df.iterrows():
        sentence_words = set(row["sentence_norm"].split())
        overlap = len(q_words.intersection(sentence_words))

        if overlap > best_score:
            best_score = overlap
            best_row = row

    return best_row if best_score >= 6 else None


def compute_human_like_error(gold_path, model_path):
    gold = pd.read_excel(gold_path)
    model = pd.read_csv(model_path)

    QUESTION_COL = "Question.1"
    PREP_COL = "Preposition"
    PERCENT_COL = "Precent"
    CORRECT_COL = "Correct"

    gold["question_norm"] = gold[QUESTION_COL].apply(normalize_text)
    model["sentence_norm"] = model["sentence"].apply(normalize_text)

    gold[PERCENT_COL] = pd.to_numeric(
        gold[PERCENT_COL],
        errors="coerce"
    )

    gold[CORRECT_COL] = pd.to_numeric(
        gold[CORRECT_COL],
        errors="coerce"
    )

    correct = 0
    total = 0
    wrong_dominant_cases = 0
    unmatched = 0

    for question, group in gold.groupby("question_norm"):
        group = group.sort_values(
            by=PERCENT_COL,
            ascending=False
        )

        if len(group) < 2:
            continue

        top = group.iloc[0]

        # Keep only questions where the most common human answer was wrong
        if top[CORRECT_COL] != 0:
            continue

        wrong_dominant_cases += 1

        human_wrong_answer = str(
            top[PREP_COL]
        ).strip().lower()

        model_row = find_best_match(question, model)

        if model_row is None:
            unmatched += 1
            continue

        probability_columns = [
            column
            for column in model.columns
            if column not in ["sentence", "sentence_norm"]
        ]

        probabilities = pd.to_numeric(
            model_row[probability_columns],
            errors="coerce"
        )

        if probabilities.isnull().all():
            continue

        model_answer = probabilities.idxmax().strip().lower()

        if model_answer == human_wrong_answer:
            correct += 1

        total += 1

    accuracy = correct / total if total else 0.0

    return {
        "accuracy": accuracy,
        "correct": correct,
        "total": total,
        "wrong_dominant_cases": wrong_dominant_cases,
        "unmatched": unmatched,
    }


print("=" * 90)
print("TABLE 5 — HUMAN-LIKE INCORRECT PREPOSITION CHOICE")
print("=" * 90)

table5_results = []

for model_name, model_path in MODEL_PATHS.items():
    result = compute_human_like_error(
        GOLD_EXCEL,
        model_path
    )

    table5_results.append({
        "Model": model_name,
        "Accuracy": result["accuracy"],
        "Correct": result["correct"],
        "Total": result["total"],
        "Wrong-dominant cases": result["wrong_dominant_cases"],
        "Unmatched": result["unmatched"],
    })

    print(
        f"{model_name:35s}: "
        f"{result['accuracy']:.4f} "
        f"({result['correct']}/{result['total']}), "
        f"wrong-dominant={result['wrong_dominant_cases']}, "
        f"unmatched={result['unmatched']}"
    )

table5_df = pd.DataFrame(table5_results)

output_file = (
    f"{PROJECT_DIR}/table5_corrected_human_like_error.csv"
)

table5_df.to_csv(output_file, index=False)

print("\nSaved to:")
print(output_file)

display(table5_df)

TABLE 5 — HUMAN-LIKE INCORRECT PREPOSITION CHOICE
English BERT Base                  : 0.0167 (1/60), wrong-dominant=62, unmatched=2
English BabyRoBERTa                : 0.3333 (20/60), wrong-dominant=62, unmatched=2
English BabyRoBERTa + MUSE         : 0.3333 (20/60), wrong-dominant=62, unmatched=2
Chinese BabyRoBERTa                : 0.2167 (13/60), wrong-dominant=62, unmatched=2

Saved to:
/content/drive/MyDrive/BabyLM_Chinese_English_Translation/table5_corrected_human_like_error.csv


,Model,Accuracy,Correct,Total,Wrong-dominant cases,Unmatched
0,English BERT Base,0.016667,1,60,62,2
1,English BabyRoBERTa,0.333333,20,60,62,2
2,English BabyRoBERTa + MUSE,0.333333,20,60,62,2
3,Chinese BabyRoBERTa,0.216667,13,60,62,2


In [ ]:
import os
import re
import math
import gc
import torch
import numpy as np
import pandas as pd
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModelForMaskedLM

# =============================================================================
# CONFIGURATION
# =============================================================================

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

PROJECT_DIR = "/content/drive/MyDrive/BabyLM_Chinese_English_Translation"

models = {
    "English BERT Base":
        "bert-base-uncased",

    "English BabyRoBERTa":
        f"{PROJECT_DIR}/babyroberta_english_same_arch_95_5_final",

    "English BabyRoBERTa + MUSE":
        f"{PROJECT_DIR}/babyroberta_english_continued_word_muse/final_model",

    "Chinese BabyRoBERTa":
        f"{PROJECT_DIR}/babyroberta_108M_same_arch_95_5_final",
}

essay_files = {
    "Native_English":
        "/content/drive/MyDrive/SLA_Project/native_wikitext.txt",

    "Native_English_ICNALE":
        "/content/drive/MyDrive/SLA_Project/ens_icnale_clean.txt",

    "Chinese_ICNALE":
        "/content/drive/MyDrive/SLA_Project/chinese_icnale_clean.txt",

    "Japanese_ICNALE":
        "/content/drive/MyDrive/SLA_Project/japanese_icnale_clean.txt",

    "Korean_ICNALE":
        "/content/drive/MyDrive/SLA_Project/kor_icnale_clean.txt",

    "Thai_ICNALE":
        "/content/drive/MyDrive/SLA_Project/tha_icnale_clean.txt",

    "Filipino_ICNALE":
        "/content/drive/MyDrive/SLA_Project/phl_icnale_clean.txt",

    "Urdu_ICNALE":
        "/content/drive/MyDrive/SLA_Project/urdu_icnale_clean.txt",
}

MAX_ESSAYS = 200
MAX_LENGTH = 128

print("=" * 90)
print("FOUR-MODEL PSEUDO-PERPLEXITY EVALUATION")
print("=" * 90)
print("Device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


# =============================================================================
# PSEUDO-PERPLEXITY FUNCTION
# =============================================================================

def compute_pseudo_perplexity(
    text,
    tokenizer,
    model,
    max_length=128
):
    """
    Masks one token at a time and measures the probability assigned
    to the original token.

    Lower pseudo-perplexity means the model is less surprised by the text.
    """

    encoded = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=max_length
    )

    input_ids = encoded["input_ids"].to(device)
    attention_mask = encoded["attention_mask"].to(device)

    seq_len = input_ids.size(1)

    log_likelihood = 0.0
    token_count = 0

    special_ids = set(
        token_id
        for token_id in [
            tokenizer.pad_token_id,
            tokenizer.bos_token_id,
            tokenizer.eos_token_id,
            tokenizer.cls_token_id,
            tokenizer.sep_token_id,
        ]
        if token_id is not None
    )

    with torch.no_grad():
        for position in range(seq_len):

            original_token_id = input_ids[0, position].item()

            # Do not evaluate padding or special tokens
            if attention_mask[0, position].item() == 0:
                continue

            if original_token_id in special_ids:
                continue

            masked_input = input_ids.clone()
            masked_input[0, position] = tokenizer.mask_token_id

            outputs = model(
                input_ids=masked_input,
                attention_mask=attention_mask
            )

            position_logits = outputs.logits[0, position]
            log_probs = torch.log_softmax(
                position_logits,
                dim=-1
            )

            log_likelihood += log_probs[
                original_token_id
            ].item()

            token_count += 1

    if token_count == 0:
        return None

    average_negative_log_likelihood = (
        -log_likelihood / token_count
    )

    # Prevent numerical overflow
    if average_negative_log_likelihood > 700:
        return None

    return math.exp(average_negative_log_likelihood)


# =============================================================================
# LOAD EVALUATION ESSAYS
# =============================================================================

def load_essays(group, path, max_essays=200):

    if not os.path.exists(path):
        raise FileNotFoundError(
            f"Evaluation file not found: {path}"
        )

    with open(
        path,
        "r",
        encoding="utf-8",
        errors="ignore"
    ) as file:
        text = file.read()

    if group == "Chinese_ICNALE":
        essays = re.split(r"(?=I\s)", text)

        essays = [
            essay.strip()
            for essay in essays
            if len(essay.split()) > 50
        ]

    else:
        essays = [
            line.strip()
            for line in text.splitlines()
            if len(line.split()) > 5
        ]

    return essays[:max_essays]


# Load each dataset once so every model receives exactly the same texts
evaluation_essays = {}

for group, path in essay_files.items():
    evaluation_essays[group] = load_essays(
        group,
        path,
        MAX_ESSAYS
    )

    print(
        f"{group:25s}: "
        f"{len(evaluation_essays[group])} essays"
    )


# =============================================================================
# EVALUATE ALL FOUR MODELS
# =============================================================================

all_results = {}
count_results = {}

for model_name, model_path in models.items():

    print("\n" + "=" * 90)
    print("Loading model:", model_name)
    print("Path:", model_path)
    print("=" * 90)

    if model_path != "bert-base-uncased":
        if not os.path.exists(model_path):
            raise FileNotFoundError(
                f"Model path does not exist: {model_path}"
            )

    tokenizer = AutoTokenizer.from_pretrained(
        model_path
    )

    model = AutoModelForMaskedLM.from_pretrained(
        model_path
    ).to(device)

    model.eval()

    model_results = {}
    model_counts = {}

    for group, essays in evaluation_essays.items():

        print(f"\nEvaluating {group}...")

        ppl_scores = []

        for essay in tqdm(
            essays,
            desc=f"{model_name} | {group}"
        ):
            ppl = compute_pseudo_perplexity(
                text=essay,
                tokenizer=tokenizer,
                model=model,
                max_length=MAX_LENGTH
            )

            if (
                ppl is not None
                and np.isfinite(ppl)
            ):
                ppl_scores.append(ppl)

        if len(ppl_scores) == 0:
            average_ppl = np.nan
        else:
            average_ppl = float(
                np.mean(ppl_scores)
            )

        model_results[group] = average_ppl
        model_counts[group] = len(ppl_scores)

        print(
            f"{group} Average Pseudo-PPL: "
            f"{average_ppl:.3f}"
        )

        print(
            f"Valid essays used: "
            f"{len(ppl_scores)}/{len(essays)}"
        )

    all_results[model_name] = model_results
    count_results[model_name] = model_counts

    # Release GPU memory before loading the next model
    del model
    del tokenizer
    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()


# =============================================================================
# FULL RESULTS
# =============================================================================

full_results_df = pd.DataFrame(
    all_results
).T

full_results_df.index.name = "Model"

full_counts_df = pd.DataFrame(
    count_results
).T

full_counts_df.index.name = "Model"

print("\n" + "=" * 90)
print("FULL PSEUDO-PERPLEXITY RESULTS")
print("=" * 90)

display(
    full_results_df.round(3)
)

FULL_RESULTS_FILE = (
    f"{PROJECT_DIR}/"
    "pseudo_perplexity_four_model_full_results.csv"
)

FULL_COUNTS_FILE = (
    f"{PROJECT_DIR}/"
    "pseudo_perplexity_four_model_valid_counts.csv"
)

full_results_df.to_csv(
    FULL_RESULTS_FILE
)

full_counts_df.to_csv(
    FULL_COUNTS_FILE
)


# =============================================================================
# PAPER TABLE 3
# EN = Native English
# ZH = Chinese ICNALE
# JA = Japanese ICNALE
# TH = Thai ICNALE
# FIL = Filipino ICNALE
# =============================================================================

table3_df = full_results_df[
    [
        "Native_English",
        "Chinese_ICNALE",
        "Japanese_ICNALE",
        "Thai_ICNALE",
        "Filipino_ICNALE",
    ]
].copy()

table3_df.columns = [
    "EN",
    "ZH",
    "JA",
    "TH",
    "FIL"
]

table3_df = table3_df.round(2)

print("\n" + "=" * 90)
print("TABLE 3 — PAPER-READY PSEUDO-PERPLEXITY RESULTS")
print("Lower scores indicate better language-model fit.")
print("=" * 90)

display(table3_df)

TABLE3_FILE = (
    f"{PROJECT_DIR}/"
    "table3_pseudo_perplexity_four_models.csv"
)

table3_df.to_csv(
    TABLE3_FILE
)


# =============================================================================
# LATEX VERSION FOR OVERLEAF
# =============================================================================

latex_table = table3_df.to_latex(
    float_format="%.2f",
    caption=(
        "Pseudo-perplexity scores for each model across native English "
        "and different learner language groups. Lower scores indicate "
        "better language-modeling fit."
    ),
    label="tab:pseudo_perplexity",
    position="ht",
)

LATEX_FILE = (
    f"{PROJECT_DIR}/"
    "table3_pseudo_perplexity_four_models.tex"
)

with open(
    LATEX_FILE,
    "w",
    encoding="utf-8"
) as file:
    file.write(latex_table)

print("\nSaved full results to:")
print(FULL_RESULTS_FILE)

print("\nSaved valid essay counts to:")
print(FULL_COUNTS_FILE)

print("\nSaved paper-ready Table 3 CSV to:")
print(TABLE3_FILE)

print("\nSaved Overleaf LaTeX table to:")
print(LATEX_FILE)

print("\nLaTeX preview:")
print(latex_table)

FOUR-MODEL PSEUDO-PERPLEXITY EVALUATION
Device: cuda
GPU: NVIDIA A100-SXM4-80GB
Native_English           : 200 essays
Native_English_ICNALE    : 200 essays
Chinese_ICNALE           : 200 essays
Japanese_ICNALE          : 200 essays
Korean_ICNALE            : 200 essays
Thai_ICNALE              : 200 essays
Filipino_ICNALE          : 200 essays
Urdu_ICNALE              : 200 essays

Loading model: English BERT Base
Path: bert-base-uncased


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

...

Urdu_ICNALE Average Pseudo-PPL: 224.533
Valid essays used: 200/200

FULL PSEUDO-PERPLEXITY RESULTS


,Native_English,Native_English_ICNALE,Chinese_ICNALE,Japanese_ICNALE,Korean_ICNALE,Thai_ICNALE,Filipino_ICNALE,Urdu_ICNALE
Model,,,,,,,,
English BERT Base,15.097,3.741,7.001,100.504,14.533,468.026,30.260,11.813
English BabyRoBERTa,481.974,148.084,127.344,114.487,119.391,141.926,128.348,140.881
English BabyRoBERTa + MUSE,466.676,151.328,129.453,116.183,120.504,143.454,129.903,142.284
Chinese BabyRoBERTa,717.502,168.742,82.214,140.183,125.348,518.487,309.541,224.533



TABLE 3 — PAPER-READY PSEUDO-PERPLEXITY RESULTS
Lower scores indicate better language-model fit.


,EN,ZH,JA,TH,FIL
Model,,,,,
English BERT Base,15.10,7.00,100.50,468.03,30.26
English BabyRoBERTa,481.97,127.34,114.49,141.93,128.35
English BabyRoBERTa + MUSE,466.68,129.45,116.18,143.45,129.90
Chinese BabyRoBERTa,717.50,82.21,140.18,518.49,309.54



Saved full results to:
/content/drive/MyDrive/BabyLM_Chinese_English_Translation/pseudo_perplexity_four_model_full_results.csv

Saved valid essay counts to:
/content/drive/MyDrive/BabyLM_Chinese_English_Translation/pseudo_perplexity_four_model_valid_counts.csv

Saved paper-ready Table 3 CSV to:
/content/drive/MyDrive/BabyLM_Chinese_English_Translation/table3_pseudo_perplexity_four_models.csv

Saved Overleaf LaTeX table to:
/content/drive/MyDrive/BabyLM_Chinese_English_Translation/table3_pseudo_perplexity_four_models.tex

LaTeX preview:
\begin{table}[ht]
\caption{Pseudo-perplexity scores for each model across native English and different learner language groups. Lower scores indicate better language-modeling fit.}
\label{tab:pseudo_perplexity}
\begin{tabular}{lrrrrr}
\toprule
 & EN & ZH & JA & TH & FIL \\
Model &  &  &  &  &  \\
\midrule
English BERT Base & 15.10 & 7.00 & 100.50 & 468.03 & 30.26 \\
English BabyRoBERTa & 481.97 & 127.34 & 114.49 & 141.93 & 128.35 \\
English BabyRoBERTa 

In [ ]:
import os
import random
import numpy as np
import torch

from datasets import Dataset
from transformers import (
    RobertaTokenizerFast,
    RobertaForMaskedLM,
    DataCollatorForLanguageModeling,
    Trainer,
    TrainingArguments,
)

# ============================================================
# Reproducibility
# ============================================================

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", device)

# ============================================================
# Project paths
# ============================================================

PROJECT_DIR = "/content/drive/MyDrive/BabyLM_Chinese_English_Translation"

BASE_MODEL = (
    f"{PROJECT_DIR}/babyroberta_english_same_arch_95_5_final"
)

TRAIN_FILE = (
    "/content/drive/MyDrive/SLA_Project/chinese_icnale_clean.txt"
)

OUTPUT_DIR = (
    f"{PROJECT_DIR}/babyroberta_english_essays_finetuned"
)

os.makedirs(OUTPUT_DIR, exist_ok=True)

print("\nBase model:")
print(BASE_MODEL)

print("\nTraining file:")
print(TRAIN_FILE)

print("\nOutput folder:")
print(OUTPUT_DIR)

print("\nEverything exists?")
print("Model :", os.path.exists(BASE_MODEL))
print("Essays:", os.path.exists(TRAIN_FILE))

Device: cuda

Base model:
/content/drive/MyDrive/BabyLM_Chinese_English_Translation/babyroberta_english_same_arch_95_5_final

Training file:
/content/drive/MyDrive/SLA_Project/chinese_icnale_clean.txt

Output folder:
/content/drive/MyDrive/BabyLM_Chinese_English_Translation/babyroberta_english_essays_finetuned

Everything exists?
Model : True
Essays: True


Load and split the Chinese ICNALE essays

In [ ]:
import re
from sklearn.model_selection import train_test_split
from datasets import Dataset, DatasetDict

print("=" * 80)
print("LOADING AND PREPARING CHINESE ICNALE ESSAYS")
print("=" * 80)

with open(
    TRAIN_FILE,
    "r",
    encoding="utf-8",
    errors="ignore"
) as f:
    raw_text = f.read()

# Split on line breaks and sentence-ending punctuation
sentence_candidates = re.split(
    r"(?<=[.!?])\s+|\n+",
    raw_text
)

sentences = []

for sentence in sentence_candidates:
    sentence = re.sub(r"\s+", " ", sentence).strip()

    # Keep usable English sentences only
    if len(sentence.split()) < 5:
        continue

    if len(sentence) < 20:
        continue

    sentences.append(sentence)

# Remove exact duplicates while preserving order
sentences = list(dict.fromkeys(sentences))

print("Usable unique sentences:", f"{len(sentences):,}")

if len(sentences) < 100:
    raise ValueError(
        "Too few usable sentences were found. "
        "Inspect the essay file before continuing."
    )

train_sentences, valid_sentences = train_test_split(
    sentences,
    test_size=0.10,
    random_state=SEED,
    shuffle=True
)

raw_datasets = DatasetDict({
    "train": Dataset.from_dict({
        "text": train_sentences
    }),
    "validation": Dataset.from_dict({
        "text": valid_sentences
    }),
})

print("Training sentences  :", f"{len(raw_datasets['train']):,}")
print("Validation sentences:", f"{len(raw_datasets['validation']):,}")

print("\nFirst 5 training examples:")
for i in range(5):
    print(f"{i + 1}: {raw_datasets['train'][i]['text']}")

LOADING AND PREPARING CHINESE ICNALE ESSAYS
Usable unique sentences: 11,539
Training sentences  : 10,385
Validation sentences: 1,154

First 5 training examples:
1: Maybe it is a little inconvenient, but for other people's sake, do not complain!For the two reasons mentioned above, and the fact that you can still smoke outside the restaurant during a meal, put your hands up to show that you are with me!
2: At least I think it is.
3: Maybe the restaurant doesn't have the authority to ban the diners from smoking, so some keeps the non-smokers apart from the smokers to protect the non-smokers.
4: This job has no relationship with the knowledge we learn.
5: Maybe people just smoke for some important purpose at first, but soon they won't help but smoking out of habit.


In [ ]:
corrected sentence splitting

In [ ]:
import re
from sklearn.model_selection import train_test_split
from datasets import Dataset, DatasetDict

print("=" * 80)
print("CLEANING AND PREPARING CHINESE ICNALE ESSAYS")
print("=" * 80)

with open(
    TRAIN_FILE,
    "r",
    encoding="utf-8",
    errors="ignore"
) as f:
    raw_text = f.read()

# Add a space after sentence-ending punctuation when missing
raw_text = re.sub(r"([.!?])(?=[A-Z])", r"\1 ", raw_text)

# Normalize whitespace
raw_text = re.sub(r"[ \t]+", " ", raw_text)

# Split on punctuation or line breaks
sentence_candidates = re.split(
    r"(?<=[.!?])\s+|\n+",
    raw_text
)

sentences = []

for sentence in sentence_candidates:
    sentence = re.sub(r"\s+", " ", sentence).strip()

    if len(sentence.split()) < 5:
        continue

    if len(sentence) < 20:
        continue

    sentences.append(sentence)

# Remove duplicate sentences while preserving order
sentences = list(dict.fromkeys(sentences))

print("Usable unique sentences:", f"{len(sentences):,}")

if len(sentences) < 100:
    raise ValueError("Too few usable sentences were found.")

train_sentences, valid_sentences = train_test_split(
    sentences,
    test_size=0.10,
    random_state=SEED,
    shuffle=True
)

raw_datasets = DatasetDict({
    "train": Dataset.from_dict({"text": train_sentences}),
    "validation": Dataset.from_dict({"text": valid_sentences}),
})

print("Training sentences  :", f"{len(raw_datasets['train']):,}")
print("Validation sentences:", f"{len(raw_datasets['validation']):,}")

print("\nFirst 5 training examples:")
for i in range(5):
    print(f"{i + 1}: {raw_datasets['train'][i]['text']}")

CLEANING AND PREPARING CHINESE ICNALE ESSAYS
Usable unique sentences: 11,569
Training sentences  : 10,412
Validation sentences: 1,157

First 5 training examples:
1: When college students graduate, they can put this into their applications.
2: So it is really hard to ban smoking completely in all places in the country.
3: Some people are in favor of smoking at restaurants while some others insist that it is unsuitable to smoke at there.
4: The previous arguments always focus on the necessity of college students having a part-time job.
5: Firstly, part-time jobs greatly teach us to be thankful owing to the fact that it is the first time in our life to earn on our own even though it is usually poorly-paid.


 Load tokenizer/model and tokenize the essay data

In [ ]:
from transformers import RobertaTokenizerFast, RobertaForMaskedLM

print("=" * 80)
print("LOADING ENGLISH BABYROBERTA AND TOKENIZING ESSAYS")
print("=" * 80)

tokenizer = RobertaTokenizerFast.from_pretrained(BASE_MODEL)

model = RobertaForMaskedLM.from_pretrained(BASE_MODEL)
model.to(device)

MAX_LENGTH = 128

def tokenize_batch(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        max_length=MAX_LENGTH,
        padding="max_length",
        return_special_tokens_mask=True,
    )

tokenized_datasets = raw_datasets.map(
    tokenize_batch,
    batched=True,
    batch_size=1000,
    num_proc=2,
    remove_columns=["text"],
)

print("Tokenizer vocab size:", tokenizer.vocab_size)
print("Model vocab size    :", model.config.vocab_size)
print("Hidden size         :", model.config.hidden_size)
print("Model device        :", next(model.parameters()).device)

print("\nTokenized datasets:")
print(tokenized_datasets)

print("\nFirst tokenized example length:")
print(len(tokenized_datasets["train"][0]["input_ids"]))

LOADING ENGLISH BABYROBERTA AND TOKENIZING ESSAYS


Loading weights:   0%|          | 0/74 [00:01<?, ?it/s]

Map (num_proc=2):   0%|          | 0/10412 [00:00<?, ? examples/s]

Map (num_proc=2):   0%|          | 0/1157 [00:00<?, ? examples/s]

Tokenizer vocab size: 32000
Model vocab size    : 32000
Hidden size         : 256
Model device        : cuda:0

Tokenized datasets:
DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask', 'special_tokens_mask'],
        num_rows: 10412
    })
    validation: Dataset({
        features: ['input_ids', 'attention_mask', 'special_tokens_mask'],
        num_rows: 1157
    })
})

First tokenized example length:
128


 Define MLM collator and training settings

In [ ]:
from transformers import (
    DataCollatorForLanguageModeling,
    TrainingArguments,
    Trainer,
)

print("=" * 80)
print("SETTING UP ESSAY FINE-TUNING")
print("=" * 80)

MLM_PROBABILITY = 0.15
EPOCHS = 3
TRAIN_BATCH_SIZE = 128
EVAL_BATCH_SIZE = 128
LEARNING_RATE = 5e-5
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.10

CHECKPOINT_DIR = f"{OUTPUT_DIR}/checkpoints"
FINAL_MODEL_DIR = f"{OUTPUT_DIR}/final_model"

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=True,
    mlm_probability=MLM_PROBABILITY,
)

training_args = TrainingArguments(
    output_dir=CHECKPOINT_DIR,

    do_train=True,
    do_eval=True,

    num_train_epochs=EPOCHS,

    per_device_train_batch_size=TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=EVAL_BATCH_SIZE,

    learning_rate=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    warmup_ratio=WARMUP_RATIO,

    logging_strategy="steps",
    logging_steps=50,

    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=3,

    bf16=True,
    fp16=False,

    dataloader_num_workers=2,
    report_to="none",

    seed=SEED,
    data_seed=SEED,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    data_collator=data_collator,
)

print("Training examples :", len(tokenized_datasets["train"]))
print("Validation examples:", len(tokenized_datasets["validation"]))
print("Epochs            :", EPOCHS)
print("Train batch size  :", TRAIN_BATCH_SIZE)
print("Learning rate     :", LEARNING_RATE)
print("MLM probability   :", MLM_PROBABILITY)

print("\nCheckpoint folder:")
print(CHECKPOINT_DIR)

print("\nFinal model folder:")
print(FINAL_MODEL_DIR)

print("\nEssay fine-tuning setup is ready.")

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


SETTING UP ESSAY FINE-TUNING
Training examples : 10412
Validation examples: 1157
Epochs            : 3
Train batch size  : 128
Learning rate     : 5e-05
MLM probability   : 0.15

Checkpoint folder:
/content/drive/MyDrive/BabyLM_Chinese_English_Translation/babyroberta_english_essays_finetuned/checkpoints

Final model folder:
/content/drive/MyDrive/BabyLM_Chinese_English_Translation/babyroberta_english_essays_finetuned/final_model

Essay fine-tuning setup is ready.


Train and save Model_essay_tuneed

In [ ]:
print("=" * 80)
print("TRAINING ENGLISH BABYROBERTA ON CHINESE-LEARNER ESSAYS")
print("=" * 80)

train_result = trainer.train()

# Save final model and tokenizer
trainer.save_model(FINAL_MODEL_DIR)
tokenizer.save_pretrained(FINAL_MODEL_DIR)

# Save metrics and trainer state
train_metrics = train_result.metrics
trainer.log_metrics("train", train_metrics)
trainer.save_metrics("train", train_metrics)
trainer.save_state()

print("\nTraining complete.")

print("\nFinal model saved to:")
print(FINAL_MODEL_DIR)

print("\nFinal training metrics:")
for key, value in train_metrics.items():
    print(f"{key}: {value}")

TRAINING ENGLISH BABYROBERTA ON CHINESE-LEARNER ESSAYS


Epoch,Training Loss,Validation Loss
1,0.715762,0.527979
2,0.614233,0.499805
3,0.605419,0.505271


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

***** train metrics *****
  epoch                    =        3.0
  total_flos               =    72786GF
  train_loss               =     0.6362
  train_runtime            = 0:00:14.70
  train_samples_per_second =   2123.849
  train_steps_per_second   =     16.726

Training complete.

Final model saved to:
/content/drive/MyDrive/BabyLM_Chinese_English_Translation/babyroberta_english_essays_finetuned/final_model

Final training metrics:
train_runtime: 14.7073
train_samples_per_second: 2123.849
train_steps_per_second: 16.726
total_flos: 78153515532288.0
train_loss: 0.6361759930122189
epoch: 3.0


In [ ]:
import os
import torch
import torch.nn.functional as F
import pandas as pd
from transformers import RobertaTokenizerFast, RobertaForMaskedLM

PROJECT_DIR = "/content/drive/MyDrive/BabyLM_Chinese_English_Translation"

INPUT_FILE = f"{PROJECT_DIR}/Mandarin_Question_MASK_CLEAN.txt"
OPTIONS_FILE = f"{PROJECT_DIR}/Mandarin_answers.txt"

MODEL_DIR = (
    f"{PROJECT_DIR}/babyroberta_english_essays_finetuned/final_model"
)

OUTPUT_DIR = (
    f"{PROJECT_DIR}/babyroberta_english_essays_preposition_eval"
)
os.makedirs(OUTPUT_DIR, exist_ok=True)

OUTPUT_FILE = (
    f"{OUTPUT_DIR}/babyroberta_english_essays_predictions.csv"
)

NORMALIZED_FILE = (
    f"{OUTPUT_DIR}/babyroberta_english_essays_normalized_scores.csv"
)

OPTIONS = [
    "on", "at", "like", "as", "with", "inside", "of", "among", "in",
    "by", "from", "during", "about", "near", "out", "round", "until",
    "along", "for", "against", "across", "to", "off", "upon", "towards",
    "under", "around", "behind", "besides", "within", "beside", "into",
    "between", "up", "over", "before", "above", "down", "after", "till",
    "toward", "without"
]

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Question exists:", os.path.exists(INPUT_FILE))
print("Answers exists :", os.path.exists(OPTIONS_FILE))
print("Model exists   :", os.path.exists(MODEL_DIR))

tokenizer = RobertaTokenizerFast.from_pretrained(MODEL_DIR)
model = RobertaForMaskedLM.from_pretrained(MODEL_DIR)

model.to(device)
model.eval()

print("Essay-fine-tuned English BabyRoBERTa loaded.")
print("Using device:", device)
print("Tokenizer vocab size:", tokenizer.vocab_size)
print("Model vocab size:", model.config.vocab_size)
print("Mask token:", tokenizer.mask_token)

def predict_masked_word(sentence, options):
    inputs = tokenizer(
        sentence,
        return_tensors="pt",
        truncation=True,
        max_length=128
    )

    inputs = {k: v.to(device) for k, v in inputs.items()}
    input_ids = inputs["input_ids"]

    mask_positions = torch.where(
        input_ids == tokenizer.mask_token_id
    )[1]

    if mask_positions.numel() == 0:
        return {word: 0.0 for word in options}

    with torch.no_grad():
        logits = model(**inputs).logits

    mask_logits = logits[0, mask_positions[0], :]
    probs = F.softmax(mask_logits, dim=-1)

    word_probs = {}

    for word in options:
        token_ids = tokenizer.encode(
            word,
            add_special_tokens=False
        )

        if len(token_ids) == 1:
            word_probs[word] = probs[token_ids[0]].item()
        else:
            probability = 1.0

            for token_id in token_ids:
                probability *= probs[token_id].item()

            word_probs[word] = probability

    return word_probs

sentences = []

with open(INPUT_FILE, "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()

        if not line:
            continue

        line = line.replace("[MASK]", "<mask>")
        line = line.replace("____", "<mask>")
        line = line.replace(" ___ ", " <mask> ")
        line = line.replace("___", "<mask>")
        line = line.replace("__", "<mask>")

        if "<mask>" not in line:
            parts = line.split()

            if len(parts) > 1:
                parts.insert(-1, "<mask>")
                line = " ".join(parts)
            else:
                line = line + " <mask>"

        line = line.replace("<mask> <mask>", "<mask>")
        sentences.append(line)

print("Loaded", len(sentences), "sentences.")

rows = []

for sentence in sentences:
    probabilities = predict_masked_word(
        sentence,
        OPTIONS
    )

    row = {"Question": sentence}
    row.update(probabilities)
    rows.append(row)

df = pd.DataFrame(rows)

df.to_csv(
    OUTPUT_FILE,
    index=False,
    encoding="utf-8-sig"
)

print("Raw predictions saved to:", OUTPUT_FILE)

with open(OPTIONS_FILE, "r", encoding="utf-8") as f:
    option_lines = [
        line.strip()
        for line in f
        if line.strip()
    ]

if len(option_lines) != len(sentences):
    raise ValueError(
        "OPTIONS_FILE line count must match INPUT_FILE line count."
    )

normalized_rows = []

for i, allowed in enumerate(option_lines):
    allowed_list = [
        word.strip()
        for word in allowed.split(",")
        if word.strip()
    ]

    sentence = sentences[i]
    row = {"sentence": sentence}

    total = sum(
        df.loc[i, word]
        for word in allowed_list
        if word in df.columns
    )

    for word in OPTIONS:
        if word in allowed_list and total > 0:
            row[word] = df.loc[i, word] / total
        else:
            row[word] = 0.0

    normalized_rows.append(row)

df_norm = pd.DataFrame(normalized_rows)

df_norm.to_csv(
    NORMALIZED_FILE,
    index=False,
    encoding="utf-8-sig"
)

print("Normalized scores saved to:", NORMALIZED_FILE)
print("DONE.")

Question exists: True
Answers exists : True
Model exists   : True


Loading weights:   0%|          | 0/74 [00:00<?, ?it/s]

Essay-fine-tuned English BabyRoBERTa loaded.
Using device: cuda
Tokenizer vocab size: 32000
Model vocab size: 32000
Mask token: <mask>
Loaded 100 sentences.
Raw predictions saved to: /content/drive/MyDrive/BabyLM_Chinese_English_Translation/babyroberta_english_essays_preposition_eval/babyroberta_english_essays_predictions.csv
Normalized scores saved to: /content/drive/MyDrive/BabyLM_Chinese_English_Translation/babyroberta_english_essays_preposition_eval/babyroberta_english_essays_normalized_scores.csv
DONE.


In [ ]:
import os
import re
import math
import torch
import numpy as np
import pandas as pd
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModelForMaskedLM

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

PROJECT_DIR = "/content/drive/MyDrive/BabyLM_Chinese_English_Translation"

# Evaluate only the corrected word-level MUSE model
models = {
    "English_BabyRoBERTa_Essays":
        f"{PROJECT_DIR}/babyroberta_english_essays_finetuned/final_model",
}

essay_files = {
    "Native_English": "/content/drive/MyDrive/SLA_Project/native_wikitext.txt",
    "Native_English_ICNALE": "/content/drive/MyDrive/SLA_Project/ens_icnale_clean.txt",
    "Chinese_ICNALE": "/content/drive/MyDrive/SLA_Project/chinese_icnale_clean.txt",
    "Japanese_ICNALE": "/content/drive/MyDrive/SLA_Project/japanese_icnale_clean.txt",
    "Korean_ICNALE": "/content/drive/MyDrive/SLA_Project/kor_icnale_clean.txt",
    "Thai_ICNALE": "/content/drive/MyDrive/SLA_Project/tha_icnale_clean.txt",
    "Filipino_ICNALE": "/content/drive/MyDrive/SLA_Project/phl_icnale_clean.txt",
    "Urdu_ICNALE": "/content/drive/MyDrive/SLA_Project/urdu_icnale_clean.txt",
}

def compute_pseudo_perplexity(text, tokenizer, model, max_length=128):
    model.eval()

    encodings = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=max_length
    )

    input_ids = encodings["input_ids"].to(device)
    attention_mask = encodings["attention_mask"].to(device)

    seq_len = input_ids.size(1)
    log_likelihood = 0.0
    token_count = 0

    with torch.no_grad():
        for i in range(1, seq_len - 1):
            masked_input = input_ids.clone()
            masked_input[0, i] = tokenizer.mask_token_id

            outputs = model(
                input_ids=masked_input,
                attention_mask=attention_mask
            )

            logits = outputs.logits
            log_probs = torch.log_softmax(logits[0, i], dim=-1)

            true_token = input_ids[0, i]

            log_likelihood += log_probs[true_token].item()
            token_count += 1

    if token_count == 0:
        return None

    return math.exp(-log_likelihood / token_count)

all_results = {}

for model_name, model_path in models.items():

    print("\n" + "=" * 80)
    print("Loading model:", model_name)
    print("Path:", model_path)
    print("=" * 80)

    if not os.path.exists(model_path):
        raise FileNotFoundError(f"Model path not found: {model_path}")

    tokenizer = AutoTokenizer.from_pretrained(model_path)
    model = AutoModelForMaskedLM.from_pretrained(model_path).to(device)
    model.eval()

    model_results = {}

    for group, path in essay_files.items():

        print(f"\nEvaluating {group}...")

        if not os.path.exists(path):
            print("Missing file:", path)
            continue

        with open(path, "r", encoding="utf-8", errors="ignore") as f:
            text = f.read()

        if group == "Chinese_ICNALE":
            essays = re.split(r"(?=I\s)", text)
            essays = [
                essay.strip()
                for essay in essays
                if len(essay.split()) > 50
            ]
        else:
            essays = [
                line.strip()
                for line in text.splitlines()
                if len(line.split()) > 5
            ]

        essays = essays[:200]
        ppl_scores = []

        for essay in tqdm(essays):
            ppl = compute_pseudo_perplexity(
                essay,
                tokenizer,
                model,
                max_length=128
            )

            if (
                ppl is not None
                and not math.isnan(ppl)
                and not math.isinf(ppl)
            ):
                ppl_scores.append(ppl)

        if not ppl_scores:
            avg_ppl = np.nan
        else:
            avg_ppl = float(np.mean(ppl_scores))

        model_results[group] = avg_ppl

        print(f"{group} Average Pseudo-PPL: {avg_ppl:.3f}")

    all_results[model_name] = model_results

print("\n" + "=" * 80)
print("FINAL ESSAY-FINETUNED BABYROBERTA PSEUDO-PERPLEXITY RESULTS")
print("=" * 80)

results_df = pd.DataFrame(all_results)
print(results_df)

output_csv = (

    f"{PROJECT_DIR}/pseudo_perplexity_english_essays_only.csv"

)

results_df.to_csv(output_csv)

print("\nSaved results to:")
print(output_csv)


Loading model: English_BabyRoBERTa_Essays
Path: /content/drive/MyDrive/BabyLM_Chinese_English_Translation/babyroberta_english_essays_finetuned/final_model


Loading weights:   0%|          | 0/74 [00:00<?, ?it/s]


Evaluating Native_English...


100%|██████████| 200/200 [01:25<00:00,  2.33it/s]


Native_English Average Pseudo-PPL: 543.043

Evaluating Native_English_ICNALE...


...

Thai_ICNALE Average Pseudo-PPL: 160.610

Evaluating Filipino_ICNALE...


100%|██████████| 200/200 [01:23<00:00,  2.39it/s]


Filipino_ICNALE Average Pseudo-PPL: 144.138

Evaluating Urdu_ICNALE...


100%|██████████| 200/200 [01:44<00:00,  1.92it/s]

Urdu_ICNALE Average Pseudo-PPL: 155.584

FINAL ESSAY-FINETUNED BABYROBERTA PSEUDO-PERPLEXITY RESULTS
                       English_BabyRoBERTa_Essays
Native_English                         543.043144
Native_English_ICNALE                  162.284731
Chinese_ICNALE                         138.668068
Japanese_ICNALE                        125.614865
Korean_ICNALE                          130.112038
Thai_ICNALE                            160.610005
Filipino_ICNALE                        144.138315
Urdu_ICNALE                            155.584049

Saved results to:
/content/drive/MyDrive/BabyLM_Chinese_English_Translation/pseudo_perplexity_english_essays_only.csv


chinese plus english mixed BabyROberta

In [ ]:
import os
from pathlib import Path

PROJECT_DIR = Path(
    "/content/drive/MyDrive/BabyLM_Chinese_English_Translation"
)

print("=" * 90)
print("MODEL B — SEARCHING FOR TRANSLATED AND ENGLISH CORPUS FILES")
print("=" * 90)

print("Project folder exists:", PROJECT_DIR.exists())
print("Project folder:", PROJECT_DIR)

keywords = [
    "translated",
    "translation",
    "w2w",
    "word",
    "chinese",
    "english",
    "train",
    "valid",
    "108m",
]

candidate_files = []

for path in PROJECT_DIR.rglob("*"):
    if not path.is_file():
        continue

    if path.suffix.lower() not in {".txt", ".csv", ".json", ".jsonl"}:
        continue

    name_lower = path.name.lower()

    if any(keyword in name_lower for keyword in keywords):
        candidate_files.append(path)

candidate_files = sorted(
    candidate_files,
    key=lambda p: p.stat().st_size,
    reverse=True
)

print("\nFound candidate files:", len(candidate_files))
print("-" * 90)

for i, path in enumerate(candidate_files[:100], start=1):
    size_mb = path.stat().st_size / (1024 ** 2)

    print(f"{i:3d}. {size_mb:10.2f} MB | {path}")

if not candidate_files:
    raise FileNotFoundError(
        "No likely corpus files were found in the project folder."
    )

MODEL B — SEARCHING FOR TRANSLATED AND ENGLISH CORPUS FILES
Project folder exists: True
Project folder: /content/drive/MyDrive/BabyLM_Chinese_English_Translation

Found candidate files: 75
------------------------------------------------------------------------------------------
  1.     580.87 MB | /content/drive/MyDrive/BabyLM_Chinese_English_Translation/final_english_training_corpus_gpt41mini_repaired.txt
  2.     551.83 MB | /content/drive/MyDrive/BabyLM_Chinese_English_Translation/babyroberta_train_95.txt
  3.     518.13 MB | /content/drive/MyDrive/BabyLM_Chinese_English_Translation/babylm_english_matched_108M_words.txt
  4.     492.32 MB | /content/drive/MyDrive/BabyLM_Chinese_English_Translation/babyroberta_english_train_95.txt
  5.     407.12 MB | /content/drive/MyDrive/BabyLM_Chinese_English_Translation/final_subtitles_english_gpt41mini_shuffled_seed42.txt
  6.     406.82 MB | /content/drive/MyDrive/BabyLM_Chinese_English_Translation/final_subtitles_english_gpt41mini_shuffled_

Inspect the two source corpora

In [ ]:
import os

print("=" * 90)
print("MODEL B — VERIFYING TRANSLATED AND ORIGINAL ENGLISH CORPORA")
print("=" * 90)

TRANSLATED_SOURCE = (
    f"{PROJECT_DIR}/final_english_training_corpus_gpt41mini_repaired.txt"
)

ENGLISH_SOURCE = (
    f"{PROJECT_DIR}/babylm_english_matched_108M_words.txt"
)

print("Translated source:")
print(TRANSLATED_SOURCE)
print("Exists:", os.path.exists(TRANSLATED_SOURCE))

print("\nOriginal English source:")
print(ENGLISH_SOURCE)
print("Exists:", os.path.exists(ENGLISH_SOURCE))

if not os.path.exists(TRANSLATED_SOURCE):
    raise FileNotFoundError(
        f"Translated corpus not found: {TRANSLATED_SOURCE}"
    )

if not os.path.exists(ENGLISH_SOURCE):
    raise FileNotFoundError(
        f"English corpus not found: {ENGLISH_SOURCE}"
    )


def inspect_text_file(path, sample_count=5):
    line_count = 0
    word_count = 0
    samples = []

    with open(
        path,
        "r",
        encoding="utf-8",
        errors="ignore"
    ) as file:
        for line in file:
            text = line.strip()

            if not text:
                continue

            line_count += 1
            word_count += len(text.split())

            if len(samples) < sample_count:
                samples.append(text)

    size_mb = os.path.getsize(path) / (1024 ** 2)

    return {
        "lines": line_count,
        "words": word_count,
        "size_mb": size_mb,
        "samples": samples,
    }


print("\n" + "-" * 90)
print("Inspecting translated Chinese→English corpus...")
translated_info = inspect_text_file(TRANSLATED_SOURCE)

print("Non-empty lines:", f"{translated_info['lines']:,}")
print("Total words    :", f"{translated_info['words']:,}")
print("File size MB   :", f"{translated_info['size_mb']:.2f}")

print("\nFirst 5 translated examples:")
for i, text in enumerate(translated_info["samples"], start=1):
    print(f"{i}: {text}")


print("\n" + "-" * 90)
print("Inspecting original English BabyLM corpus...")
english_info = inspect_text_file(ENGLISH_SOURCE)

print("Non-empty lines:", f"{english_info['lines']:,}")
print("Total words    :", f"{english_info['words']:,}")
print("File size MB   :", f"{english_info['size_mb']:.2f}")

print("\nFirst 5 original English examples:")
for i, text in enumerate(english_info["samples"], start=1):
    print(f"{i}: {text}")


print("\n" + "=" * 90)
print("REQUIRED FOR MODEL B")
print("=" * 90)
print("Translated words required:", f"{70_000_000:,}")
print("English words required   :", f"{30_000_000:,}")

print(
    "Enough translated data  :",
    translated_info["words"] >= 70_000_000
)

print(
    "Enough English data     :",
    english_info["words"] >= 30_000_000
)

MODEL B — VERIFYING TRANSLATED AND ORIGINAL ENGLISH CORPORA
Translated source:
/content/drive/MyDrive/BabyLM_Chinese_English_Translation/final_english_training_corpus_gpt41mini_repaired.txt
Exists: True

Original English source:
/content/drive/MyDrive/BabyLM_Chinese_English_Translation/babylm_english_matched_108M_words.txt
Exists: True

------------------------------------------------------------------------------------------
Inspecting translated Chinese→English corpus...
Non-empty lines: 4,414,524
Total words    : 108,042,603
File size MB   : 580.87

First 5 translated examples:
1: One one sheep child surged out of pen gate, jumping running toward boundless grassland
2: Herdsmen plural mount swift horse, chase that joyful sheep flock
3: Distance, one group group sheep child like cluster cluster white cloud at floating, blue sky under echoing shepherd's song
4: From text find out following words' antonyms
5: Enter--exit have--not near--far black--white

-------------------------------

 Create exact 70M translated + 30M original English corpus

In [ ]:
import os
import random

print("=" * 90)
print("MODEL B — CREATING 70M TRANSLATED + 30M ENGLISH MIX")
print("=" * 90)

SEED = 42

TRANSLATED_TARGET_WORDS = 70_000_000
ENGLISH_TARGET_WORDS = 30_000_000

MIXED_CORPUS_FILE = (
    f"{PROJECT_DIR}/babyroberta_mix_70M_translated_30M_english_shuffled.txt"
)

TEMP_TRANSLATED_FILE = (
    f"{PROJECT_DIR}/temp_translated_70M.txt"
)

TEMP_ENGLISH_FILE = (
    f"{PROJECT_DIR}/temp_english_30M.txt"
)


def extract_exact_words(source_path, output_path, target_words):
    written_words = 0
    written_lines = 0

    with open(
        source_path,
        "r",
        encoding="utf-8",
        errors="ignore"
    ) as source, open(
        output_path,
        "w",
        encoding="utf-8"
    ) as output:

        for line in source:
            text = line.strip()

            if not text:
                continue

            words = text.split()

            remaining = target_words - written_words

            if remaining <= 0:
                break

            if len(words) <= remaining:
                output.write(" ".join(words) + "\n")
                written_words += len(words)
                written_lines += 1
            else:
                output.write(" ".join(words[:remaining]) + "\n")
                written_words += remaining
                written_lines += 1
                break

    if written_words != target_words:
        raise ValueError(
            f"Expected {target_words:,} words but wrote {written_words:,}"
        )

    return written_words, written_lines


print("\nExtracting translated portion...")

translated_words, translated_lines = extract_exact_words(
    TRANSLATED_SOURCE,
    TEMP_TRANSLATED_FILE,
    TRANSLATED_TARGET_WORDS
)

print("Translated words:", f"{translated_words:,}")
print("Translated lines:", f"{translated_lines:,}")


print("\nExtracting original English portion...")

english_words, english_lines = extract_exact_words(
    ENGLISH_SOURCE,
    TEMP_ENGLISH_FILE,
    ENGLISH_TARGET_WORDS
)

print("English words:", f"{english_words:,}")
print("English lines:", f"{english_lines:,}")


print("\nLoading extracted lines for deterministic shuffle...")

with open(
    TEMP_TRANSLATED_FILE,
    "r",
    encoding="utf-8"
) as f:
    translated_lines_data = [
        line.rstrip("\n")
        for line in f
        if line.strip()
    ]

with open(
    TEMP_ENGLISH_FILE,
    "r",
    encoding="utf-8"
) as f:
    english_lines_data = [
        line.rstrip("\n")
        for line in f
        if line.strip()
    ]

mixed_lines = translated_lines_data + english_lines_data

print("Total lines before shuffle:", f"{len(mixed_lines):,}")

random.Random(SEED).shuffle(mixed_lines)

with open(
    MIXED_CORPUS_FILE,
    "w",
    encoding="utf-8"
) as f:
    for line in mixed_lines:
        f.write(line + "\n")


print("\nMixed corpus created.")

print("Saved to:")
print(MIXED_CORPUS_FILE)

print("File exists:", os.path.exists(MIXED_CORPUS_FILE))
print(
    "File size MB:",
    round(os.path.getsize(MIXED_CORPUS_FILE) / (1024 ** 2), 2)
)

print("\nFirst 10 shuffled lines:")

for i, line in enumerate(mixed_lines[:10], start=1):
    print(f"{i}: {line}")

print("\nFinal word composition:")
print("Translated Chinese→English:", f"{TRANSLATED_TARGET_WORDS:,}", "(70%)")
print("Original English          :", f"{ENGLISH_TARGET_WORDS:,}", "(30%)")
print("Total                     :", f"{TRANSLATED_TARGET_WORDS + ENGLISH_TARGET_WORDS:,}")

MODEL B — CREATING 70M TRANSLATED + 30M ENGLISH MIX

Extracting translated portion...
Translated words: 70,000,000
Translated lines: 3,362,253

Extracting original English portion...
English words: 30,000,000
English lines: 5,229,937

Loading extracted lines for deterministic shuffle...
Total lines before shuffle: 8,592,190

Mixed corpus created.
Saved to:
/content/drive/MyDrive/BabyLM_Chinese_English_Translation/babyroberta_mix_70M_translated_30M_english_shuffled.txt
File exists: True
File size MB: 531.9

First 10 shuffled lines:
1: *MOT: would you like to have a tea party?
2: Really kicked very good ah you see that ball too fierce I Generally go out play they several people you for example say go where place holiday good not easy place a holiday right arrange about seven eight people This cannot say you must say say.
3: Sichuan Liangshan area has a Yi ethnic orphan, heard Qionghai lake inside has a divine frog, knows those several bowls water's secret
4: *E: I'm gonna call the police

Create 95/5 train-validation split

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
import os
import random

print("=" * 90)
print("MODEL B — CREATING 95/5 TRAIN-VALIDATION SPLIT")
print("=" * 90)

MIX_SEED = 42
TRAIN_RATIO = 0.95

MIX_TRAIN_FILE = (
    f"{PROJECT_DIR}/babyroberta_mix_70_30_train_95.txt"
)

MIX_VALID_FILE = (
    f"{PROJECT_DIR}/babyroberta_mix_70_30_valid_5.txt"
)

with open(
    MIXED_CORPUS_FILE,
    "r",
    encoding="utf-8",
    errors="ignore"
) as f:
    all_lines = [
        line.rstrip("\n")
        for line in f
        if line.strip()
    ]

print("Total non-empty lines:", f"{len(all_lines):,}")

# The corpus is already shuffled, but this keeps the split deterministic.
random.Random(MIX_SEED).shuffle(all_lines)

split_index = int(len(all_lines) * TRAIN_RATIO)

train_lines = all_lines[:split_index]
valid_lines = all_lines[split_index:]

with open(
    MIX_TRAIN_FILE,
    "w",
    encoding="utf-8"
) as f:
    for line in train_lines:
        f.write(line + "\n")

with open(
    MIX_VALID_FILE,
    "w",
    encoding="utf-8"
) as f:
    for line in valid_lines:
        f.write(line + "\n")

print("\nSplit complete.")
print("Training lines  :", f"{len(train_lines):,}")
print("Validation lines:", f"{len(valid_lines):,}")

print("\nTraining file:")
print(MIX_TRAIN_FILE)

print("\nValidation file:")
print(MIX_VALID_FILE)

print("\nTrain exists:", os.path.exists(MIX_TRAIN_FILE))
print("Valid exists:", os.path.exists(MIX_VALID_FILE))

print(
    "Train size MB:",
    round(os.path.getsize(MIX_TRAIN_FILE) / (1024 ** 2), 2)
)

print(
    "Valid size MB:",
    round(os.path.getsize(MIX_VALID_FILE) / (1024 ** 2), 2)
)

MODEL B — CREATING 95/5 TRAIN-VALIDATION SPLIT
Total non-empty lines: 8,592,190

Split complete.
Training lines  : 8,162,580
Validation lines: 429,610

Training file:
/content/drive/MyDrive/BabyLM_Chinese_English_Translation/babyroberta_mix_70_30_train_95.txt

Validation file:
/content/drive/MyDrive/BabyLM_Chinese_English_Translation/babyroberta_mix_70_30_valid_5.txt

Train exists: True
Valid exists: True
Train size MB: 505.29
Valid size MB: 26.6


Load the same tokenizer and create a fresh BabyRoBERTa model

In [ ]:
import torch
from transformers import (
    RobertaTokenizerFast,
    RobertaConfig,
    RobertaForMaskedLM,
)

print("=" * 90)
print("MODEL B — LOADING TOKENIZER AND CREATING FRESH BABYROBERTA")
print("=" * 90)

TOKENIZER_DIR = (
    f"{PROJECT_DIR}/babyroberta_tokenizer_32k"
)

print("Tokenizer directory:")
print(TOKENIZER_DIR)

tokenizer = RobertaTokenizerFast.from_pretrained(
    TOKENIZER_DIR
)

config = RobertaConfig(
    vocab_size=tokenizer.vocab_size,
    max_position_embeddings=130,
    hidden_size=256,
    num_hidden_layers=4,
    num_attention_heads=4,
    intermediate_size=1024,
    hidden_act="gelu",
    hidden_dropout_prob=0.1,
    attention_probs_dropout_prob=0.1,
    type_vocab_size=1,
    pad_token_id=tokenizer.pad_token_id,
    bos_token_id=tokenizer.bos_token_id,
    eos_token_id=tokenizer.eos_token_id,
)

mix_model = RobertaForMaskedLM(config)
mix_model.to(device)

total_parameters = sum(
    parameter.numel()
    for parameter in mix_model.parameters()
)

print("\nTokenizer vocab size:", tokenizer.vocab_size)
print("Mask token          :", tokenizer.mask_token)
print("Pad token           :", tokenizer.pad_token)
print("BOS token           :", tokenizer.bos_token)
print("EOS token           :", tokenizer.eos_token)

print("\nModel architecture:")
print("Hidden size         :", config.hidden_size)
print("Transformer layers  :", config.num_hidden_layers)
print("Attention heads     :", config.num_attention_heads)
print("Intermediate size   :", config.intermediate_size)
print("Max positions       :", config.max_position_embeddings)

print("\nTotal parameters    :", f"{total_parameters:,}")
print("Parameters in M     :", round(total_parameters / 1_000_000, 2))
print("Model device        :", next(mix_model.parameters()).device)

if total_parameters != 11_483_392:
    print(
        "\nWarning: parameter count differs from the earlier "
        "11.48M BabyRoBERTa models."
    )
else:
    print("\nArchitecture exactly matches the earlier BabyRoBERTa models.")

MODEL B — LOADING TOKENIZER AND CREATING FRESH BABYROBERTA
Tokenizer directory:
/content/drive/MyDrive/BabyLM_Chinese_English_Translation/babyroberta_tokenizer_32k

Tokenizer vocab size: 32000
Mask token          : <mask>
Pad token           : <pad>
BOS token           : <s>
EOS token           : </s>

Model architecture:
Hidden size         : 256
Transformer layers  : 4
Attention heads     : 4
Intermediate size   : 1024
Max positions       : 130

Total parameters    : 11,483,392
Parameters in M     : 11.48
Model device        : cuda:0

Architecture exactly matches the earlier BabyRoBERTa models.


Load and tokenize the 95/5 mixed corpus

In [ ]:
from datasets import load_dataset

print("=" * 90)
print("MODEL B — LOADING AND TOKENIZING 70/30 MIX DATA")
print("=" * 90)

BLOCK_SIZE = 128

raw_mix_datasets = load_dataset(
    "text",
    data_files={
        "train": MIX_TRAIN_FILE,
        "validation": MIX_VALID_FILE,
    }
)

print(raw_mix_datasets)

def tokenize_mix_batch(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        max_length=BLOCK_SIZE,
        padding="max_length",
        return_special_tokens_mask=True,
    )

tokenized_mix_datasets = raw_mix_datasets.map(
    tokenize_mix_batch,
    batched=True,
    batch_size=1000,
    num_proc=2,
    remove_columns=["text"],
)

print("\nTokenization complete.")
print(tokenized_mix_datasets)

print("\nTraining examples  :", f"{len(tokenized_mix_datasets['train']):,}")
print("Validation examples:", f"{len(tokenized_mix_datasets['validation']):,}")
print("Sequence length    :", BLOCK_SIZE)

MODEL B — LOADING AND TOKENIZING 70/30 MIX DATA


Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text'],
        num_rows: 8162580
    })
    validation: Dataset({
        features: ['text'],
        num_rows: 429610
    })
})


Map (num_proc=2):   0%|          | 0/8162580 [00:00<?, ? examples/s]

Map (num_proc=2):   0%|          | 0/429610 [00:00<?, ? examples/s]


Tokenization complete.
DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask', 'special_tokens_mask'],
        num_rows: 8162580
    })
    validation: Dataset({
        features: ['input_ids', 'attention_mask', 'special_tokens_mask'],
        num_rows: 429610
    })
})

Training examples  : 8,162,580
Validation examples: 429,610
Sequence length    : 128


In [ ]:
import os
from transformers import (
    DataCollatorForLanguageModeling,
    TrainingArguments,
    Trainer,
)

print("=" * 90)
print("MODEL B — TRAINING SETUP")
print("=" * 90)

MIX_OUTPUT_DIR = (
    f"{PROJECT_DIR}/babyroberta_mix_70_30_same_arch_95_5"
)

MIX_CHECKPOINT_DIR = (
    f"{MIX_OUTPUT_DIR}/checkpoints"
)

MIX_FINAL_MODEL_DIR = (
    f"{MIX_OUTPUT_DIR}/final_model"
)

os.makedirs(MIX_OUTPUT_DIR, exist_ok=True)

# Same MLM masking as the earlier BabyRoBERTa models
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=True,
    mlm_probability=0.15,
)

TRAIN_BATCH_SIZE = 128
EVAL_BATCH_SIZE = 128
MAX_STEPS = 98_295
LEARNING_RATE = 5e-4
WEIGHT_DECAY = 0.01
WARMUP_STEPS = 9_830

mix_training_args = TrainingArguments(
    output_dir=MIX_CHECKPOINT_DIR,

    do_train=True,
    do_eval=True,

    max_steps=MAX_STEPS,

    per_device_train_batch_size=TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=EVAL_BATCH_SIZE,
    gradient_accumulation_steps=1,

    learning_rate=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    warmup_steps=WARMUP_STEPS,

    logging_strategy="steps",
    logging_steps=5_000,

    eval_strategy="steps",
    eval_steps=5_000,

    save_strategy="steps",
    save_steps=5_000,
    save_total_limit=3,

    bf16=True,
    fp16=False,

    dataloader_num_workers=2,
    report_to="none",

    seed=42,
    data_seed=42,
)

mix_trainer = Trainer(
    model=mix_model,
    args=mix_training_args,
    train_dataset=tokenized_mix_datasets["train"],
    eval_dataset=tokenized_mix_datasets["validation"],
    data_collator=data_collator,
)

print("Training examples   :", f"{len(tokenized_mix_datasets['train']):,}")
print("Validation examples :", f"{len(tokenized_mix_datasets['validation']):,}")
print("Batch size          :", TRAIN_BATCH_SIZE)
print("Max training steps  :", f"{MAX_STEPS:,}")
print("Learning rate       :", LEARNING_RATE)
print("Warmup steps        :", f"{WARMUP_STEPS:,}")
print("MLM probability     :", 0.15)

print("\nCheckpoint directory:")
print(MIX_CHECKPOINT_DIR)

print("\nFinal model directory:")
print(MIX_FINAL_MODEL_DIR)

print("\nModel B training setup is ready.")

MODEL B — TRAINING SETUP
Training examples   : 8,162,580
Validation examples : 429,610
Batch size          : 128
Max training steps  : 98,295
Learning rate       : 0.0005
Warmup steps        : 9,830
MLM probability     : 0.15

Checkpoint directory:
/content/drive/MyDrive/BabyLM_Chinese_English_Translation/babyroberta_mix_70_30_same_arch_95_5/checkpoints

Final model directory:
/content/drive/MyDrive/BabyLM_Chinese_English_Translation/babyroberta_mix_70_30_same_arch_95_5/final_model

Model B training setup is ready.


Train and save the 70/30 Mix model

In [ ]:
print("=" * 90)
print("TRAINING MODEL B — 70% TRANSLATED + 30% ORIGINAL ENGLISH")
print("=" * 90)

mix_train_result = mix_trainer.train()

# Save final model and tokenizer
mix_trainer.save_model(MIX_FINAL_MODEL_DIR)
tokenizer.save_pretrained(MIX_FINAL_MODEL_DIR)

# Save metrics and trainer state
mix_metrics = mix_train_result.metrics

mix_trainer.log_metrics(
    "mix_train",
    mix_metrics
)

mix_trainer.save_metrics(
    "mix_train",
    mix_metrics
)

mix_trainer.save_state()

print("\nModel B training complete.")

print("\nFinal model saved to:")
print(MIX_FINAL_MODEL_DIR)

print("\nTraining metrics:")
for key, value in mix_metrics.items():
    print(f"{key}: {value}")

TRAINING MODEL B — 70% TRANSLATED + 30% ORIGINAL ENGLISH


Step,Training Loss,Validation Loss
5000,3.196352,1.767497
10000,1.533639,1.152290
15000,1.140526,0.940361
20000,1.003415,0.851183
25000,0.929945,0.795406
30000,0.879695,0.752524
35000,0.841802,0.720491
40000,0.812864,0.693417
45000,0.785956,0.673481
50000,0.763330,0.652400


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

***** mix_train metrics *****
  epoch                    =     1.5414
  total_flos               = 29317767GF
  train_loss               =     0.9365
  train_runtime            = 1:24:03.85
  train_samples_per_second =   2494.474
  train_steps_per_second   =     19.488

Model B training complete.

Final model saved to:
/content/drive/MyDrive/BabyLM_Chinese_English_Translation/babyroberta_mix_70_30_same_arch_95_5/final_model

Training metrics:
train_runtime: 5043.8526
train_samples_per_second: 2494.474
train_steps_per_second: 19.488
total_flos: 3.1479713631830016e+16
train_loss: 0.9365462408478623
epoch: 1.5413746060121372


In [ ]:
import os
import torch
import torch.nn.functional as F
import pandas as pd
from transformers import RobertaTokenizerFast, RobertaForMaskedLM

PROJECT_DIR = "/content/drive/MyDrive/BabyLM_Chinese_English_Translation"

INPUT_FILE = f"{PROJECT_DIR}/Mandarin_Question_MASK_CLEAN.txt"
OPTIONS_FILE = f"{PROJECT_DIR}/Mandarin_answers.txt"

MODEL_DIR = (
    f"{PROJECT_DIR}/babyroberta_mix_70_30_same_arch_95_5/final_model"
)

OUTPUT_DIR = (
    f"{PROJECT_DIR}/babyroberta_mix_70_30_preposition_eval"
)
os.makedirs(OUTPUT_DIR, exist_ok=True)

OUTPUT_FILE = (
    f"{OUTPUT_DIR}/babyroberta_mix_70_30_predictions.csv"
)

NORMALIZED_FILE = (
    f"{OUTPUT_DIR}/babyroberta_mix_70_30_normalized_scores.csv"
)

OPTIONS = [
    "on", "at", "like", "as", "with", "inside", "of", "among", "in",
    "by", "from", "during", "about", "near", "out", "round", "until",
    "along", "for", "against", "across", "to", "off", "upon", "towards",
    "under", "around", "behind", "besides", "within", "beside", "into",
    "between", "up", "over", "before", "above", "down", "after", "till",
    "toward", "without"
]

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Question exists:", os.path.exists(INPUT_FILE))
print("Answers exists :", os.path.exists(OPTIONS_FILE))
print("Model exists   :", os.path.exists(MODEL_DIR))

tokenizer = RobertaTokenizerFast.from_pretrained(MODEL_DIR)
model = RobertaForMaskedLM.from_pretrained(MODEL_DIR)

model.to(device)
model.eval()

print("BabyRoBERTa 70% Translated + 30% English loaded.")
print("Using device:", device)
print("Tokenizer vocab size:", tokenizer.vocab_size)
print("Model vocab size:", model.config.vocab_size)
print("Mask token:", tokenizer.mask_token)

def predict_masked_word(sentence, options):
    inputs = tokenizer(
        sentence,
        return_tensors="pt",
        truncation=True,
        max_length=128
    )

    inputs = {k: v.to(device) for k, v in inputs.items()}
    input_ids = inputs["input_ids"]

    mask_positions = torch.where(
        input_ids == tokenizer.mask_token_id
    )[1]

    if mask_positions.numel() == 0:
        return {word: 0.0 for word in options}

    with torch.no_grad():
        logits = model(**inputs).logits

    mask_logits = logits[0, mask_positions[0], :]
    probs = F.softmax(mask_logits, dim=-1)

    word_probs = {}

    for word in options:
        token_ids = tokenizer.encode(
            word,
            add_special_tokens=False
        )

        if len(token_ids) == 1:
            word_probs[word] = probs[token_ids[0]].item()
        else:
            probability = 1.0

            for token_id in token_ids:
                probability *= probs[token_id].item()

            word_probs[word] = probability

    return word_probs

sentences = []

with open(INPUT_FILE, "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()

        if not line:
            continue

        line = line.replace("[MASK]", "<mask>")
        line = line.replace("____", "<mask>")
        line = line.replace(" ___ ", " <mask> ")
        line = line.replace("___", "<mask>")
        line = line.replace("__", "<mask>")

        if "<mask>" not in line:
            parts = line.split()

            if len(parts) > 1:
                parts.insert(-1, "<mask>")
                line = " ".join(parts)
            else:
                line = line + " <mask>"

        line = line.replace("<mask> <mask>", "<mask>")
        sentences.append(line)

print("Loaded", len(sentences), "sentences.")

rows = []

for sentence in sentences:
    probabilities = predict_masked_word(
        sentence,
        OPTIONS
    )

    row = {"Question": sentence}
    row.update(probabilities)
    rows.append(row)

df = pd.DataFrame(rows)

df.to_csv(
    OUTPUT_FILE,
    index=False,
    encoding="utf-8-sig"
)

print("Raw predictions saved to:", OUTPUT_FILE)

with open(OPTIONS_FILE, "r", encoding="utf-8") as f:
    option_lines = [
        line.strip()
        for line in f
        if line.strip()
    ]

if len(option_lines) != len(sentences):
    raise ValueError(
        "OPTIONS_FILE line count must match INPUT_FILE line count."
    )

normalized_rows = []

for i, allowed in enumerate(option_lines):
    allowed_list = [
        word.strip()
        for word in allowed.split(",")
        if word.strip()
    ]

    sentence = sentences[i]
    row = {"sentence": sentence}

    total = sum(
        df.loc[i, word]
        for word in allowed_list
        if word in df.columns
    )

    for word in OPTIONS:
        if word in allowed_list and total > 0:
            row[word] = df.loc[i, word] / total
        else:
            row[word] = 0.0

    normalized_rows.append(row)

df_norm = pd.DataFrame(normalized_rows)

df_norm.to_csv(
    NORMALIZED_FILE,
    index=False,
    encoding="utf-8-sig"
)

print("Normalized scores saved to:", NORMALIZED_FILE)
print("DONE.")

Question exists: True
Answers exists : True
Model exists   : True


Loading weights:   0%|          | 0/74 [00:00<?, ?it/s]

BabyRoBERTa 70% Translated + 30% English loaded.
Using device: cuda
Tokenizer vocab size: 32000
Model vocab size: 32000
Mask token: <mask>
Loaded 100 sentences.
Raw predictions saved to: /content/drive/MyDrive/BabyLM_Chinese_English_Translation/babyroberta_mix_70_30_preposition_eval/babyroberta_mix_70_30_predictions.csv
Normalized scores saved to: /content/drive/MyDrive/BabyLM_Chinese_English_Translation/babyroberta_mix_70_30_preposition_eval/babyroberta_mix_70_30_normalized_scores.csv
DONE.


In [ ]:
import os
import re
import math
import torch
import numpy as np
import pandas as pd
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModelForMaskedLM

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

PROJECT_DIR = "/content/drive/MyDrive/BabyLM_Chinese_English_Translation"

# Evaluate only the 70% translated + 30% English Mix model
models = {
    "BabyRoBERTa_Mix_70_30":
        f"{PROJECT_DIR}/babyroberta_mix_70_30_same_arch_95_5/final_model",
}

essay_files = {
    "Native_English": "/content/drive/MyDrive/SLA_Project/native_wikitext.txt",
    "Native_English_ICNALE": "/content/drive/MyDrive/SLA_Project/ens_icnale_clean.txt",
    "Chinese_ICNALE": "/content/drive/MyDrive/SLA_Project/chinese_icnale_clean.txt",
    "Japanese_ICNALE": "/content/drive/MyDrive/SLA_Project/japanese_icnale_clean.txt",
    "Korean_ICNALE": "/content/drive/MyDrive/SLA_Project/kor_icnale_clean.txt",
    "Thai_ICNALE": "/content/drive/MyDrive/SLA_Project/tha_icnale_clean.txt",
    "Filipino_ICNALE": "/content/drive/MyDrive/SLA_Project/phl_icnale_clean.txt",
    "Urdu_ICNALE": "/content/drive/MyDrive/SLA_Project/urdu_icnale_clean.txt",
}

def compute_pseudo_perplexity(text, tokenizer, model, max_length=128):
    model.eval()

    encodings = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=max_length
    )

    input_ids = encodings["input_ids"].to(device)
    attention_mask = encodings["attention_mask"].to(device)

    seq_len = input_ids.size(1)
    log_likelihood = 0.0
    token_count = 0

    with torch.no_grad():
        for i in range(1, seq_len - 1):
            masked_input = input_ids.clone()
            masked_input[0, i] = tokenizer.mask_token_id

            outputs = model(
                input_ids=masked_input,
                attention_mask=attention_mask
            )

            logits = outputs.logits
            log_probs = torch.log_softmax(logits[0, i], dim=-1)

            true_token = input_ids[0, i]

            log_likelihood += log_probs[true_token].item()
            token_count += 1

    if token_count == 0:
        return None

    return math.exp(-log_likelihood / token_count)

all_results = {}

for model_name, model_path in models.items():

    print("\n" + "=" * 80)
    print("Loading model:", model_name)
    print("Path:", model_path)
    print("=" * 80)

    if not os.path.exists(model_path):
        raise FileNotFoundError(f"Model path not found: {model_path}")

    tokenizer = AutoTokenizer.from_pretrained(model_path)
    model = AutoModelForMaskedLM.from_pretrained(model_path).to(device)
    model.eval()

    model_results = {}

    for group, path in essay_files.items():

        print(f"\nEvaluating {group}...")

        if not os.path.exists(path):
            print("Missing file:", path)
            continue

        with open(path, "r", encoding="utf-8", errors="ignore") as f:
            text = f.read()

        if group == "Chinese_ICNALE":
            essays = re.split(r"(?=I\s)", text)
            essays = [
                essay.strip()
                for essay in essays
                if len(essay.split()) > 50
            ]
        else:
            essays = [
                line.strip()
                for line in text.splitlines()
                if len(line.split()) > 5
            ]

        essays = essays[:200]
        ppl_scores = []

        for essay in tqdm(essays):
            ppl = compute_pseudo_perplexity(
                essay,
                tokenizer,
                model,
                max_length=128
            )

            if (
                ppl is not None
                and not math.isnan(ppl)
                and not math.isinf(ppl)
            ):
                ppl_scores.append(ppl)

        if not ppl_scores:
            avg_ppl = np.nan
        else:
            avg_ppl = float(np.mean(ppl_scores))

        model_results[group] = avg_ppl

        print(f"{group} Average Pseudo-PPL: {avg_ppl:.3f}")

    all_results[model_name] = model_results

print("\n" + "=" * 80)
print("FINAL 70/30 MIX BABYROBERTA PSEUDO-PERPLEXITY RESULTS")
print("=" * 80)

results_df = pd.DataFrame(all_results)
print(results_df)

output_csv = (
    f"{PROJECT_DIR}/pseudo_perplexity_mix_70_30_only.csv"
)

results_df.to_csv(output_csv)

print("\nSaved results to:")
print(output_csv)


Loading model: BabyRoBERTa_Mix_70_30
Path: /content/drive/MyDrive/BabyLM_Chinese_English_Translation/babyroberta_mix_70_30_same_arch_95_5/final_model


Loading weights:   0%|          | 0/74 [00:00<?, ?it/s]


Evaluating Native_English...


100%|██████████| 200/200 [01:13<00:00,  2.74it/s]


Native_English Average Pseudo-PPL: 391.128

Evaluating Native_English_ICNALE...


...

Thai_ICNALE Average Pseudo-PPL: 103.233

Evaluating Filipino_ICNALE...


100%|██████████| 200/200 [01:08<00:00,  2.91it/s]


Filipino_ICNALE Average Pseudo-PPL: 97.012

Evaluating Urdu_ICNALE...


100%|██████████| 200/200 [01:26<00:00,  2.31it/s]

Urdu_ICNALE Average Pseudo-PPL: 102.620

FINAL 70/30 MIX BABYROBERTA PSEUDO-PERPLEXITY RESULTS
                       BabyRoBERTa_Mix_70_30
Native_English                    391.128174
Native_English_ICNALE             129.056410
Chinese_ICNALE                     97.509663
Japanese_ICNALE                    85.209381
Korean_ICNALE                      88.250398
Thai_ICNALE                       103.232595
Filipino_ICNALE                    97.011746
Urdu_ICNALE                       102.619968

Saved results to:
/content/drive/MyDrive/BabyLM_Chinese_English_Translation/pseudo_perplexity_mix_70_30_only.csv


In [ ]:
import os
import re
import pandas as pd

PROJECT_DIR = "/content/drive/MyDrive/BabyLM_Chinese_English_Translation"

GOLD_EXCEL = "/content/drive/MyDrive/mandarin_Questions_Full.xlsx"

MODEL_PATHS = {
    "English BERT Base":
        f"{PROJECT_DIR}/bert_base_english_preposition_eval/"
        "bert_base_english_normalized_scores.csv",

    "English BabyRoBERTa":
        f"{PROJECT_DIR}/babyroberta_english_preposition_eval/"
        "babyroberta_english_normalized_scores.csv",

    "English BabyRoBERTa + MUSE":
        f"{PROJECT_DIR}/babyroberta_english_word_muse_preposition_eval/"
        "babyroberta_english_word_muse_normalized_scores.csv",

    "Chinese BabyRoBERTa":
        f"{PROJECT_DIR}/babyroberta_108M_preposition_eval/"
        "babyroberta_108M_normalized_scores.csv",

    # New Model A: English BabyRoBERTa fine-tuned on Chinese learner essays
    "English BabyRoBERTa + Essays":
        f"{PROJECT_DIR}/babyroberta_english_essays_preposition_eval/"
        "babyroberta_english_essays_normalized_scores.csv",

    # New Model B: 70% translated Chinese-to-English + 30% original English
    "BabyRoBERTa Mix 70/30":
        f"{PROJECT_DIR}/babyroberta_mix_70_30_preposition_eval/"
        "babyroberta_mix_70_30_normalized_scores.csv",
}

print("=" * 90)
print("CHECKING MODEL PREDICTION FILES")
print("=" * 90)

for model_name, model_path in MODEL_PATHS.items():
    print(f"{model_name:38s}: {os.path.exists(model_path)}")
    print(f"  {model_path}")

missing_files = [
    path
    for path in MODEL_PATHS.values()
    if not os.path.exists(path)
]

if missing_files:
    raise FileNotFoundError(
        "One or more normalized prediction files are missing:\n"
        + "\n".join(missing_files)
    )


def normalize_text(text):
    text = str(text).lower()

    text = text.replace("，", ",")
    text = text.replace("’", "'")
    text = text.replace("`", "'")

    text = text.replace("[mask]", " mask ")
    text = text.replace("<mask>", " mask ")
    text = text.replace("____", " mask ")
    text = text.replace("___", " mask ")
    text = text.replace("__", " mask ")

    text = re.sub(r"[^\w\s]", " ", text)
    text = re.sub(r"\s+", " ", text)

    return text.strip()


def find_best_match(question, model_df):
    q_words = set(question.split())

    best_row = None
    best_score = 0

    for _, row in model_df.iterrows():
        sentence_words = set(row["sentence_norm"].split())
        overlap = len(q_words.intersection(sentence_words))

        if overlap > best_score:
            best_score = overlap
            best_row = row

    return best_row if best_score >= 6 else None


def compute_human_like_error(gold_path, model_path):
    gold = pd.read_excel(gold_path)
    model = pd.read_csv(model_path)

    QUESTION_COL = "Question.1"
    PREP_COL = "Preposition"
    PERCENT_COL = "Precent"
    CORRECT_COL = "Correct"

    gold["question_norm"] = gold[QUESTION_COL].apply(normalize_text)
    model["sentence_norm"] = model["sentence"].apply(normalize_text)

    gold[PERCENT_COL] = pd.to_numeric(
        gold[PERCENT_COL],
        errors="coerce"
    )

    gold[CORRECT_COL] = pd.to_numeric(
        gold[CORRECT_COL],
        errors="coerce"
    )

    probability_columns = [
        column
        for column in model.columns
        if column not in ["sentence", "sentence_norm"]
    ]

    correct = 0
    total = 0
    wrong_dominant_cases = 0
    unmatched = 0

    for question, group in gold.groupby("question_norm"):
        group = group.sort_values(
            by=PERCENT_COL,
            ascending=False
        )

        if len(group) < 2:
            continue

        top = group.iloc[0]

        # Keep only questions where the most common human answer was wrong
        if top[CORRECT_COL] != 0:
            continue

        wrong_dominant_cases += 1

        human_wrong_answer = str(
            top[PREP_COL]
        ).strip().lower()

        model_row = find_best_match(
            question,
            model
        )

        if model_row is None:
            unmatched += 1
            continue

        probabilities = pd.to_numeric(
            model_row[probability_columns],
            errors="coerce"
        )

        if probabilities.isnull().all():
            continue

        model_answer = (
            probabilities
            .idxmax()
            .strip()
            .lower()
        )

        if model_answer == human_wrong_answer:
            correct += 1

        total += 1

    accuracy = correct / total if total else 0.0

    return {
        "accuracy": accuracy,
        "correct": correct,
        "total": total,
        "wrong_dominant_cases": wrong_dominant_cases,
        "unmatched": unmatched,
    }


print("\n" + "=" * 90)
print("TABLE 5 — HUMAN-LIKE INCORRECT PREPOSITION CHOICE")
print("=" * 90)

table5_results = []

for model_name, model_path in MODEL_PATHS.items():
    result = compute_human_like_error(
        GOLD_EXCEL,
        model_path
    )

    table5_results.append({
        "Model": model_name,
        "Accuracy": result["accuracy"],
        "Correct": result["correct"],
        "Total": result["total"],
        "Wrong-dominant cases": result["wrong_dominant_cases"],
        "Unmatched": result["unmatched"],
    })

    print(
        f"{model_name:38s}: "
        f"{result['accuracy']:.4f} "
        f"({result['correct']}/{result['total']}), "
        f"wrong-dominant={result['wrong_dominant_cases']}, "
        f"unmatched={result['unmatched']}"
    )

table5_df = pd.DataFrame(table5_results)

output_file = (
    f"{PROJECT_DIR}/"
    "table5_human_like_error_all_six_models.csv"
)

table5_df.to_csv(
    output_file,
    index=False
)

print("\nSaved to:")
print(output_file)

display(table5_df)

CHECKING MODEL PREDICTION FILES
English BERT Base                     : True
  /content/drive/MyDrive/BabyLM_Chinese_English_Translation/bert_base_english_preposition_eval/bert_base_english_normalized_scores.csv
English BabyRoBERTa                   : True
  /content/drive/MyDrive/BabyLM_Chinese_English_Translation/babyroberta_english_preposition_eval/babyroberta_english_normalized_scores.csv
English BabyRoBERTa + MUSE            : True
  /content/drive/MyDrive/BabyLM_Chinese_English_Translation/babyroberta_english_word_muse_preposition_eval/babyroberta_english_word_muse_normalized_scores.csv
Chinese BabyRoBERTa                   : True
  /content/drive/MyDrive/BabyLM_Chinese_English_Translation/babyroberta_108M_preposition_eval/babyroberta_108M_normalized_scores.csv
English BabyRoBERTa + Essays          : True
  /content/drive/MyDrive/BabyLM_Chinese_English_Translation/babyroberta_english_essays_preposition_eval/babyroberta_english_essays_normalized_scores.csv
BabyRoBERTa Mix 70/30   

,Model,Accuracy,Correct,Total,Wrong-dominant cases,Unmatched
0,English BERT Base,0.016667,1,60,62,2
1,English BabyRoBERTa,0.333333,20,60,62,2
2,English BabyRoBERTa + MUSE,0.333333,20,60,62,2
3,Chinese BabyRoBERTa,0.216667,13,60,62,2
4,English BabyRoBERTa + Essays,0.333333,20,60,62,2
5,BabyRoBERTa Mix 70/30,0.283333,17,60,62,2


In [ ]:
import os
import re
import math
import gc
import torch
import numpy as np
import pandas as pd
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModelForMaskedLM

# =============================================================================
# CONFIGURATION
# =============================================================================

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

PROJECT_DIR = "/content/drive/MyDrive/BabyLM_Chinese_English_Translation"

models = {
    "English BERT Base":
        "bert-base-uncased",

    "English BabyRoBERTa":
        f"{PROJECT_DIR}/babyroberta_english_same_arch_95_5_final",

    "English BabyRoBERTa + MUSE":
        f"{PROJECT_DIR}/babyroberta_english_continued_word_muse/final_model",

    "Chinese BabyRoBERTa":
        f"{PROJECT_DIR}/babyroberta_108M_same_arch_95_5_final",

    # Model A
    "English BabyRoBERTa + Essays":
        f"{PROJECT_DIR}/babyroberta_english_essays_finetuned/final_model",

    # Model B
    "BabyRoBERTa Mix 70/30":
        f"{PROJECT_DIR}/babyroberta_mix_70_30_same_arch_95_5/final_model",
}

essay_files = {
    "Native_English":
        "/content/drive/MyDrive/SLA_Project/native_wikitext.txt",

    "Native_English_ICNALE":
        "/content/drive/MyDrive/SLA_Project/ens_icnale_clean.txt",

    "Chinese_ICNALE":
        "/content/drive/MyDrive/SLA_Project/chinese_icnale_clean.txt",

    "Japanese_ICNALE":
        "/content/drive/MyDrive/SLA_Project/japanese_icnale_clean.txt",

    "Korean_ICNALE":
        "/content/drive/MyDrive/SLA_Project/kor_icnale_clean.txt",

    "Thai_ICNALE":
        "/content/drive/MyDrive/SLA_Project/tha_icnale_clean.txt",

    "Filipino_ICNALE":
        "/content/drive/MyDrive/SLA_Project/phl_icnale_clean.txt",

    "Urdu_ICNALE":
        "/content/drive/MyDrive/SLA_Project/urdu_icnale_clean.txt",
}

MAX_ESSAYS = 200
MAX_LENGTH = 128

print("=" * 90)
print("ALL BABYROBERTA MODEL PSEUDO-PERPLEXITY EVALUATION")
print("=" * 90)
print("Device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


# =============================================================================
# PSEUDO-PERPLEXITY FUNCTION
# =============================================================================

def compute_pseudo_perplexity(
    text,
    tokenizer,
    model,
    max_length=128
):
    """
    Masks one token at a time and measures the probability assigned
    to the original token.

    Lower pseudo-perplexity means the model is less surprised by the text.
    """

    encoded = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=max_length
    )

    input_ids = encoded["input_ids"].to(device)
    attention_mask = encoded["attention_mask"].to(device)

    seq_len = input_ids.size(1)

    log_likelihood = 0.0
    token_count = 0

    special_ids = set(
        token_id
        for token_id in [
            tokenizer.pad_token_id,
            tokenizer.bos_token_id,
            tokenizer.eos_token_id,
            tokenizer.cls_token_id,
            tokenizer.sep_token_id,
        ]
        if token_id is not None
    )

    with torch.no_grad():
        for position in range(seq_len):

            original_token_id = input_ids[0, position].item()

            # Do not evaluate padding or special tokens
            if attention_mask[0, position].item() == 0:
                continue

            if original_token_id in special_ids:
                continue

            masked_input = input_ids.clone()
            masked_input[0, position] = tokenizer.mask_token_id

            outputs = model(
                input_ids=masked_input,
                attention_mask=attention_mask
            )

            position_logits = outputs.logits[0, position]
            log_probs = torch.log_softmax(
                position_logits,
                dim=-1
            )

            log_likelihood += log_probs[
                original_token_id
            ].item()

            token_count += 1

    if token_count == 0:
        return None

    average_negative_log_likelihood = (
        -log_likelihood / token_count
    )

    # Prevent numerical overflow
    if average_negative_log_likelihood > 700:
        return None

    return math.exp(average_negative_log_likelihood)


# =============================================================================
# LOAD EVALUATION ESSAYS
# =============================================================================

def load_essays(group, path, max_essays=200):

    if not os.path.exists(path):
        raise FileNotFoundError(
            f"Evaluation file not found: {path}"
        )

    with open(
        path,
        "r",
        encoding="utf-8",
        errors="ignore"
    ) as file:
        text = file.read()

    if group == "Chinese_ICNALE":
        essays = re.split(r"(?=I\s)", text)

        essays = [
            essay.strip()
            for essay in essays
            if len(essay.split()) > 50
        ]

    else:
        essays = [
            line.strip()
            for line in text.splitlines()
            if len(line.split()) > 5
        ]

    return essays[:max_essays]


# Load each dataset once so every model receives exactly the same texts
evaluation_essays = {}

for group, path in essay_files.items():
    evaluation_essays[group] = load_essays(
        group,
        path,
        MAX_ESSAYS
    )

    print(
        f"{group:25s}: "
        f"{len(evaluation_essays[group])} essays"
    )


# =============================================================================
#  EVALUATE ALL SIX MODELS
# =============================================================================

all_results = {}
count_results = {}

for model_name, model_path in models.items():

    print("\n" + "=" * 90)
    print("Loading model:", model_name)
    print("Path:", model_path)
    print("=" * 90)

    if model_path != "bert-base-uncased":
        if not os.path.exists(model_path):
            raise FileNotFoundError(
                f"Model path does not exist: {model_path}"
            )

    tokenizer = AutoTokenizer.from_pretrained(
        model_path
    )

    model = AutoModelForMaskedLM.from_pretrained(
        model_path
    ).to(device)

    model.eval()

    model_results = {}
    model_counts = {}

    for group, essays in evaluation_essays.items():

        print(f"\nEvaluating {group}...")

        ppl_scores = []

        for essay in tqdm(
            essays,
            desc=f"{model_name} | {group}"
        ):
            ppl = compute_pseudo_perplexity(
                text=essay,
                tokenizer=tokenizer,
                model=model,
                max_length=MAX_LENGTH
            )

            if (
                ppl is not None
                and np.isfinite(ppl)
            ):
                ppl_scores.append(ppl)

        if len(ppl_scores) == 0:
            average_ppl = np.nan
        else:
            average_ppl = float(
                np.mean(ppl_scores)
            )

        model_results[group] = average_ppl
        model_counts[group] = len(ppl_scores)

        print(
            f"{group} Average Pseudo-PPL: "
            f"{average_ppl:.3f}"
        )

        print(
            f"Valid essays used: "
            f"{len(ppl_scores)}/{len(essays)}"
        )

    all_results[model_name] = model_results
    count_results[model_name] = model_counts

    # Release GPU memory before loading the next model
    del model
    del tokenizer
    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()


# =============================================================================
# FULL RESULTS
# =============================================================================

full_results_df = pd.DataFrame(
    all_results
).T

full_results_df.index.name = "Model"

full_counts_df = pd.DataFrame(
    count_results
).T

full_counts_df.index.name = "Model"

print("\n" + "=" * 90)
print("FULL PSEUDO-PERPLEXITY RESULTS")
print("=" * 90)

display(
    full_results_df.round(3)
)

FULL_RESULTS_FILE = (

    f"{PROJECT_DIR}/"

    "pseudo_perplexity_all_models_full_results.csv"

)

FULL_COUNTS_FILE = (

    f"{PROJECT_DIR}/"

    "pseudo_perplexity_all_models_valid_counts.csv"

)

full_results_df.to_csv(
    FULL_RESULTS_FILE
)

full_counts_df.to_csv(
    FULL_COUNTS_FILE
)


# =============================================================================
# PAPER TABLE 3
# EN = Native English
# ZH = Chinese ICNALE
# JA = Japanese ICNALE
# TH = Thai ICNALE
# FIL = Filipino ICNALE
# =============================================================================

table3_df = full_results_df[
    [
        "Native_English",
        "Chinese_ICNALE",
        "Japanese_ICNALE",
        "Thai_ICNALE",
        "Filipino_ICNALE",
    ]
].copy()

table3_df.columns = [
    "EN",
    "ZH",
    "JA",
    "TH",
    "FIL"
]

table3_df = table3_df.round(2)

print("\n" + "=" * 90)
print("TABLE 3 — PAPER-READY PSEUDO-PERPLEXITY RESULTS")
print("Lower scores indicate better language-model fit.")
print("=" * 90)

display(table3_df)

TABLE3_FILE = (
    f"{PROJECT_DIR}/"
    "table3_pseudo_perplexity_all_models.csv"
)
table3_df.to_csv(
    TABLE3_FILE
)


# =============================================================================
# LATEX VERSION FOR OVERLEAF
# =============================================================================

latex_table = table3_df.to_latex(
    float_format="%.2f",
    caption=(
        "Pseudo-perplexity scores for each model across native English "
        "and different learner language groups. Lower scores indicate "
        "better language-modeling fit."
    ),
    label="tab:pseudo_perplexity",
    position="ht",
)

LATEX_FILE = (
    f"{PROJECT_DIR}/"
    "table3_pseudo_perplexity_all_models.tex"
)

with open(
    LATEX_FILE,
    "w",
    encoding="utf-8"
) as file:
    file.write(latex_table)

print("\nSaved full results to:")
print(FULL_RESULTS_FILE)

print("\nSaved valid essay counts to:")
print(FULL_COUNTS_FILE)

print("\nSaved paper-ready Table 3 CSV to:")
print(TABLE3_FILE)

print("\nSaved Overleaf LaTeX table to:")
print(LATEX_FILE)

print("\nLaTeX preview:")
print(latex_table)

ALL BABYROBERTA MODEL PSEUDO-PERPLEXITY EVALUATION
Device: cuda
GPU: NVIDIA A100-SXM4-80GB
Native_English           : 200 essays
Native_English_ICNALE    : 200 essays
Chinese_ICNALE           : 200 essays
Japanese_ICNALE          : 200 essays
Korean_ICNALE            : 200 essays
Thai_ICNALE              : 200 essays
Filipino_ICNALE          : 200 essays
Urdu_ICNALE              : 200 essays

Loading model: English BERT Base
Path: bert-base-uncased


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

...

Urdu_ICNALE Average Pseudo-PPL: 102.620
Valid essays used: 200/200

FULL PSEUDO-PERPLEXITY RESULTS


,Native_English,Native_English_ICNALE,Chinese_ICNALE,Japanese_ICNALE,Korean_ICNALE,Thai_ICNALE,Filipino_ICNALE,Urdu_ICNALE
Model,,,,,,,,
English BERT Base,15.097,3.741,7.001,100.504,14.533,468.026,30.260,11.813
English BabyRoBERTa,481.974,148.084,127.344,114.487,119.391,141.926,128.348,140.881
English BabyRoBERTa + MUSE,466.676,151.328,129.453,116.183,120.504,143.454,129.903,142.284
Chinese BabyRoBERTa,717.502,168.742,82.214,140.183,125.348,518.487,309.541,224.533
English BabyRoBERTa + Essays,543.043,162.285,138.668,125.615,130.112,160.610,144.138,155.584
BabyRoBERTa Mix 70/30,391.128,129.056,97.510,85.209,88.250,103.233,97.012,102.620



TABLE 3 — PAPER-READY PSEUDO-PERPLEXITY RESULTS
Lower scores indicate better language-model fit.


,EN,ZH,JA,TH,FIL
Model,,,,,
English BERT Base,15.10,7.00,100.50,468.03,30.26
English BabyRoBERTa,481.97,127.34,114.49,141.93,128.35
English BabyRoBERTa + MUSE,466.68,129.45,116.18,143.45,129.90
Chinese BabyRoBERTa,717.50,82.21,140.18,518.49,309.54
English BabyRoBERTa + Essays,543.04,138.67,125.61,160.61,144.14
BabyRoBERTa Mix 70/30,391.13,97.51,85.21,103.23,97.01



Saved full results to:
/content/drive/MyDrive/BabyLM_Chinese_English_Translation/pseudo_perplexity_all_models_full_results.csv

Saved valid essay counts to:
/content/drive/MyDrive/BabyLM_Chinese_English_Translation/pseudo_perplexity_all_models_valid_counts.csv

Saved paper-ready Table 3 CSV to:
/content/drive/MyDrive/BabyLM_Chinese_English_Translation/table3_pseudo_perplexity_all_models.csv

Saved Overleaf LaTeX table to:
/content/drive/MyDrive/BabyLM_Chinese_English_Translation/table3_pseudo_perplexity_all_models.tex

LaTeX preview:
\begin{table}[ht]
\caption{Pseudo-perplexity scores for each model across native English and different learner language groups. Lower scores indicate better language-modeling fit.}
\label{tab:pseudo_perplexity}
\begin{tabular}{lrrrrr}
\toprule
 & EN & ZH & JA & TH & FIL \\
Model &  &  &  &  &  \\
\midrule
English BERT Base & 15.10 & 7.00 & 100.50 & 468.03 & 30.26 \\
English BabyRoBERTa & 481.97 & 127.34 & 114.49 & 141.93 & 128.35 \\
English BabyRoBERTa + 

In [ ]:
import os
import re
import pandas as pd

# =============================================================================
# PATHS
# =============================================================================

PROJECT_DIR = "/content/drive/MyDrive/BabyLM_Chinese_English_Translation"

GOLD_CSV = "/content/drive/MyDrive/gold.csv"
QUESTION_EXCEL = "/content/drive/MyDrive/mandarin_Questions_Full.xlsx"

MODEL_PATHS = {
    "English BERT Base":
        f"{PROJECT_DIR}/bert_base_english_preposition_eval/"
        "bert_base_english_normalized_scores.csv",

    "English BabyRoBERTa":
        f"{PROJECT_DIR}/babyroberta_english_preposition_eval/"
        "babyroberta_english_normalized_scores.csv",

    "English BabyRoBERTa + MUSE":
        f"{PROJECT_DIR}/babyroberta_english_word_muse_preposition_eval/"
        "babyroberta_english_word_muse_normalized_scores.csv",

    "Chinese BabyRoBERTa":
        f"{PROJECT_DIR}/babyroberta_108M_preposition_eval/"
        "babyroberta_108M_normalized_scores.csv",

    "English BabyRoBERTa + Essays":
        f"{PROJECT_DIR}/babyroberta_english_essays_preposition_eval/"
        "babyroberta_english_essays_normalized_scores.csv",

    "BabyRoBERTa Mix 70/30":
        f"{PROJECT_DIR}/babyroberta_mix_70_30_preposition_eval/"
        "babyroberta_mix_70_30_normalized_scores.csv",
}

# =============================================================================
# NORMALIZE QUESTION TEXT
# =============================================================================

def normalize_question(text):
    text = str(text).lower()

    text = text.replace("，", ",")
    text = text.replace("’", "'")
    text = text.replace("`", "'")

    # Make all blank/mask forms identical
    text = text.replace("[mask]", " mask ")
    text = text.replace("<mask>", " mask ")
    text = text.replace("____", " mask ")
    text = text.replace("___", " mask ")
    text = text.replace("__", " mask ")

    text = re.sub(r"[^\w\s]", " ", text)
    text = re.sub(r"\s+", " ", text)

    return text.strip()


# =============================================================================
# LOAD HUMAN RESPONSE DATA
# =============================================================================

gold = pd.read_csv(GOLD_CSV)

gold["question_norm"] = gold["question"].apply(normalize_question)
gold["preposition"] = (
    gold["preposition"]
    .astype(str)
    .str.strip()
    .str.lower()
)
gold["Precent"] = pd.to_numeric(gold["Precent"], errors="coerce")

# Human top choice for each question
human_top = (
    gold.sort_values(
        ["question_norm", "Precent"],
        ascending=[True, False]
    )
    .groupby("question_norm", as_index=False)
    .first()
    [["question_norm", "preposition", "Precent"]]
    .rename(columns={"preposition": "human_top_choice"})
)

print("Human questions in gold.csv:", len(human_top))


# =============================================================================
# LOAD GRAMMATICALLY CORRECT ANSWERS
# Needed only to identify Table 5 wrong-dominant cases
# =============================================================================

correct_data = pd.read_excel(QUESTION_EXCEL)

question_col = "Question.1"
prep_col = "Preposition"
correct_col = "Correct"

correct_data["question_norm"] = (
    correct_data[question_col].apply(normalize_question)
)

correct_data[prep_col] = (
    correct_data[prep_col]
    .astype(str)
    .str.strip()
    .str.lower()
)

correct_data[correct_col] = pd.to_numeric(
    correct_data[correct_col],
    errors="coerce"
)

correct_answers = (
    correct_data[correct_data[correct_col] == 1]
    .drop_duplicates("question_norm")
    [["question_norm", prep_col]]
    .rename(columns={prep_col: "grammatical_answer"})
)

human_top = human_top.merge(
    correct_answers,
    on="question_norm",
    how="left"
)

human_top["human_top_is_incorrect"] = (
    human_top["human_top_choice"]
    != human_top["grammatical_answer"]
)


# =============================================================================
# EVALUATE ONE MODEL
# =============================================================================

def evaluate_model(model_name, model_path):
    model = pd.read_csv(model_path)

    sentence_col = (
        "sentence" if "sentence" in model.columns else "Question"
    )

    model["question_norm"] = model[sentence_col].apply(
        normalize_question
    )

    excluded = {
        "sentence",
        "Question",
        "question_norm",
    }

    probability_columns = [
        column
        for column in model.columns
        if column not in excluded
    ]

    # Convert all probability columns to numeric
    model[probability_columns] = model[
        probability_columns
    ].apply(pd.to_numeric, errors="coerce")

    model["model_choice"] = model[
        probability_columns
    ].idxmax(axis=1)

    model["model_choice"] = (
        model["model_choice"]
        .astype(str)
        .str.strip()
        .str.lower()
    )

    # Exact question matching
    comparison = human_top.merge(
        model[["question_norm", "model_choice"]],
        on="question_norm",
        how="inner"
    )

    # -------------------------------------------------------------------------
    # TABLE 4:
    # Agreement with the most common Mandarin-learner answer
    # -------------------------------------------------------------------------

    table4_correct = (
        comparison["model_choice"]
        == comparison["human_top_choice"]
    )

    table4_total = len(comparison)
    table4_count = int(table4_correct.sum())
    table4_accuracy = (
        table4_count / table4_total
        if table4_total else 0
    )

    # -------------------------------------------------------------------------
    # TABLE 5:
    # Keep only questions where the learners' top answer was incorrect.
    # Credit model when it chooses that same incorrect answer.
    # -------------------------------------------------------------------------

    wrong_cases = comparison[
        comparison["human_top_is_incorrect"]
    ].copy()

    table5_correct = (
        wrong_cases["model_choice"]
        == wrong_cases["human_top_choice"]
    )

    table5_total = len(wrong_cases)
    table5_count = int(table5_correct.sum())
    table5_accuracy = (
        table5_count / table5_total
        if table5_total else 0
    )

    return {
        "Model": model_name,
        "Matched questions": table4_total,
        "Table 4 correct": table4_count,
        "Table 4 accuracy": table4_accuracy,
        "Wrong-dominant questions": table5_total,
        "Table 5 correct": table5_count,
        "Table 5 accuracy": table5_accuracy,
    }


# =============================================================================
# RUN ALL MODELS
# =============================================================================

results = []

for model_name, model_path in MODEL_PATHS.items():
    print(f"Evaluating {model_name}...")
    results.append(
        evaluate_model(model_name, model_path)
    )

results_df = pd.DataFrame(results)

print("\n" + "=" * 90)
print("CORRECTED TABLE 4 — AGREEMENT WITH HUMAN TOP CHOICE")
print("=" * 90)

for _, row in results_df.iterrows():
    print(
        f"{row['Model']:35s}: "
        f"{row['Table 4 accuracy']:.4f} "
        f"({int(row['Table 4 correct'])}/"
        f"{int(row['Matched questions'])})"
    )

print("\n" + "=" * 90)
print("CORRECTED TABLE 5 — SAME INCORRECT CHOICE AS HUMANS")
print("=" * 90)

for _, row in results_df.iterrows():
    print(
        f"{row['Model']:35s}: "
        f"{row['Table 5 accuracy']:.4f} "
        f"({int(row['Table 5 correct'])}/"
        f"{int(row['Wrong-dominant questions'])})"
    )

OUTPUT = f"{PROJECT_DIR}/corrected_preposition_results_all_models.csv"
results_df.to_csv(OUTPUT, index=False)

print("\nSaved to:")
print(OUTPUT)

Human questions in gold.csv: 100
Evaluating English BERT Base...
Evaluating English BabyRoBERTa...
Evaluating English BabyRoBERTa + MUSE...
Evaluating Chinese BabyRoBERTa...
Evaluating English BabyRoBERTa + Essays...
Evaluating BabyRoBERTa Mix 70/30...

CORRECTED TABLE 4 — AGREEMENT WITH HUMAN TOP CHOICE
English BERT Base                  : 0.2500 (25/100)
English BabyRoBERTa                : 0.3000 (30/100)
English BabyRoBERTa + MUSE         : 0.3000 (30/100)
Chinese BabyRoBERTa                : 0.2000 (20/100)
English BabyRoBERTa + Essays       : 0.3100 (31/100)
BabyRoBERTa Mix 70/30              : 0.2600 (26/100)

CORRECTED TABLE 5 — SAME INCORRECT CHOICE AS HUMANS
English BERT Base                  : 0.2347 (23/98)
English BabyRoBERTa                : 0.3061 (30/98)
English BabyRoBERTa + MUSE         : 0.3061 (30/98)
Chinese BabyRoBERTa                : 0.1939 (19/98)
English BabyRoBERTa + Essays       : 0.3163 (31/98)
BabyRoBERTa Mix 70/30              : 0.2653 (26/98)

Saved to:


In [3]:
import os

PROJECT_DIR = "/content/drive/MyDrive/BabyLM_Chinese_English_Translation"

CHINESE_SOURCE = (
    f"{PROJECT_DIR}/chinese_train_correct.txt"
)

ENGLISH_SOURCE = (
    f"{PROJECT_DIR}/babylm_english_matched_108M_words.txt"
)

print("=" * 90)
print("BABY_EN-ZH — VERIFYING ORIGINAL CHINESE AND ENGLISH CORPORA")
print("=" * 90)

print("Original Chinese source:")
print(CHINESE_SOURCE)
print("Exists:", os.path.exists(CHINESE_SOURCE))

print("\nOriginal English source:")
print(ENGLISH_SOURCE)
print("Exists:", os.path.exists(ENGLISH_SOURCE))


def inspect_text_file(path, sample_count=5):
    nonempty_lines = 0
    word_count = 0
    samples = []

    with open(
        path,
        "r",
        encoding="utf-8",
        errors="ignore"
    ) as file:
        for line in file:
            text = line.strip()

            if not text:
                continue

            nonempty_lines += 1
            word_count += len(text.split())

            if len(samples) < sample_count:
                samples.append(text)

    size_mb = os.path.getsize(path) / (1024 ** 2)

    return {
        "lines": nonempty_lines,
        "words_by_whitespace": word_count,
        "size_mb": size_mb,
        "samples": samples,
    }


if not os.path.exists(CHINESE_SOURCE):
    raise FileNotFoundError(
        f"Chinese corpus not found: {CHINESE_SOURCE}"
    )

if not os.path.exists(ENGLISH_SOURCE):
    raise FileNotFoundError(
        f"English corpus not found: {ENGLISH_SOURCE}"
    )


print("\n" + "-" * 90)
print("Inspecting original Chinese corpus...")

chinese_info = inspect_text_file(CHINESE_SOURCE)

print("Non-empty lines      :", f"{chinese_info['lines']:,}")
print("Whitespace word count:", f"{chinese_info['words_by_whitespace']:,}")
print("File size MB         :", f"{chinese_info['size_mb']:.2f}")

print("\nFirst 5 Chinese examples:")
for i, text in enumerate(chinese_info["samples"], start=1):
    print(f"{i}: {text}")


print("\n" + "-" * 90)
print("Inspecting original English corpus...")

english_info = inspect_text_file(ENGLISH_SOURCE)

print("Non-empty lines      :", f"{english_info['lines']:,}")
print("Whitespace word count:", f"{english_info['words_by_whitespace']:,}")
print("File size MB         :", f"{english_info['size_mb']:.2f}")

print("\nFirst 5 English examples:")
for i, text in enumerate(english_info["samples"], start=1):
    print(f"{i}: {text}")

BABY_EN-ZH — VERIFYING ORIGINAL CHINESE AND ENGLISH CORPORA
Original Chinese source:
/content/drive/MyDrive/BabyLM_Chinese_English_Translation/chinese_train_correct.txt
Exists: True

Original English source:
/content/drive/MyDrive/BabyLM_Chinese_English_Translation/babylm_english_matched_108M_words.txt
Exists: True

------------------------------------------------------------------------------------------
Inspecting original Chinese corpus...
Non-empty lines      : 629,715
Whitespace word count: 11,395,527
File size MB         : 375.50

First 5 Chinese examples:
1: 来自南方都市报播报武汉大学老牌坊被撞损一事持续引发关注 六月八十日 南都记者从武汉市公安局洪山分局一位工作人员处获悉 撞损牌坊的肇事司机因涉嫌过失损毁文物罪被依法刑事立案 南都此前报道 六月七日 武汉大学官方微博发布说明 位于洪山区街道口区域劝业场的武汉大学老牌坊被附近工地施工的混凝土搅拌车碰擦 导致部分受损 该老牌坊建于一九三四年 是全国重点保护文物 武汉大学早期建筑之一 文保部门已紧急为受损老牌坊搭设了临时支护措施 并启动抢险加固和文物修复工作 六月六日 洪山公安分局工作人员向南都记者介绍 肇事车辆已被交通大队查扣 待调查取证后将定责赔偿
2: 来关注时政方面八月二十一号国新办举行发布会 聚焦农村饮水安全脱贫攻坚问题 水利部副部长田学斌介绍 到今年六月底 按照现行标准 贫困人口饮水安全问题已得到全面解决 八成以上的农村人口用上了自来水 水质明显得到改善 告别了吃水发愁 缺水找水的历史 据了解 截至二零一九年底 还有二点五万贫困人口饮水问题未

Build a 70/30 tokenizer-training sample

In [4]:
import os
import random

print("=" * 90)
print("BABY_EN-ZH — PREPARING 70/30 DATA FOR JOINT TOKENIZER")
print("=" * 90)

SEED = 42
CHUNK_CHARACTERS = 256

# Use all available Chinese chunks and match English to preserve 70/30
CHINESE_CHUNK_TARGET = 545_227
ENGLISH_CHUNK_TARGET = 233_669

TOKENIZER_TRAIN_FILE = (
    f"{PROJECT_DIR}/baby_en_zh_tokenizer_train_70_30.txt"
)

rng = random.Random(SEED)


def create_text_chunks(
    source_path,
    target_chunks,
    chunk_characters=256
):
    chunks = []
    buffer = ""

    with open(
        source_path,
        "r",
        encoding="utf-8",
        errors="ignore"
    ) as source:
        for line in source:
            text = " ".join(line.strip().split())

            if not text:
                continue

            buffer += text + " "

            while len(buffer) >= chunk_characters:
                chunk = buffer[:chunk_characters].strip()
                buffer = buffer[chunk_characters:]

                if chunk:
                    chunks.append(chunk)

                if len(chunks) >= target_chunks:
                    return chunks

    if buffer.strip() and len(chunks) < target_chunks:
        chunks.append(buffer.strip())

    return chunks


print("Creating original Chinese chunks...")

chinese_chunks = create_text_chunks(
    CHINESE_SOURCE,
    CHINESE_CHUNK_TARGET,
    CHUNK_CHARACTERS
)

print("Chinese chunks created:", f"{len(chinese_chunks):,}")


print("\nCreating original English chunks...")

english_chunks = create_text_chunks(
    ENGLISH_SOURCE,
    ENGLISH_CHUNK_TARGET,
    CHUNK_CHARACTERS
)

print("English chunks created:", f"{len(english_chunks):,}")


if len(chinese_chunks) != CHINESE_CHUNK_TARGET:
    raise ValueError(
        f"Expected {CHINESE_CHUNK_TARGET:,} Chinese chunks, "
        f"but created {len(chinese_chunks):,}."
    )

if len(english_chunks) != ENGLISH_CHUNK_TARGET:
    raise ValueError(
        f"Expected {ENGLISH_CHUNK_TARGET:,} English chunks, "
        f"but created {len(english_chunks):,}."
    )


mixed_chunks = (
    [("ZH", text) for text in chinese_chunks]
    + [("EN", text) for text in english_chunks]
)

rng.shuffle(mixed_chunks)

with open(
    TOKENIZER_TRAIN_FILE,
    "w",
    encoding="utf-8"
) as output:
    for _, text in mixed_chunks:
        output.write(text + "\n")


zh_count = sum(
    1 for language, _ in mixed_chunks
    if language == "ZH"
)

en_count = sum(
    1 for language, _ in mixed_chunks
    if language == "EN"
)

print("\nTokenizer training file created:")
print(TOKENIZER_TRAIN_FILE)

print("\nFinal composition:")
print(
    "Original Chinese chunks:",
    f"{zh_count:,}",
    f"({zh_count / len(mixed_chunks):.1%})"
)
print(
    "Original English chunks:",
    f"{en_count:,}",
    f"({en_count / len(mixed_chunks):.1%})"
)
print("Total chunks           :", f"{len(mixed_chunks):,}")

print(
    "File size MB          :",
    round(
        os.path.getsize(TOKENIZER_TRAIN_FILE) / (1024 ** 2),
        2
    )
)

print("\nFirst 10 shuffled examples:")

for index, (language, text) in enumerate(
    mixed_chunks[:10],
    start=1
):
    print(f"{index}. [{language}] {text}")

BABY_EN-ZH — PREPARING 70/30 DATA FOR JOINT TOKENIZER
Creating original Chinese chunks...
Chinese chunks created: 545,227

Creating original English chunks...
English chunks created: 233,669

Tokenizer training file created:
/content/drive/MyDrive/BabyLM_Chinese_English_Translation/baby_en_zh_tokenizer_train_70_30.txt

Final composition:
Original Chinese chunks: 545,227 (70.0%)
Original English chunks: 233,669 (30.0%)
Total chunks           : 778,896
File size MB          : 433.12

First 10 shuffled examples:
1. [ZH] 艺人的演员 我问你一下你家后厨这么缺人吗 那么接下来我要告诉你们在这个舞台上有一票否决制 这没办法真的没办法别看都绿了我要说不过还真是过不了对来吧聊聊吧为什么让我通过你们 我把这个送给你 就是我们看无论角色是怎么样剧本怎么样把自己演好大家能够喜欢我们也 你们说了算我说了算 我说了算啊那就过吧 对喜剧表演我们的态度就是哪怕我们演一个路人甲我们也会把他当C位来演你喜欢表演喜欢喜剧 那你就一直做总有一天它会让你觉得你值得 有请下一位选手 我放学了 我大儿子回来了半年没看见他我藏起来给他来点惊喜 大儿子 你这半年没
2. [ZH] 多纳多尼我知道啊!在AC米兰期间共获得了5次意甲联赛冠军,3次冠军杯冠军和3次欧洲超级杯冠军。 我去!你这挺清楚的啊!这是百科脑子吧?哈哈哈。 低调低调!也是之后多纳多尼退役当了教练,你继续说! 嗯嗯,多纳多尼先是夸了一顿恒大,说现在的成绩和他们的教练是分不开的。 但的确是这样啊!教练给的政策、给的方案都是很重要的啊! 是,然后后面说,现在深足的阵容不是很完美,所以他自己也没什么信心。 这还没开始呢,就已经开始先说自己不

In [5]:
import os

from tokenizers import ByteLevelBPETokenizer
from transformers import RobertaTokenizerFast

print("=" * 90)
print("BABY_EN-ZH — TRAINING JOINT CHINESE–ENGLISH TOKENIZER")
print("=" * 90)

VOCAB_SIZE = 32_000
MIN_FREQUENCY = 2

JOINT_TOKENIZER_DIR = (
    f"{PROJECT_DIR}/babyroberta_en_zh_tokenizer_70_30"
)

os.makedirs(
    JOINT_TOKENIZER_DIR,
    exist_ok=True
)

print("Tokenizer training file:")
print(TOKENIZER_TRAIN_FILE)

print("\nOutput directory:")
print(JOINT_TOKENIZER_DIR)

print("\nTraining file exists:", os.path.exists(TOKENIZER_TRAIN_FILE))

if not os.path.exists(TOKENIZER_TRAIN_FILE):
    raise FileNotFoundError(
        f"Tokenizer training file not found: {TOKENIZER_TRAIN_FILE}"
    )

# Train a new Byte-Level BPE tokenizer
bpe_tokenizer = ByteLevelBPETokenizer()

bpe_tokenizer.train(
    files=[TOKENIZER_TRAIN_FILE],
    vocab_size=VOCAB_SIZE,
    min_frequency=MIN_FREQUENCY,
    special_tokens=[
        "<s>",
        "<pad>",
        "</s>",
        "<unk>",
        "<mask>",
    ],
    show_progress=True,
)

# Save vocab.json and merges.txt
bpe_tokenizer.save_model(
    JOINT_TOKENIZER_DIR
)

# Convert it into a Hugging Face RoBERTa tokenizer
joint_tokenizer = RobertaTokenizerFast(
    vocab_file=f"{JOINT_TOKENIZER_DIR}/vocab.json",
    merges_file=f"{JOINT_TOKENIZER_DIR}/merges.txt",
    bos_token="<s>",
    eos_token="</s>",
    sep_token="</s>",
    cls_token="<s>",
    unk_token="<unk>",
    pad_token="<pad>",
    mask_token="<mask>",
    add_prefix_space=True,
)

joint_tokenizer.model_max_length = 128

# Save complete tokenizer configuration
joint_tokenizer.save_pretrained(
    JOINT_TOKENIZER_DIR
)

print("\nTokenizer training complete.")

print("Vocabulary size :", joint_tokenizer.vocab_size)
print("BOS token       :", joint_tokenizer.bos_token)
print("EOS token       :", joint_tokenizer.eos_token)
print("PAD token       :", joint_tokenizer.pad_token)
print("UNK token       :", joint_tokenizer.unk_token)
print("MASK token      :", joint_tokenizer.mask_token)

print("\nSpecial token IDs:")
print("BOS ID :", joint_tokenizer.bos_token_id)
print("PAD ID :", joint_tokenizer.pad_token_id)
print("EOS ID :", joint_tokenizer.eos_token_id)
print("UNK ID :", joint_tokenizer.unk_token_id)
print("MASK ID:", joint_tokenizer.mask_token_id)

# -------------------------------------------------------------------------
# Sanity checks
# -------------------------------------------------------------------------

test_examples = [
    "我今天去学校学习英语",
    "This is an English sentence for tokenizer testing.",
    "我今天 go to school 学习 English",
]

print("\nTokenizer sanity checks:")

for text in test_examples:
    encoding = joint_tokenizer(
        text,
        add_special_tokens=True,
        truncation=True,
        max_length=128,
    )

    tokens = joint_tokenizer.convert_ids_to_tokens(
        encoding["input_ids"]
    )

    decoded = joint_tokenizer.decode(
        encoding["input_ids"],
        skip_special_tokens=True,
    )

    print("\nOriginal:")
    print(text)

    print("Token IDs:")
    print(encoding["input_ids"])

    print("Tokens:")
    print(tokens)

    print("Decoded:")
    print(decoded)

print("\nSaved tokenizer files:")

for filename in sorted(os.listdir(JOINT_TOKENIZER_DIR)):
    print("-", filename)

BABY_EN-ZH — TRAINING JOINT CHINESE–ENGLISH TOKENIZER
Tokenizer training file:
/content/drive/MyDrive/BabyLM_Chinese_English_Translation/baby_en_zh_tokenizer_train_70_30.txt

Output directory:
/content/drive/MyDrive/BabyLM_Chinese_English_Translation/babyroberta_en_zh_tokenizer_70_30

Training file exists: True


ValueError: Couldn't instantiate the backend tokenizer from one of: 
(1) a `tokenizers` library serialization file, 
(2) a slow tokenizer instance to convert or 
(3) an equivalent slow tokenizer class to instantiate and convert. 
You need to have sentencepiece or tiktoken installed to convert a slow tokenizer to a fast one.

In [6]:
import os

from tokenizers import Tokenizer
from transformers import RobertaTokenizerFast

print("=" * 90)
print("BABY_EN-ZH — FIXING AND SAVING JOINT TOKENIZER")
print("=" * 90)

JOINT_TOKENIZER_DIR = (
    f"{PROJECT_DIR}/babyroberta_en_zh_tokenizer_70_30"
)

VOCAB_FILE = f"{JOINT_TOKENIZER_DIR}/vocab.json"
MERGES_FILE = f"{JOINT_TOKENIZER_DIR}/merges.txt"
TOKENIZER_JSON = f"{JOINT_TOKENIZER_DIR}/tokenizer.json"

print("Vocabulary exists:", os.path.exists(VOCAB_FILE))
print("Merges exist    :", os.path.exists(MERGES_FILE))

if not os.path.exists(VOCAB_FILE):
    raise FileNotFoundError(VOCAB_FILE)

if not os.path.exists(MERGES_FILE):
    raise FileNotFoundError(MERGES_FILE)

# Save the already-trained backend tokenizer as tokenizer.json.
# bpe_tokenizer still exists from the previous cell.
bpe_tokenizer.save(TOKENIZER_JSON)

print("Tokenizer JSON saved:", os.path.exists(TOKENIZER_JSON))

# Load directly from tokenizer.json.
joint_tokenizer = RobertaTokenizerFast(
    tokenizer_file=TOKENIZER_JSON,
    bos_token="<s>",
    eos_token="</s>",
    sep_token="</s>",
    cls_token="<s>",
    unk_token="<unk>",
    pad_token="<pad>",
    mask_token="<mask>",
    add_prefix_space=True,
    model_max_length=128,
)

joint_tokenizer.save_pretrained(JOINT_TOKENIZER_DIR)

print("\nTokenizer successfully loaded and saved.")
print("Vocabulary size:", joint_tokenizer.vocab_size)
print("BOS token      :", joint_tokenizer.bos_token)
print("EOS token      :", joint_tokenizer.eos_token)
print("PAD token      :", joint_tokenizer.pad_token)
print("UNK token      :", joint_tokenizer.unk_token)
print("MASK token     :", joint_tokenizer.mask_token)

print("\nSpecial-token IDs:")
print("BOS ID :", joint_tokenizer.bos_token_id)
print("PAD ID :", joint_tokenizer.pad_token_id)
print("EOS ID :", joint_tokenizer.eos_token_id)
print("UNK ID :", joint_tokenizer.unk_token_id)
print("MASK ID:", joint_tokenizer.mask_token_id)

test_examples = [
    "我今天去学校学习英语",
    "This is an English sentence for tokenizer testing.",
    "我今天 go to school 学习 English",
]

print("\nTokenizer sanity checks:")

for text in test_examples:
    encoded = joint_tokenizer(
        text,
        add_special_tokens=True,
        truncation=True,
        max_length=128,
    )

    tokens = joint_tokenizer.convert_ids_to_tokens(
        encoded["input_ids"]
    )

    decoded = joint_tokenizer.decode(
        encoded["input_ids"],
        skip_special_tokens=True,
    )

    print("\nOriginal:")
    print(text)

    print("Tokens:")
    print(tokens)

    print("Decoded:")
    print(decoded)

print("\nSaved files:")
for filename in sorted(os.listdir(JOINT_TOKENIZER_DIR)):
    print("-", filename)

BABY_EN-ZH — FIXING AND SAVING JOINT TOKENIZER
Vocabulary exists: True
Merges exist    : True
Tokenizer JSON saved: True

Tokenizer successfully loaded and saved.
Vocabulary size: 32000
BOS token      : <s>
EOS token      : </s>
PAD token      : <pad>
UNK token      : <unk>
MASK token     : <mask>

Special-token IDs:
BOS ID : 0
PAD ID : 1
EOS ID : 2
UNK ID : 3
MASK ID: 4

Tokenizer sanity checks:

Original:
我今天去学校学习英语
Tokens:
['æĪĳä»Ĭå¤©', 'åİ»', 'åŃ¦æł¡', 'åŃ¦ä¹ł', 'èĭ±è¯Ń']
Decoded:
我今天去学校学习英语

Original:
This is an English sentence for tokenizer testing.
Tokens:
['T', 'his', 'Ġis', 'Ġan', 'ĠEnglish', 'Ġsentence', 'Ġfor', 'Ġto', 'ken', 'iz', 'er', 'Ġtest', 'ing', '.']
Decoded:
This is an English sentence for tokenizer testing.

Original:
我今天 go to school 学习 English
Tokens:
['æĪĳä»Ĭå¤©', 'Ġgo', 'Ġto', 'Ġschool', 'ĠåŃ¦ä¹ł', 'ĠEnglish']
Decoded:
我今天 go to school 学习 English

Saved files:
- merges.txt
- tokenizer.json
- tokenizer_config.json
- vocab.json


In [7]:
import os

print("=" * 90)
print("BABY_EN-ZH — CREATING 95/5 TRAIN-VALIDATION SPLIT")
print("=" * 90)

TRAIN_RATIO = 0.95

EN_ZH_TRAIN_FILE = (
    f"{PROJECT_DIR}/baby_en_zh_original_70_30_train_95.txt"
)

EN_ZH_VALID_FILE = (
    f"{PROJECT_DIR}/baby_en_zh_original_70_30_valid_5.txt"
)

# The tokenizer-training corpus is already shuffled with seed 42.
with open(
    TOKENIZER_TRAIN_FILE,
    "r",
    encoding="utf-8",
    errors="ignore"
) as source:
    all_lines = [
        line.rstrip("\n")
        for line in source
        if line.strip()
    ]

total_lines = len(all_lines)
split_index = int(total_lines * TRAIN_RATIO)

train_lines = all_lines[:split_index]
valid_lines = all_lines[split_index:]

with open(
    EN_ZH_TRAIN_FILE,
    "w",
    encoding="utf-8"
) as output:
    for line in train_lines:
        output.write(line + "\n")

with open(
    EN_ZH_VALID_FILE,
    "w",
    encoding="utf-8"
) as output:
    for line in valid_lines:
        output.write(line + "\n")

print("Total mixed examples :", f"{total_lines:,}")
print("Training examples    :", f"{len(train_lines):,}")
print("Validation examples  :", f"{len(valid_lines):,}")

print("\nTraining file:")
print(EN_ZH_TRAIN_FILE)

print("\nValidation file:")
print(EN_ZH_VALID_FILE)

print("\nFiles created:")
print("Train exists:", os.path.exists(EN_ZH_TRAIN_FILE))
print("Valid exists:", os.path.exists(EN_ZH_VALID_FILE))

print(
    "Train size MB:",
    round(
        os.path.getsize(EN_ZH_TRAIN_FILE) / (1024 ** 2),
        2
    )
)

print(
    "Valid size MB:",
    round(
        os.path.getsize(EN_ZH_VALID_FILE) / (1024 ** 2),
        2
    )
)

print("\nFirst 3 training examples:")
for index, text in enumerate(train_lines[:3], start=1):
    print(f"{index}: {text[:300]}")

print("\nFirst 3 validation examples:")
for index, text in enumerate(valid_lines[:3], start=1):
    print(f"{index}: {text[:300]}")

BABY_EN-ZH — CREATING 95/5 TRAIN-VALIDATION SPLIT
Total mixed examples : 778,896
Training examples    : 739,951
Validation examples  : 38,945

Training file:
/content/drive/MyDrive/BabyLM_Chinese_English_Translation/baby_en_zh_original_70_30_train_95.txt

Validation file:
/content/drive/MyDrive/BabyLM_Chinese_English_Translation/baby_en_zh_original_70_30_valid_5.txt

Files created:
Train exists: True
Valid exists: True
Train size MB: 411.49
Valid size MB: 21.64

First 3 training examples:
1: 艺人的演员 我问你一下你家后厨这么缺人吗 那么接下来我要告诉你们在这个舞台上有一票否决制 这没办法真的没办法别看都绿了我要说不过还真是过不了对来吧聊聊吧为什么让我通过你们 我把这个送给你 就是我们看无论角色是怎么样剧本怎么样把自己演好大家能够喜欢我们也 你们说了算我说了算 我说了算啊那就过吧 对喜剧表演我们的态度就是哪怕我们演一个路人甲我们也会把他当C位来演你喜欢表演喜欢喜剧 那你就一直做总有一天它会让你觉得你值得 有请下一位选手 我放学了 我大儿子回来了半年没看见他我藏起来给他来点惊喜 大儿子 你这半年没
2: 多纳多尼我知道啊!在AC米兰期间共获得了5次意甲联赛冠军,3次冠军杯冠军和3次欧洲超级杯冠军。 我去!你这挺清楚的啊!这是百科脑子吧?哈哈哈。 低调低调!也是之后多纳多尼退役当了教练,你继续说! 嗯嗯,多纳多尼先是夸了一顿恒大,说现在的成绩和他们的教练是分不开的。 但的确是这样啊!教练给的政策、给的方案都是很重要的啊! 是,然后后面说,现在深足的阵容不是很完美,所以他自己也没什么信心。 这还没开始呢,就已经开始先说自己不行了吗?不知道这是不是烟雾弹呀! 就是说啊!更像是烟雾弹呢!主

In [8]:
from datasets import load_dataset

print("=" * 90)
print("BABY_EN-ZH — LOADING AND TOKENIZING TRAINING DATA")
print("=" * 90)

BLOCK_SIZE = 128

raw_en_zh_datasets = load_dataset(
    "text",
    data_files={
        "train": EN_ZH_TRAIN_FILE,
        "validation": EN_ZH_VALID_FILE,
    },
)

print(raw_en_zh_datasets)


def tokenize_en_zh_batch(examples):
    return joint_tokenizer(
        examples["text"],
        truncation=True,
        max_length=BLOCK_SIZE,
        padding="max_length",
        return_special_tokens_mask=True,
    )


tokenized_en_zh_datasets = raw_en_zh_datasets.map(
    tokenize_en_zh_batch,
    batched=True,
    batch_size=1000,
    num_proc=2,
    remove_columns=["text"],
)

print("\nTokenization complete.")
print(tokenized_en_zh_datasets)

print("\nTraining examples  :", f"{len(tokenized_en_zh_datasets['train']):,}")
print("Validation examples:", f"{len(tokenized_en_zh_datasets['validation']):,}")
print("Sequence length    :", BLOCK_SIZE)

print("\nFirst tokenized training example length:")
print(len(tokenized_en_zh_datasets["train"][0]["input_ids"]))

BABY_EN-ZH — LOADING AND TOKENIZING TRAINING DATA


Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text'],
        num_rows: 739951
    })
    validation: Dataset({
        features: ['text'],
        num_rows: 38945
    })
})


Map (num_proc=2):   0%|          | 0/739951 [00:00<?, ? examples/s]

Map (num_proc=2):   0%|          | 0/38945 [00:00<?, ? examples/s]


Tokenization complete.
DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask', 'special_tokens_mask'],
        num_rows: 739951
    })
    validation: Dataset({
        features: ['input_ids', 'attention_mask', 'special_tokens_mask'],
        num_rows: 38945
    })
})

Training examples  : 739,951
Validation examples: 38,945
Sequence length    : 128

First tokenized training example length:
128


In [9]:
import torch
from transformers import RobertaConfig, RobertaForMaskedLM

print("=" * 90)
print("BABY_EN-ZH — CREATING FRESH BABYROBERTA MODEL")
print("=" * 90)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

en_zh_config = RobertaConfig(
    vocab_size=joint_tokenizer.vocab_size,
    max_position_embeddings=130,
    hidden_size=256,
    num_hidden_layers=4,
    num_attention_heads=4,
    intermediate_size=1024,
    hidden_act="gelu",
    hidden_dropout_prob=0.1,
    attention_probs_dropout_prob=0.1,
    type_vocab_size=1,
    pad_token_id=joint_tokenizer.pad_token_id,
    bos_token_id=joint_tokenizer.bos_token_id,
    eos_token_id=joint_tokenizer.eos_token_id,
)

en_zh_model = RobertaForMaskedLM(
    en_zh_config
)

en_zh_model.to(device)

total_parameters = sum(
    parameter.numel()
    for parameter in en_zh_model.parameters()
)

trainable_parameters = sum(
    parameter.numel()
    for parameter in en_zh_model.parameters()
    if parameter.requires_grad
)

print("Device              :", device)
print("Tokenizer vocab size:", joint_tokenizer.vocab_size)
print("Model vocab size    :", en_zh_config.vocab_size)
print("Hidden size         :", en_zh_config.hidden_size)
print("Transformer layers  :", en_zh_config.num_hidden_layers)
print("Attention heads     :", en_zh_config.num_attention_heads)
print("Intermediate size   :", en_zh_config.intermediate_size)
print("Max positions       :", en_zh_config.max_position_embeddings)

print("\nTotal parameters    :", f"{total_parameters:,}")
print("Trainable parameters:", f"{trainable_parameters:,}")
print("Parameters in M     :", round(total_parameters / 1_000_000, 2))
print("Model device        :", next(en_zh_model.parameters()).device)

if total_parameters == 11_483_392:
    print("\nArchitecture exactly matches the earlier BabyRoBERTa models.")
else:
    print(
        "\nWarning: parameter count differs from the earlier "
        "11.48M BabyRoBERTa models."
    )

BABY_EN-ZH — CREATING FRESH BABYROBERTA MODEL
Device              : cuda
Tokenizer vocab size: 32000
Model vocab size    : 32000
Hidden size         : 256
Transformer layers  : 4
Attention heads     : 4
Intermediate size   : 1024
Max positions       : 130

Total parameters    : 11,483,392
Trainable parameters: 11,483,392
Parameters in M     : 11.48
Model device        : cuda:0

Architecture exactly matches the earlier BabyRoBERTa models.


In [10]:
import os

from transformers import (
    DataCollatorForLanguageModeling,
    TrainingArguments,
    Trainer,
)

print("=" * 90)
print("BABY_EN-ZH — TRAINING SETUP")
print("=" * 90)

EN_ZH_OUTPUT_DIR = (
    f"{PROJECT_DIR}/babyroberta_en_zh_original_70_30_same_arch_95_5"
)

EN_ZH_CHECKPOINT_DIR = (
    f"{EN_ZH_OUTPUT_DIR}/checkpoints"
)

EN_ZH_FINAL_MODEL_DIR = (
    f"{EN_ZH_OUTPUT_DIR}/final_model"
)

os.makedirs(
    EN_ZH_OUTPUT_DIR,
    exist_ok=True
)

data_collator = DataCollatorForLanguageModeling(
    tokenizer=joint_tokenizer,
    mlm=True,
    mlm_probability=0.15,
)

TRAIN_BATCH_SIZE = 128
EVAL_BATCH_SIZE = 128
MAX_STEPS = 98_295
LEARNING_RATE = 5e-4
WEIGHT_DECAY = 0.01
WARMUP_STEPS = 9_830

en_zh_training_args = TrainingArguments(
    output_dir=EN_ZH_CHECKPOINT_DIR,

    do_train=True,
    do_eval=True,

    max_steps=MAX_STEPS,

    per_device_train_batch_size=TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=EVAL_BATCH_SIZE,
    gradient_accumulation_steps=1,

    learning_rate=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    warmup_steps=WARMUP_STEPS,

    logging_strategy="steps",
    logging_steps=5_000,

    eval_strategy="steps",
    eval_steps=5_000,

    save_strategy="steps",
    save_steps=5_000,
    save_total_limit=3,

    bf16=True,
    fp16=False,

    dataloader_num_workers=2,
    report_to="none",

    seed=42,
    data_seed=42,
)

en_zh_trainer = Trainer(
    model=en_zh_model,
    args=en_zh_training_args,
    train_dataset=tokenized_en_zh_datasets["train"],
    eval_dataset=tokenized_en_zh_datasets["validation"],
    data_collator=data_collator,
)

print("Training examples   :", f"{len(tokenized_en_zh_datasets['train']):,}")
print("Validation examples :", f"{len(tokenized_en_zh_datasets['validation']):,}")
print("Batch size          :", TRAIN_BATCH_SIZE)
print("Max training steps  :", f"{MAX_STEPS:,}")
print("Learning rate       :", LEARNING_RATE)
print("Warmup steps        :", f"{WARMUP_STEPS:,}")
print("MLM probability     :", 0.15)

print("\nCheckpoint directory:")
print(EN_ZH_CHECKPOINT_DIR)

print("\nFinal model directory:")
print(EN_ZH_FINAL_MODEL_DIR)

print("\nBaby_EN-ZH training setup is ready.")

BABY_EN-ZH — TRAINING SETUP
Training examples   : 739,951
Validation examples : 38,945
Batch size          : 128
Max training steps  : 98,295
Learning rate       : 0.0005
Warmup steps        : 9,830
MLM probability     : 0.15

Checkpoint directory:
/content/drive/MyDrive/BabyLM_Chinese_English_Translation/babyroberta_en_zh_original_70_30_same_arch_95_5/checkpoints

Final model directory:
/content/drive/MyDrive/BabyLM_Chinese_English_Translation/babyroberta_en_zh_original_70_30_same_arch_95_5/final_model

Baby_EN-ZH training setup is ready.


In [11]:
print("=" * 90)
print("TRAINING BABY_EN-ZH — 70% ORIGINAL CHINESE + 30% ORIGINAL ENGLISH")
print("=" * 90)

en_zh_train_result = en_zh_trainer.train()

# Save final model and tokenizer
en_zh_trainer.save_model(
    EN_ZH_FINAL_MODEL_DIR
)

joint_tokenizer.save_pretrained(
    EN_ZH_FINAL_MODEL_DIR
)

# Save metrics and trainer state
en_zh_metrics = en_zh_train_result.metrics

en_zh_trainer.log_metrics(
    "en_zh_train",
    en_zh_metrics
)

en_zh_trainer.save_metrics(
    "en_zh_train",
    en_zh_metrics
)

en_zh_trainer.save_state()

print("\nBaby_EN-ZH training complete.")

print("\nFinal model saved to:")
print(EN_ZH_FINAL_MODEL_DIR)

print("\nTraining metrics:")

for key, value in en_zh_metrics.items():
    print(f"{key}: {value}")

TRAINING BABY_EN-ZH — 70% ORIGINAL CHINESE + 30% ORIGINAL ENGLISH


Step,Training Loss,Validation Loss
5000,8.187623,7.456093
10000,7.304046,6.925048
15000,6.092568,5.367465
20000,5.329064,4.991068
25000,5.061681,4.783673
30000,4.900983,4.645048
35000,4.787296,4.554463
40000,4.698070,4.475326
45000,4.625068,4.408677
50000,4.569288,4.354389


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

***** en_zh_train metrics *****
  epoch                    =    17.0031
  total_flos               = 29317346GF
  train_loss               =     4.9715
  train_runtime            = 1:04:06.17
  train_samples_per_second =   3271.235
  train_steps_per_second   =     25.557

Baby_EN-ZH training complete.

Final model saved to:
/content/drive/MyDrive/BabyLM_Chinese_English_Translation/babyroberta_en_zh_original_70_30_same_arch_95_5/final_model

Training metrics:
train_runtime: 3846.18
train_samples_per_second: 3271.235
train_steps_per_second: 25.557
total_flos: 3.147926076378317e+16
train_loss: 4.971473237845897
epoch: 17.003113648157758


In [2]:
import os
import torch
import torch.nn.functional as F
import pandas as pd

from transformers import RobertaTokenizerFast, RobertaForMaskedLM

# =============================================================================
# PROJECT CONFIGURATION
# =============================================================================

PROJECT_DIR = "/content/drive/MyDrive/BabyLM_Chinese_English_Translation"

INPUT_FILE = f"{PROJECT_DIR}/Mandarin_Question_MASK_CLEAN.txt"
OPTIONS_FILE = f"{PROJECT_DIR}/Mandarin_answers.txt"

MODEL_DIR = (
    f"{PROJECT_DIR}/"
    "babyroberta_en_zh_original_70_30_same_arch_95_5/final_model"
)

OUTPUT_DIR = (
    f"{PROJECT_DIR}/babyroberta_en_zh_preposition_eval"
)

os.makedirs(OUTPUT_DIR, exist_ok=True)

OUTPUT_FILE = (
    f"{OUTPUT_DIR}/babyroberta_en_zh_predictions.csv"
)

NORMALIZED_FILE = (
    f"{OUTPUT_DIR}/babyroberta_en_zh_normalized_scores.csv"
)

OPTIONS = [
    "on", "at", "like", "as", "with", "inside", "of", "among", "in",
    "by", "from", "during", "about", "near", "out", "round", "until",
    "along", "for", "against", "across", "to", "off", "upon", "towards",
    "under", "around", "behind", "besides", "within", "beside", "into",
    "between", "up", "over", "before", "above", "down", "after", "till",
    "toward", "without"
]

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

# =============================================================================
# VERIFY FILES
# =============================================================================

print("=" * 90)
print("BABY_EN-ZH — PREPOSITION EVALUATION")
print("=" * 90)

print("Question file exists:", os.path.exists(INPUT_FILE))
print("Options file exists :", os.path.exists(OPTIONS_FILE))
print("Model exists        :", os.path.exists(MODEL_DIR))

if not os.path.exists(INPUT_FILE):
    raise FileNotFoundError(INPUT_FILE)

if not os.path.exists(OPTIONS_FILE):
    raise FileNotFoundError(OPTIONS_FILE)

if not os.path.exists(MODEL_DIR):
    raise FileNotFoundError(MODEL_DIR)

# =============================================================================
# LOAD BABY_EN-ZH MODEL
# =============================================================================

tokenizer = RobertaTokenizerFast.from_pretrained(
    MODEL_DIR
)

model = RobertaForMaskedLM.from_pretrained(
    MODEL_DIR
)

model.to(device)
model.eval()

print("\nBaby_EN-ZH model loaded.")
print("Using device        :", device)
print("Tokenizer vocab size:", tokenizer.vocab_size)
print("Model vocab size    :", model.config.vocab_size)
print("Mask token          :", tokenizer.mask_token)

# =============================================================================
# MASK PREDICTION FUNCTION
# =============================================================================

def predict_masked_word(sentence, options):
    inputs = tokenizer(
        sentence,
        return_tensors="pt",
        truncation=True,
        max_length=128
    )

    inputs = {
        key: value.to(device)
        for key, value in inputs.items()
    }

    input_ids = inputs["input_ids"]

    mask_positions = torch.where(
        input_ids == tokenizer.mask_token_id
    )[1]

    if mask_positions.numel() == 0:
        return {
            word: 0.0
            for word in options
        }

    with torch.no_grad():
        logits = model(**inputs).logits

    mask_logits = logits[
        0,
        mask_positions[0],
        :
    ]

    probabilities = F.softmax(
        mask_logits,
        dim=-1
    )

    word_probabilities = {}

    for word in options:
        token_ids = tokenizer.encode(
            word,
            add_special_tokens=False
        )

        if len(token_ids) == 1:
            word_probabilities[word] = (
                probabilities[token_ids[0]].item()
            )

        else:
            probability = 1.0

            for token_id in token_ids:
                probability *= (
                    probabilities[token_id].item()
                )

            word_probabilities[word] = probability

    return word_probabilities

# =============================================================================
# LOAD SENTENCES
# =============================================================================

sentences = []

with open(
    INPUT_FILE,
    "r",
    encoding="utf-8"
) as file:
    for line in file:
        line = line.strip()

        if not line:
            continue

        # Convert all mask formats to RoBERTa mask
        line = line.replace("[MASK]", "<mask>")
        line = line.replace("____", "<mask>")
        line = line.replace(" ___ ", " <mask> ")
        line = line.replace("___", "<mask>")
        line = line.replace("__", "<mask>")

        if "<mask>" not in line:
            parts = line.split()

            if len(parts) > 1:
                parts.insert(
                    -1,
                    "<mask>"
                )
                line = " ".join(parts)

            else:
                line = line + " <mask>"

        while "<mask> <mask>" in line:
            line = line.replace(
                "<mask> <mask>",
                "<mask>"
            )

        sentences.append(line)

print("\nLoaded", len(sentences), "sentences.")

# =============================================================================
# RAW PREDICTIONS
# =============================================================================

rows = []

for sentence in sentences:
    probabilities = predict_masked_word(
        sentence,
        OPTIONS
    )

    row = {
        "Question": sentence
    }

    row.update(probabilities)
    rows.append(row)

df = pd.DataFrame(rows)

df.to_csv(
    OUTPUT_FILE,
    index=False,
    encoding="utf-8-sig"
)

print("\nRaw predictions saved to:")
print(OUTPUT_FILE)

# =============================================================================
# NORMALIZED SCORES
# =============================================================================

with open(
    OPTIONS_FILE,
    "r",
    encoding="utf-8"
) as file:
    option_lines = [
        line.strip()
        for line in file
        if line.strip()
    ]

if len(option_lines) != len(sentences):
    raise ValueError(
        "OPTIONS_FILE line count must match INPUT_FILE line count."
    )

normalized_rows = []

for index, allowed in enumerate(option_lines):
    allowed_list = [
        word.strip().lower()
        for word in allowed.split(",")
        if word.strip()
    ]

    sentence = sentences[index]

    row = {
        "sentence": sentence
    }

    total = sum(
        float(df.loc[index, word])
        for word in allowed_list
        if word in df.columns
    )

    for word in OPTIONS:
        if word in allowed_list and total > 0:
            row[word] = (
                float(df.loc[index, word]) / total
            )
        else:
            row[word] = 0.0

    normalized_rows.append(row)

df_norm = pd.DataFrame(
    normalized_rows
)

df_norm.to_csv(
    NORMALIZED_FILE,
    index=False,
    encoding="utf-8-sig"
)

print("\nNormalized scores saved to:")
print(NORMALIZED_FILE)

print("\nDONE.")

BABY_EN-ZH — PREPOSITION EVALUATION
Question file exists: True
Options file exists : True
Model exists        : True


Loading weights:   0%|          | 0/74 [00:00<?, ?it/s]


Baby_EN-ZH model loaded.
Using device        : cuda
Tokenizer vocab size: 32000
Model vocab size    : 32000
Mask token          : <mask>

Loaded 100 sentences.

Raw predictions saved to:
/content/drive/MyDrive/BabyLM_Chinese_English_Translation/babyroberta_en_zh_preposition_eval/babyroberta_en_zh_predictions.csv

Normalized scores saved to:
/content/drive/MyDrive/BabyLM_Chinese_English_Translation/babyroberta_en_zh_preposition_eval/babyroberta_en_zh_normalized_scores.csv

DONE.


In [4]:
import os
import re
import math
import gc
import torch
import numpy as np
import pandas as pd
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModelForMaskedLM

# =============================================================================
# CONFIGURATION
# =============================================================================

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

PROJECT_DIR = "/content/drive/MyDrive/BabyLM_Chinese_English_Translation"

models = {
    "Baby_EN-ZH":
        f"{PROJECT_DIR}/babyroberta_en_zh_original_70_30_same_arch_95_5/final_model",
}

essay_files = {
    "Native_English":
        "/content/drive/MyDrive/SLA_Project/native_wikitext.txt",

    "Native_English_ICNALE":
        "/content/drive/MyDrive/SLA_Project/ens_icnale_clean.txt",

    "Chinese_ICNALE":
        "/content/drive/MyDrive/SLA_Project/chinese_icnale_clean.txt",

    "Japanese_ICNALE":
        "/content/drive/MyDrive/SLA_Project/japanese_icnale_clean.txt",

    "Korean_ICNALE":
        "/content/drive/MyDrive/SLA_Project/kor_icnale_clean.txt",

    "Thai_ICNALE":
        "/content/drive/MyDrive/SLA_Project/tha_icnale_clean.txt",

    "Filipino_ICNALE":
        "/content/drive/MyDrive/SLA_Project/phl_icnale_clean.txt",

    "Urdu_ICNALE":
        "/content/drive/MyDrive/SLA_Project/urdu_icnale_clean.txt",
}

MAX_ESSAYS = 200
MAX_LENGTH = 128

print("=" * 90)
print("BABY_EN-ZH PSEUDO-PERPLEXITY EVALUATION")
print("=" * 90)
print("Device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


# =============================================================================
# PSEUDO-PERPLEXITY FUNCTION
# =============================================================================

def compute_pseudo_perplexity(
    text,
    tokenizer,
    model,
    max_length=128
):
    """
    Masks one token at a time and measures the probability assigned
    to the original token.

    Lower pseudo-perplexity means the model is less surprised by the text.
    """

    encoded = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=max_length
    )

    input_ids = encoded["input_ids"].to(device)
    attention_mask = encoded["attention_mask"].to(device)

    seq_len = input_ids.size(1)

    log_likelihood = 0.0
    token_count = 0

    special_ids = set(
        token_id
        for token_id in [
            tokenizer.pad_token_id,
            tokenizer.bos_token_id,
            tokenizer.eos_token_id,
            tokenizer.cls_token_id,
            tokenizer.sep_token_id,
        ]
        if token_id is not None
    )

    with torch.no_grad():
        for position in range(seq_len):

            original_token_id = input_ids[0, position].item()

            # Do not evaluate padding or special tokens
            if attention_mask[0, position].item() == 0:
                continue

            if original_token_id in special_ids:
                continue

            masked_input = input_ids.clone()
            masked_input[0, position] = tokenizer.mask_token_id

            outputs = model(
                input_ids=masked_input,
                attention_mask=attention_mask
            )

            position_logits = outputs.logits[0, position]
            log_probs = torch.log_softmax(
                position_logits,
                dim=-1
            )

            log_likelihood += log_probs[
                original_token_id
            ].item()

            token_count += 1

    if token_count == 0:
        return None

    average_negative_log_likelihood = (
        -log_likelihood / token_count
    )

    # Prevent numerical overflow
    if average_negative_log_likelihood > 700:
        return None

    return math.exp(average_negative_log_likelihood)


# =============================================================================
# LOAD EVALUATION ESSAYS
# =============================================================================

def load_essays(group, path, max_essays=200):

    if not os.path.exists(path):
        raise FileNotFoundError(
            f"Evaluation file not found: {path}"
        )

    with open(
        path,
        "r",
        encoding="utf-8",
        errors="ignore"
    ) as file:
        text = file.read()

    if group == "Chinese_ICNALE":
        essays = re.split(r"(?=I\s)", text)

        essays = [
            essay.strip()
            for essay in essays
            if len(essay.split()) > 50
        ]

    else:
        essays = [
            line.strip()
            for line in text.splitlines()
            if len(line.split()) > 5
        ]

    return essays[:max_essays]


# Load each dataset once so every model receives exactly the same texts
evaluation_essays = {}

for group, path in essay_files.items():
    evaluation_essays[group] = load_essays(
        group,
        path,
        MAX_ESSAYS
    )

    print(
        f"{group:25s}: "
        f"{len(evaluation_essays[group])} essays"
    )


# =============================================================================
#  # =============================================================================
#  EVALUATE ALL SEVEN MODELS
# =============================================================================
# =============================================================================

all_results = {}
count_results = {}

for model_name, model_path in models.items():

    print("\n" + "=" * 90)
    print("Loading model:", model_name)
    print("Path:", model_path)
    print("=" * 90)

    if model_path != "bert-base-uncased":
        if not os.path.exists(model_path):
            raise FileNotFoundError(
                f"Model path does not exist: {model_path}"
            )

    tokenizer = AutoTokenizer.from_pretrained(
        model_path
    )

    model = AutoModelForMaskedLM.from_pretrained(
        model_path
    ).to(device)

    model.eval()

    model_results = {}
    model_counts = {}

    for group, essays in evaluation_essays.items():

        print(f"\nEvaluating {group}...")

        ppl_scores = []

        for essay in tqdm(
            essays,
            desc=f"{model_name} | {group}"
        ):
            ppl = compute_pseudo_perplexity(
                text=essay,
                tokenizer=tokenizer,
                model=model,
                max_length=MAX_LENGTH
            )

            if (
                ppl is not None
                and np.isfinite(ppl)
            ):
                ppl_scores.append(ppl)

        if len(ppl_scores) == 0:
            average_ppl = np.nan
        else:
            average_ppl = float(
                np.mean(ppl_scores)
            )

        model_results[group] = average_ppl
        model_counts[group] = len(ppl_scores)

        print(
            f"{group} Average Pseudo-PPL: "
            f"{average_ppl:.3f}"
        )

        print(
            f"Valid essays used: "
            f"{len(ppl_scores)}/{len(essays)}"
        )

    all_results[model_name] = model_results
    count_results[model_name] = model_counts

    # Release GPU memory before loading the next model
    del model
    del tokenizer
    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()


# =============================================================================
# FULL RESULTS
# =============================================================================

full_results_df = pd.DataFrame(
    all_results
).T

full_results_df.index.name = "Model"

full_counts_df = pd.DataFrame(
    count_results
).T

full_counts_df.index.name = "Model"

print("\n" + "=" * 90)
print("FULL PSEUDO-PERPLEXITY RESULTS")
print("=" * 90)

display(
    full_results_df.round(3)
)

FULL_RESULTS_FILE = (

    f"{PROJECT_DIR}/"

    "pseudo_perplexity_baby_en_zh_full_results.csv"

)

FULL_COUNTS_FILE = (

    f"{PROJECT_DIR}/"

    "pseudo_perplexity_baby_en_zh_valid_counts.csv"

)

full_results_df.to_csv(
    FULL_RESULTS_FILE
)

full_counts_df.to_csv(
    FULL_COUNTS_FILE
)


# =============================================================================
# PAPER TABLE 3
# EN = Native English
# ZH = Chinese ICNALE
# JA = Japanese ICNALE
# TH = Thai ICNALE
# FIL = Filipino ICNALE
# =============================================================================

table3_df = full_results_df[
    [
        "Native_English",
        "Chinese_ICNALE",
        "Japanese_ICNALE",
        "Thai_ICNALE",
        "Filipino_ICNALE",
    ]
].copy()

table3_df.columns = [
    "EN",
    "ZH",
    "JA",
    "TH",
    "FIL"
]

table3_df = table3_df.round(2)

print("\n" + "=" * 90)
print("TABLE 3 — PAPER-READY PSEUDO-PERPLEXITY RESULTS")
print("Lower scores indicate better language-model fit.")
print("=" * 90)

display(table3_df)

TABLE3_FILE = (

    f"{PROJECT_DIR}/"

    "table3_pseudo_perplexity_baby_en_zh.csv"

)
table3_df.to_csv(
    TABLE3_FILE
)


# =============================================================================
# LATEX VERSION FOR OVERLEAF
# =============================================================================

latex_table = table3_df.to_latex(
    float_format="%.2f",
    caption=(
        "Pseudo-perplexity scores for each model across native English "
        "and different learner language groups. Lower scores indicate "
        "better language-modeling fit."
    ),
    label="tab:pseudo_perplexity",
    position="ht",
)

LATEX_FILE = (

    f"{PROJECT_DIR}/"

    "table3_pseudo_perplexity_baby_en_zh.tex"

)
with open(
    LATEX_FILE,
    "w",
    encoding="utf-8"
) as file:
    file.write(latex_table)

print("\nSaved full results to:")
print(FULL_RESULTS_FILE)

print("\nSaved valid essay counts to:")
print(FULL_COUNTS_FILE)

print("\nSaved paper-ready Table 3 CSV to:")
print(TABLE3_FILE)

print("\nSaved Overleaf LaTeX table to:")
print(LATEX_FILE)

print("\nLaTeX preview:")
print(latex_table)

BABY_EN-ZH PSEUDO-PERPLEXITY EVALUATION
Device: cuda
GPU: NVIDIA A100-SXM4-80GB
Native_English           : 200 essays
Native_English_ICNALE    : 200 essays
Chinese_ICNALE           : 200 essays
Japanese_ICNALE          : 200 essays
Korean_ICNALE            : 200 essays
Thai_ICNALE              : 200 essays
Filipino_ICNALE          : 200 essays
Urdu_ICNALE              : 200 essays

Loading model: Baby_EN-ZH
Path: /content/drive/MyDrive/BabyLM_Chinese_English_Translation/babyroberta_en_zh_original_70_30_same_arch_95_5/final_model


Loading weights:   0%|          | 0/74 [00:00<?, ?it/s]


Evaluating Native_English...


Baby_EN-ZH | Native_English: 100%|██████████| 200/200 [01:01<00:00,  3.25it/s]


Native_English Average Pseudo-PPL: 180.740
Valid essays used: 200/200

Evaluating Native_English_ICNALE...


...

Urdu_ICNALE Average Pseudo-PPL: 130.763
Valid essays used: 200/200

FULL PSEUDO-PERPLEXITY RESULTS


,Native_English,Native_English_ICNALE,Chinese_ICNALE,Japanese_ICNALE,Korean_ICNALE,Thai_ICNALE,Filipino_ICNALE,Urdu_ICNALE
Model,,,,,,,,
Baby_EN-ZH,180.74,88.664,62.004,65.787,68.889,90.175,43.016,130.763



TABLE 3 — PAPER-READY PSEUDO-PERPLEXITY RESULTS
Lower scores indicate better language-model fit.


,EN,ZH,JA,TH,FIL
Model,,,,,
Baby_EN-ZH,180.74,62.0,65.79,90.18,43.02



Saved full results to:
/content/drive/MyDrive/BabyLM_Chinese_English_Translation/pseudo_perplexity_baby_en_zh_full_results.csv

Saved valid essay counts to:
/content/drive/MyDrive/BabyLM_Chinese_English_Translation/pseudo_perplexity_baby_en_zh_valid_counts.csv

Saved paper-ready Table 3 CSV to:
/content/drive/MyDrive/BabyLM_Chinese_English_Translation/table3_pseudo_perplexity_baby_en_zh.csv

Saved Overleaf LaTeX table to:
/content/drive/MyDrive/BabyLM_Chinese_English_Translation/table3_pseudo_perplexity_baby_en_zh.tex

LaTeX preview:
\begin{table}[ht]
\caption{Pseudo-perplexity scores for each model across native English and different learner language groups. Lower scores indicate better language-modeling fit.}
\label{tab:pseudo_perplexity}
\begin{tabular}{lrrrrr}
\toprule
 & EN & ZH & JA & TH & FIL \\
Model &  &  &  &  &  \\
\midrule
Baby_EN-ZH & 180.74 & 62.00 & 65.79 & 90.18 & 43.02 \\
\bottomrule
\end{tabular}
\end{table}



In [2]:
import os
import re
import pandas as pd

PROJECT_DIR = "/content/drive/MyDrive/BabyLM_Chinese_English_Translation"

GOLD_EXCEL = "/content/drive/MyDrive/mandarin_Questions_Full.xlsx"

MODEL_PATHS = {
    "English BERT Base":
        f"{PROJECT_DIR}/bert_base_english_preposition_eval/"
        "bert_base_english_normalized_scores.csv",

    "English BabyRoBERTa":
        f"{PROJECT_DIR}/babyroberta_english_preposition_eval/"
        "babyroberta_english_normalized_scores.csv",

    "English BabyRoBERTa + MUSE":
        f"{PROJECT_DIR}/babyroberta_english_word_muse_preposition_eval/"
        "babyroberta_english_word_muse_normalized_scores.csv",

    "Baby_W2W-ZH":
        f"{PROJECT_DIR}/babyroberta_108M_preposition_eval/"
        "babyroberta_108M_normalized_scores.csv",

    "English BabyRoBERTa + Essays":
        f"{PROJECT_DIR}/babyroberta_english_essays_preposition_eval/"
        "babyroberta_english_essays_normalized_scores.csv",

    "Baby_W2W-Mix":
        f"{PROJECT_DIR}/babyroberta_mix_70_30_preposition_eval/"
        "babyroberta_mix_70_30_normalized_scores.csv",

    # New Model C: 70% original Chinese + 30% original English
    "Baby_EN-ZH":
        f"{PROJECT_DIR}/babyroberta_en_zh_preposition_eval/"
        "babyroberta_en_zh_normalized_scores.csv",
}
print("=" * 90)
print("CHECKING MODEL PREDICTION FILES")
print("=" * 90)

for model_name, model_path in MODEL_PATHS.items():
    print(f"{model_name:38s}: {os.path.exists(model_path)}")
    print(f"  {model_path}")

missing_files = [
    path
    for path in MODEL_PATHS.values()
    if not os.path.exists(path)
]

if missing_files:
    raise FileNotFoundError(
        "One or more normalized prediction files are missing:\n"
        + "\n".join(missing_files)
    )


def normalize_text(text):
    text = str(text).lower()

    text = text.replace("，", ",")
    text = text.replace("’", "'")
    text = text.replace("`", "'")

    text = text.replace("[mask]", " mask ")
    text = text.replace("<mask>", " mask ")
    text = text.replace("____", " mask ")
    text = text.replace("___", " mask ")
    text = text.replace("__", " mask ")

    text = re.sub(r"[^\w\s]", " ", text)
    text = re.sub(r"\s+", " ", text)

    return text.strip()


def find_best_match(question, model_df):
    q_words = set(question.split())

    best_row = None
    best_score = 0

    for _, row in model_df.iterrows():
        sentence_words = set(row["sentence_norm"].split())
        overlap = len(q_words.intersection(sentence_words))

        if overlap > best_score:
            best_score = overlap
            best_row = row

    return best_row if best_score >= 6 else None


def compute_human_like_error(gold_path, model_path):
    gold = pd.read_excel(gold_path)
    model = pd.read_csv(model_path)

    QUESTION_COL = "Question.1"
    PREP_COL = "Preposition"
    PERCENT_COL = "Precent"
    CORRECT_COL = "Correct"

    gold["question_norm"] = gold[QUESTION_COL].apply(normalize_text)
    model["sentence_norm"] = model["sentence"].apply(normalize_text)

    gold[PERCENT_COL] = pd.to_numeric(
        gold[PERCENT_COL],
        errors="coerce"
    )

    gold[CORRECT_COL] = pd.to_numeric(
        gold[CORRECT_COL],
        errors="coerce"
    )

    probability_columns = [
        column
        for column in model.columns
        if column not in ["sentence", "sentence_norm"]
    ]

    correct = 0
    total = 0
    wrong_dominant_cases = 0
    unmatched = 0

    for question, group in gold.groupby("question_norm"):
        group = group.sort_values(
            by=PERCENT_COL,
            ascending=False
        )

        if len(group) < 2:
            continue

        top = group.iloc[0]

        # Keep only questions where the most common human answer was wrong
        if top[CORRECT_COL] != 0:
            continue

        wrong_dominant_cases += 1

        human_wrong_answer = str(
            top[PREP_COL]
        ).strip().lower()

        model_row = find_best_match(
            question,
            model
        )

        if model_row is None:
            unmatched += 1
            continue

        probabilities = pd.to_numeric(
            model_row[probability_columns],
            errors="coerce"
        )

        if probabilities.isnull().all():
            continue

        model_answer = (
            probabilities
            .idxmax()
            .strip()
            .lower()
        )

        if model_answer == human_wrong_answer:
            correct += 1

        total += 1

    accuracy = correct / total if total else 0.0

    return {
        "accuracy": accuracy,
        "correct": correct,
        "total": total,
        "wrong_dominant_cases": wrong_dominant_cases,
        "unmatched": unmatched,
    }


print("\n" + "=" * 90)
print("TABLE 5 — HUMAN-LIKE INCORRECT PREPOSITION CHOICE")
print("=" * 90)

table5_results = []

for model_name, model_path in MODEL_PATHS.items():
    result = compute_human_like_error(
        GOLD_EXCEL,
        model_path
    )

    table5_results.append({
        "Model": model_name,
        "Accuracy": result["accuracy"],
        "Correct": result["correct"],
        "Total": result["total"],
        "Wrong-dominant cases": result["wrong_dominant_cases"],
        "Unmatched": result["unmatched"],
    })

    print(
        f"{model_name:38s}: "
        f"{result['accuracy']:.4f} "
        f"({result['correct']}/{result['total']}), "
        f"wrong-dominant={result['wrong_dominant_cases']}, "
        f"unmatched={result['unmatched']}"
    )

table5_df = pd.DataFrame(table5_results)

output_file = (
    f"{PROJECT_DIR}/"
    "table5_human_like_error_all_seven_models.csv"
)

table5_df.to_csv(
    output_file,
    index=False
)

print("\nSaved to:")
print(output_file)

display(table5_df)

CHECKING MODEL PREDICTION FILES
English BERT Base                     : True
  /content/drive/MyDrive/BabyLM_Chinese_English_Translation/bert_base_english_preposition_eval/bert_base_english_normalized_scores.csv
English BabyRoBERTa                   : True
  /content/drive/MyDrive/BabyLM_Chinese_English_Translation/babyroberta_english_preposition_eval/babyroberta_english_normalized_scores.csv
English BabyRoBERTa + MUSE            : True
  /content/drive/MyDrive/BabyLM_Chinese_English_Translation/babyroberta_english_word_muse_preposition_eval/babyroberta_english_word_muse_normalized_scores.csv
Baby_W2W-ZH                           : True
  /content/drive/MyDrive/BabyLM_Chinese_English_Translation/babyroberta_108M_preposition_eval/babyroberta_108M_normalized_scores.csv
English BabyRoBERTa + Essays          : True
  /content/drive/MyDrive/BabyLM_Chinese_English_Translation/babyroberta_english_essays_preposition_eval/babyroberta_english_essays_normalized_scores.csv
Baby_W2W-Mix            

,Model,Accuracy,Correct,Total,Wrong-dominant cases,Unmatched
0,English BERT Base,0.016667,1,60,62,2
1,English BabyRoBERTa,0.333333,20,60,62,2
2,English BabyRoBERTa + MUSE,0.333333,20,60,62,2
3,Baby_W2W-ZH,0.216667,13,60,62,2
4,English BabyRoBERTa + Essays,0.333333,20,60,62,2
5,Baby_W2W-Mix,0.283333,17,60,62,2
6,Baby_EN-ZH,0.266667,16,60,62,2


In [2]:
import os
import re
import pandas as pd

# =============================================================================
# PATHS
# =============================================================================

PROJECT_DIR = "/content/drive/MyDrive/BabyLM_Chinese_English_Translation"

GOLD_CSV = "/content/drive/MyDrive/gold.csv"
QUESTION_EXCEL = "/content/drive/MyDrive/mandarin_Questions_Full.xlsx"

MODEL_PATHS = {
    "English BERT Base":
        f"{PROJECT_DIR}/bert_base_english_preposition_eval/"
        "bert_base_english_normalized_scores.csv",

    "English BabyRoBERTa":
        f"{PROJECT_DIR}/babyroberta_english_preposition_eval/"
        "babyroberta_english_normalized_scores.csv",

    "English BabyRoBERTa + MUSE":
        f"{PROJECT_DIR}/babyroberta_english_word_muse_preposition_eval/"
        "babyroberta_english_word_muse_normalized_scores.csv",

    "Baby_W2W-ZH":
        f"{PROJECT_DIR}/babyroberta_108M_preposition_eval/"
        "babyroberta_108M_normalized_scores.csv",

    "English BabyRoBERTa + Essays":
        f"{PROJECT_DIR}/babyroberta_english_essays_preposition_eval/"
        "babyroberta_english_essays_normalized_scores.csv",

    "Baby_W2W-Mix":
        f"{PROJECT_DIR}/babyroberta_mix_70_30_preposition_eval/"
        "babyroberta_mix_70_30_normalized_scores.csv",

    # New model: 70% original Chinese + 30% original English
    "Baby_EN-ZH":
        f"{PROJECT_DIR}/babyroberta_en_zh_preposition_eval/"
        "babyroberta_en_zh_normalized_scores.csv",
}

# =============================================================================
# NORMALIZE QUESTION TEXT
# =============================================================================

def normalize_question(text):
    text = str(text).lower()

    text = text.replace("，", ",")
    text = text.replace("’", "'")
    text = text.replace("`", "'")

    # Make all blank/mask forms identical
    text = text.replace("[mask]", " mask ")
    text = text.replace("<mask>", " mask ")
    text = text.replace("____", " mask ")
    text = text.replace("___", " mask ")
    text = text.replace("__", " mask ")

    text = re.sub(r"[^\w\s]", " ", text)
    text = re.sub(r"\s+", " ", text)

    return text.strip()

# =============================================================================
# LOAD HUMAN RESPONSE DATA
# =============================================================================

gold = pd.read_csv(GOLD_CSV)

gold["question_norm"] = gold["question"].apply(normalize_question)
gold["preposition"] = (
    gold["preposition"]
    .astype(str)
    .str.strip()
    .str.lower()
)
gold["Precent"] = pd.to_numeric(gold["Precent"], errors="coerce")

# Human top choice for each question
human_top = (
    gold.sort_values(
        ["question_norm", "Precent"],
        ascending=[True, False]
    )
    .groupby("question_norm", as_index=False)
    .first()
    [["question_norm", "preposition", "Precent"]]
    .rename(columns={"preposition": "human_top_choice"})
)

print("Human questions in gold.csv:", len(human_top))

# =============================================================================
# LOAD GRAMMATICALLY CORRECT ANSWERS
# Needed only to identify Table 5 wrong-dominant cases
# =============================================================================

correct_data = pd.read_excel(QUESTION_EXCEL)

question_col = "Question.1"
prep_col = "Preposition"
correct_col = "Correct"

correct_data["question_norm"] = (
    correct_data[question_col].apply(normalize_question)
)

correct_data[prep_col] = (
    correct_data[prep_col]
    .astype(str)
    .str.strip()
    .str.lower()
)

correct_data[correct_col] = pd.to_numeric(
    correct_data[correct_col],
    errors="coerce"
)

correct_answers = (
    correct_data[correct_data[correct_col] == 1]
    .drop_duplicates("question_norm")
    [["question_norm", prep_col]]
    .rename(columns={prep_col: "grammatical_answer"})
)

human_top = human_top.merge(
    correct_answers,
    on="question_norm",
    how="left"
)

human_top["human_top_is_incorrect"] = (
    human_top["human_top_choice"]
    != human_top["grammatical_answer"]
)

# =============================================================================
# EVALUATE ONE MODEL
# =============================================================================

def evaluate_model(model_name, model_path):
    model = pd.read_csv(model_path)

    sentence_col = (
        "sentence" if "sentence" in model.columns else "Question"
    )

    model["question_norm"] = model[sentence_col].apply(
        normalize_question
    )

    excluded = {
        "sentence",
        "Question",
        "question_norm",
    }

    probability_columns = [
        column
        for column in model.columns
        if column not in excluded
    ]

    # Convert all probability columns to numeric
    model[probability_columns] = model[
        probability_columns
    ].apply(pd.to_numeric, errors="coerce")

    model["model_choice"] = model[
        probability_columns
    ].idxmax(axis=1)

    model["model_choice"] = (
        model["model_choice"]
        .astype(str)
        .str.strip()
        .str.lower()
    )

    # Exact question matching
    comparison = human_top.merge(
        model[["question_norm", "model_choice"]],
        on="question_norm",
        how="inner"
    )

    # -------------------------------------------------------------------------
    # TABLE 4:
    # Agreement with the most common Mandarin-learner answer
    # -------------------------------------------------------------------------

    table4_correct = (
        comparison["model_choice"]
        == comparison["human_top_choice"]
    )

    table4_total = len(comparison)
    table4_count = int(table4_correct.sum())
    table4_accuracy = (
        table4_count / table4_total
        if table4_total else 0
    )

    # -------------------------------------------------------------------------
    # TABLE 5:
    # Keep only questions where the learners' top answer was incorrect.
    # Credit model when it chooses that same incorrect answer.
    # -------------------------------------------------------------------------

    wrong_cases = comparison[
        comparison["human_top_is_incorrect"]
    ].copy()

    table5_correct = (
        wrong_cases["model_choice"]
        == wrong_cases["human_top_choice"]
    )

    table5_total = len(wrong_cases)
    table5_count = int(table5_correct.sum())
    table5_accuracy = (
        table5_count / table5_total
        if table5_total else 0
    )

    return {
        "Model": model_name,
        "Matched questions": table4_total,
        "Table 4 correct": table4_count,
        "Table 4 accuracy": table4_accuracy,
        "Wrong-dominant questions": table5_total,
        "Table 5 correct": table5_count,
        "Table 5 accuracy": table5_accuracy,
    }

# =============================================================================
# RUN ALL MODELS
# =============================================================================

results = []

for model_name, model_path in MODEL_PATHS.items():
    print(f"Evaluating {model_name}...")
    results.append(
        evaluate_model(model_name, model_path)
    )

results_df = pd.DataFrame(results)

print("\n" + "=" * 90)
print("CORRECTED TABLE 4 — AGREEMENT WITH HUMAN TOP CHOICE")
print("=" * 90)

for _, row in results_df.iterrows():
    print(
        f"{row['Model']:35s}: "
        f"{row['Table 4 accuracy']:.4f} "
        f"({int(row['Table 4 correct'])}/"
        f"{int(row['Matched questions'])})"
    )

print("\n" + "=" * 90)
print("CORRECTED TABLE 5 — SAME INCORRECT CHOICE AS HUMANS")
print("=" * 90)

for _, row in results_df.iterrows():
    print(
        f"{row['Model']:35s}: "
        f"{row['Table 5 accuracy']:.4f} "
        f"({int(row['Table 5 correct'])}/"
        f"{int(row['Wrong-dominant questions'])})"
    )

OUTPUT = (
    f"{PROJECT_DIR}/"
    "corrected_preposition_results_all_seven_models.csv"
)
results_df.to_csv(OUTPUT, index=False)

print("\nSaved to:")
print(OUTPUT)

Human questions in gold.csv: 100
Evaluating English BERT Base...
Evaluating English BabyRoBERTa...
Evaluating English BabyRoBERTa + MUSE...
Evaluating Baby_W2W-ZH...
Evaluating English BabyRoBERTa + Essays...
Evaluating Baby_W2W-Mix...
Evaluating Baby_EN-ZH...

CORRECTED TABLE 4 — AGREEMENT WITH HUMAN TOP CHOICE
English BERT Base                  : 0.2500 (25/100)
English BabyRoBERTa                : 0.3000 (30/100)
English BabyRoBERTa + MUSE         : 0.3000 (30/100)
Baby_W2W-ZH                        : 0.2000 (20/100)
English BabyRoBERTa + Essays       : 0.3100 (31/100)
Baby_W2W-Mix                       : 0.2600 (26/100)
Baby_EN-ZH                         : 0.2400 (24/100)

CORRECTED TABLE 5 — SAME INCORRECT CHOICE AS HUMANS
English BERT Base                  : 0.2347 (23/98)
English BabyRoBERTa                : 0.3061 (30/98)
English BabyRoBERTa + MUSE         : 0.3061 (30/98)
Baby_W2W-ZH                        : 0.1939 (19/98)
English BabyRoBERTa + Essays       : 0.3163 (31/98)
Ba

In [4]:
import os

EVAL_ROOT = "/content/evaluation-pipeline-2025"

if not os.path.exists(EVAL_ROOT):
    !git clone https://github.com/babylm/evaluation-pipeline-2025.git "$EVAL_ROOT"
else:
    print("Repository already exists:", EVAL_ROOT)

%cd "$EVAL_ROOT"

print("\nCurrent folder:")
!pwd

print("\nRepository files:")
!ls

Repository already exists: /content/evaluation-pipeline-2025
/content/evaluation-pipeline-2025

Current folder:
/content/evaluation-pipeline-2025

Repository files:
evaluation_data


Install the evaluation dependencies

In [5]:
%cd /content/evaluation-pipeline-2025

print("Installing BabyLM evaluation dependencies...")

!pip install -q \
    "transformers==4.51.3" \
    "datasets==3.6.0" \
    "tokenizers==0.21.1" \
    "safetensors==0.5.3" \
    "scikit-learn==1.6.1" \
    "statsmodels==0.14.4" \
    "nltk==3.9.1" \
    "einops==0.8.1" \
    "polars==1.30.0" \
    "pyspellchecker==0.8.3"

print("\nChecking installed versions...")

import torch
import transformers
import datasets
import tokenizers

print("Torch       :", torch.__version__)
print("Transformers:", transformers.__version__)
print("Datasets    :", datasets.__version__)
print("Tokenizers  :", tokenizers.__version__)
print("CUDA        :", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU         :", torch.cuda.get_device_name(0))

/content/evaluation-pipeline-2025
Installing BabyLM evaluation dependencies...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 127.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.5/491.5 kB 43.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 73.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 471.6/471.6 kB 38.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 95.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.4/64.4 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 36.3/36.3 MB 73.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.2/7.2 MB 101.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 50.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.19.0 requires huggingfa

In [6]:
%cd /content/evaluation-pipeline-2025

!pip install -q osfclient

print("=" * 90)
print("LISTING BABYLM 2025 EVALUATION DATA FROM OSF")
print("=" * 90)

!osf -p ryjfm list

/content/evaluation-pipeline-2025
LISTING BABYLM 2025 EVALUATION DATA FROM OSF
osfstorage/multimodal_data/readme.md
osfstorage/evaluation_data/fast_eval/supplement_fast/subject_aux_inversion.jsonl
osfstorage/evaluation_data/fast_eval/supplement_fast/turn_taking.jsonl
osfstorage/evaluation_data/fast_eval/supplement_fast/hypernym.jsonl
osfstorage/evaluation_data/fast_eval/supplement_fast/qa_congruence_easy.jsonl
osfstorage/evaluation_data/fast_eval/supplement_fast/qa_congruence_tricky.jsonl
osfstorage/evaluation_data/fast_eval/ewok_fast.zip
osfstorage/evaluation_data/fast_eval/reading/reading_data.csv
osfstorage/evaluation_data/fast_eval/wug_adj_nominalization/wug_adj_nominalization.jsonl
osfstorage/evaluation_data/fast_eval/entity_tracking_fast/regular.jsonl
osfstorage/evaluation_data/fast_eval/blimp_fast/coordinate_structure_constraint_object_extraction.jsonl
osfstorage/evaluation_data/fast_eval/blimp_fast/inchoative.jsonl
osfstorage/evaluation_data/fast_eval/blimp_fast/sentential_subj

Download full BLiMP data only

In [5]:
import os
from osfclient.api import OSF

PROJECT_ID = "ryjfm"

REMOTE_PREFIX = (
    "/evaluation_data/full_eval/blimp_filtered/"
)

LOCAL_BLIMP_DIR = (
    "/content/evaluation-pipeline-2025/"
    "evaluation_data/full_eval/blimp_filtered"
)

os.makedirs(LOCAL_BLIMP_DIR, exist_ok=True)

print("=" * 90)
print("DOWNLOADING FULL BLIMP EVALUATION DATA")
print("=" * 90)

osf = OSF()
project = osf.project(PROJECT_ID)
storage = project.storage("osfstorage")


def walk_osf_files(folder):
    """Recursively yield files from an OSF folder."""
    for item in folder.files:
        if getattr(item, "is_file", False):
            yield item
        else:
            yield from walk_osf_files(item)


downloaded = 0
skipped = 0

for osf_file in walk_osf_files(storage):
    remote_path = osf_file.path

    if not remote_path.startswith(REMOTE_PREFIX):
        continue

    relative_path = remote_path[len(REMOTE_PREFIX):]

    if not relative_path:
        continue

    local_path = os.path.join(
        LOCAL_BLIMP_DIR,
        relative_path
    )

    os.makedirs(
        os.path.dirname(local_path),
        exist_ok=True
    )

    if (
        os.path.exists(local_path)
        and os.path.getsize(local_path) > 0
    ):
        skipped += 1
        continue

    with open(local_path, "wb") as output:
        osf_file.write_to(output)

    downloaded += 1

    if downloaded % 10 == 0:
        print(f"Downloaded {downloaded} files...")


jsonl_files = sorted(
    filename
    for filename in os.listdir(LOCAL_BLIMP_DIR)
    if filename.endswith(".jsonl")
)

print("\nDownload complete.")
print("New files downloaded:", downloaded)
print("Existing files skipped:", skipped)
print("BLiMP JSONL files found:", len(jsonl_files))
print("Local folder:")
print(LOCAL_BLIMP_DIR)

print("\nFirst 10 files:")
for filename in jsonl_files[:10]:
    print("-", filename)

if len(jsonl_files) < 60:
    raise RuntimeError(
        "Too few BLiMP files were downloaded. "
        f"Expected approximately 67, found {len(jsonl_files)}."
    )

DOWNLOADING FULL BLIMP EVALUATION DATA


AttributeError: 'File' object has no attribute 'files'

In [3]:
%cd /content/evaluation-pipeline-2025

!find evaluation_data/full_eval/blimp_filtered -type f | head -20

print("\nTotal files:")

!find evaluation_data/full_eval/blimp_filtered -type f | wc -l

/content/evaluation-pipeline-2025

Total files:
0


In [9]:
import os
import subprocess

PROJECT_ID = "ryjfm"

REMOTE_PREFIX = (
    "osfstorage/evaluation_data/full_eval/blimp_filtered/"
)

LOCAL_DIR = (
    "/content/evaluation-pipeline-2025/"
    "evaluation_data/full_eval/blimp_filtered"
)

os.makedirs(LOCAL_DIR, exist_ok=True)

print("=" * 90)
print("DOWNLOADING BLIMP FILES INDIVIDUALLY")
print("=" * 90)

# Get the complete OSF file listing
result = subprocess.run(
    ["osf", "-p", PROJECT_ID, "list"],
    capture_output=True,
    text=True,
    check=True,
)

remote_files = [
    line.strip()
    for line in result.stdout.splitlines()
    if line.strip().startswith(REMOTE_PREFIX)
    and line.strip().endswith(".jsonl")
]

print("BLiMP files found remotely:", len(remote_files))

if not remote_files:
    raise RuntimeError("No BLiMP files found in the OSF listing.")

downloaded = 0
skipped = 0
failed = []

for index, remote_path in enumerate(remote_files, start=1):
    filename = os.path.basename(remote_path)
    local_path = os.path.join(LOCAL_DIR, filename)

    if os.path.exists(local_path) and os.path.getsize(local_path) > 0:
        skipped += 1
        continue

    command = [
        "osf",
        "-p",
        PROJECT_ID,
        "fetch",
        remote_path,
        local_path,
    ]

    fetch_result = subprocess.run(
        command,
        capture_output=True,
        text=True,
    )

    if fetch_result.returncode == 0:
        downloaded += 1
    else:
        failed.append(
            (remote_path, fetch_result.stderr.strip())
        )

    if index % 10 == 0:
        print(f"Processed {index}/{len(remote_files)} files...")

local_files = sorted(
    filename
    for filename in os.listdir(LOCAL_DIR)
    if filename.endswith(".jsonl")
    and os.path.getsize(os.path.join(LOCAL_DIR, filename)) > 0
)

print("\nDownload complete.")
print("Downloaded :", downloaded)
print("Skipped    :", skipped)
print("Failed     :", len(failed))
print("Local files:", len(local_files))

if failed:
    print("\nFirst failed downloads:")
    for path, error in failed[:5]:
        print(path)
        print(error)

print("\nFirst 10 local files:")
for filename in local_files[:10]:
    print("-", filename)

if len(local_files) < 60:
    raise RuntimeError(
        f"Only {len(local_files)} BLiMP files are available. "
        "Expected roughly 67."
    )

DOWNLOADING BLIMP FILES INDIVIDUALLY
BLiMP files found remotely: 67
Processed 10/67 files...
Processed 20/67 files...
Processed 30/67 files...
Processed 40/67 files...
Processed 50/67 files...
Processed 60/67 files...

Download complete.
Downloaded : 65
Skipped    : 2
Failed     : 0
Local files: 65

First 10 local files:
- adjunct_island.jsonl
- anaphor_gender_agreement.jsonl
- anaphor_number_agreement.jsonl
- animate_subject_passive.jsonl
- animate_subject_trans.jsonl
- causative.jsonl
- complex_NP_island.jsonl
- coordinate_structure_constraint_complex_left_branch.jsonl
- coordinate_structure_constraint_object_extraction.jsonl
- determiner_noun_agreement_1.jsonl


In [8]:
import os

print(os.getcwd())

print("\nFolders:")
for x in sorted(os.listdir("/content/evaluation-pipeline-2025")):
    print(x)

/content/evaluation-pipeline-2025

Folders:
evaluation_data


In [15]:
!cd /content/evaluation-pipeline-2025

!ls evaluation_data/full_eval/blimp_filtered | wc -l

65


In [10]:
!find /content/evaluation-pipeline-2025 -maxdepth 3 -type f | grep run.py

In [17]:
!find evaluation_pipeline -name "run.py"

find: ‘evaluation_pipeline’: No such file or directory


In [18]:
!find evaluation_pipeline -path "*sentence_zero_shot*"

find: ‘evaluation_pipeline’: No such file or directory


In [19]:
!pwd

/content/evaluation-pipeline-2025


In [20]:
!ls

evaluation_data


In [21]:
!ls /content/evaluation-pipeline-2025

evaluation_data


In [22]:
!find /content/evaluation-pipeline-2025 -maxdepth 2 -type d

/content/evaluation-pipeline-2025
/content/evaluation-pipeline-2025/evaluation_data
/content/evaluation-pipeline-2025/evaluation_data/full_eval


In [23]:
import os

BLIMP_DIR = "/content/evaluation-pipeline-2025/evaluation_data/full_eval/blimp_filtered"

print("Exists:", os.path.exists(BLIMP_DIR))

files = sorted(
    f for f in os.listdir(BLIMP_DIR)
    if f.endswith(".jsonl")
)

print("Number of BLiMP files:", len(files))
print(files[:10])

Exists: True
Number of BLiMP files: 65
['adjunct_island.jsonl', 'anaphor_gender_agreement.jsonl', 'anaphor_number_agreement.jsonl', 'animate_subject_passive.jsonl', 'animate_subject_trans.jsonl', 'causative.jsonl', 'complex_NP_island.jsonl', 'coordinate_structure_constraint_complex_left_branch.jsonl', 'coordinate_structure_constraint_object_extraction.jsonl', 'determiner_noun_agreement_1.jsonl']


In [27]:
import os
import shutil
import subprocess

EVAL_ROOT = "/content/evaluation-pipeline-2025"
TEMP_REPO = "/content/evaluation-pipeline-2025-temp"

BLIMP_DIR = (
    f"{EVAL_ROOT}/evaluation_data/full_eval/blimp_filtered"
)

EXPECTED_RUN_FILE = (
    f"{EVAL_ROOT}/evaluation_pipeline/"
    "sentence_zero_shot/run.py"
)

print("=" * 90)
print("RESTORING BABYLM EVALUATION CODE")
print("=" * 90)

print("BLiMP data exists:", os.path.isdir(BLIMP_DIR))

if os.path.isdir(BLIMP_DIR):
    blimp_files = [
        filename
        for filename in os.listdir(BLIMP_DIR)
        if filename.endswith(".jsonl")
    ]
else:
    blimp_files = []

print("Existing BLiMP files:", len(blimp_files))
print("Evaluator currently exists:", os.path.isfile(EXPECTED_RUN_FILE))

# Clone the official code into a temporary directory
if os.path.exists(TEMP_REPO):
    shutil.rmtree(TEMP_REPO)

subprocess.run(
    [
        "git",
        "clone",
        "--depth",
        "1",
        "https://github.com/babylm/evaluation-pipeline-2025.git",
        TEMP_REPO,
    ],
    check=True,
)

# Keep the already-downloaded evaluation data, but restore repository code
os.makedirs(EVAL_ROOT, exist_ok=True)

items_to_restore = [
    "evaluation_pipeline",
    "eval_zero_shot.sh",
    "eval_zero_shot_fast.sh",
    "requirements.txt",
    "README.md",
]

for item in items_to_restore:
    source = os.path.join(TEMP_REPO, item)
    destination = os.path.join(EVAL_ROOT, item)

    if os.path.isdir(source):
        if os.path.exists(destination):
            shutil.rmtree(destination)
        shutil.copytree(source, destination)

    elif os.path.isfile(source):
        shutil.copy2(source, destination)

shutil.rmtree(TEMP_REPO)

print("\nRepair complete.")
print("Evaluator exists:", os.path.isfile(EXPECTED_RUN_FILE))
print("BLiMP files remain:", len(blimp_files))

if not os.path.isfile(EXPECTED_RUN_FILE):
    raise FileNotFoundError(
        f"Evaluator still missing:\n{EXPECTED_RUN_FILE}"
    )

if len(blimp_files) < 60:
    raise RuntimeError(
        "BLiMP data is incomplete."
    )

print("\nREADY FOR EVALUATION.")

RESTORING BABYLM EVALUATION CODE
BLiMP data exists: True
Existing BLiMP files: 65
Evaluator currently exists: False

Repair complete.
Evaluator exists: True
BLiMP files remain: 65

READY FOR EVALUATION.


BLIMP EVALUATION — SIX BABYROBERTA MODELS


FileNotFoundError: Evaluation repository not found:
/content/evaluation-pipeline-2025

In [3]:
import os
import re
import pandas as pd

# =============================================================================
# CONFIGURATION
# =============================================================================

PROJECT_DIR = (
    "/content/drive/MyDrive/"
    "BabyLM_Chinese_English_Translation"
)

RESULTS_ROOT = (
    f"{PROJECT_DIR}/blimp_evaluation_results"
)

OUTPUT_DIR = (
    f"{PROJECT_DIR}/blimp_aggregated_results"
)

os.makedirs(OUTPUT_DIR, exist_ok=True)

# Internal folder name -> clear paper-ready model name
MODEL_NAMES = {
    "Baby_EN":
        "English BabyRoBERTa",

    "Baby_L2E":
        "English BabyRoBERTa + Essays",

    "Baby_MUSE":
        "English BabyRoBERTa + MUSE",

    "Baby_W2W_ZH":
        "Word-to-Word Chinese BabyRoBERTa",

    "Baby_W2W_Mix":
        "70% W2W Chinese + 30% English",

    "Baby_EN_ZH":
        "70% Original Chinese + 30% English",
}

# Preferred order in all output tables
MODEL_ORDER = list(MODEL_NAMES.values())


# =============================================================================
# FIND REPORT FILES
# =============================================================================

def find_report(model_folder):
    """
    Recursively find best_temperature_report.txt inside one model folder.
    """

    model_root = os.path.join(
        RESULTS_ROOT,
        model_folder
    )

    if not os.path.isdir(model_root):
        raise FileNotFoundError(
            f"Model result folder not found:\n{model_root}"
        )

    matches = []

    for root, _, files in os.walk(model_root):
        if "best_temperature_report.txt" in files:
            matches.append(
                os.path.join(
                    root,
                    "best_temperature_report.txt"
                )
            )

    if not matches:
        raise FileNotFoundError(
            "No best_temperature_report.txt found under:\n"
            f"{model_root}"
        )

    if len(matches) > 1:
        print(
            f"Warning: multiple reports found for {model_folder}. "
            "Using the first one:"
        )
        for path in matches:
            print(" -", path)

    return sorted(matches)[0]


# =============================================================================
# PARSE ONE REPORT
# =============================================================================

def parse_report(report_path):
    """
    Extract:
      - temperature
      - field accuracy
      - UID accuracy
      - linguistics-term accuracy
      - average accuracy
    """

    with open(
        report_path,
        "r",
        encoding="utf-8"
    ) as file:
        lines = [
            line.strip()
            for line in file
        ]

    data = {
        "temperature": None,
        "field": {},
        "uid": {},
        "linguistics_term": {},
        "average_accuracy": None,
    }

    current_section = None

    for line in lines:

        if not line:
            continue

        if line.startswith("TEMPERATURE:"):
            value = line.split(
                ":",
                1
            )[1].strip()

            try:
                data["temperature"] = float(value)
            except ValueError:
                data["temperature"] = value

            continue

        if line == "### FIELD ACCURACY":
            current_section = "field"
            continue

        if line == "### UID ACCURACY":
            current_section = "uid"
            continue

        if line == "### LINGUISTICS_TERM ACCURACY":
            current_section = "linguistics_term"
            continue

        if line == "### AVERAGE ACCURACY":
            current_section = "average_accuracy"
            continue

        if line.startswith("###"):
            current_section = None
            continue

        if current_section in {
            "field",
            "uid",
            "linguistics_term"
        }:
            match = re.match(
                r"^(.+?):\s*(-?\d+(?:\.\d+)?)$",
                line
            )

            if match:
                key = match.group(1).strip()
                value = float(match.group(2))

                data[current_section][key] = value

            continue

        if current_section == "average_accuracy":
            try:
                data["average_accuracy"] = float(line)
                current_section = None
            except ValueError:
                pass

    return data


# =============================================================================
# READ ALL SIX REPORTS
# =============================================================================

all_results = {}
report_paths = {}

print("=" * 100)
print("READING BLIMP REPORTS")
print("=" * 100)

for folder_name, paper_name in MODEL_NAMES.items():

    report_path = find_report(
        folder_name
    )

    parsed = parse_report(
        report_path
    )

    all_results[paper_name] = parsed
    report_paths[paper_name] = report_path

    print(
        f"{paper_name:45s} | "
        f"Overall: {parsed['average_accuracy']}"
    )

    print(
        f"  Report: {report_path}"
    )


# =============================================================================
# BUILD CATEGORY TABLE
# LINGUISTICS_TERM ACCURACY
# =============================================================================

category_keys = sorted(
    {
        category
        for result in all_results.values()
        for category in result["linguistics_term"].keys()
    }
)

category_table = pd.DataFrame(
    {
        model_name: [
            all_results[model_name][
                "linguistics_term"
            ].get(category)
            for category in category_keys
        ]
        for model_name in MODEL_ORDER
    },
    index=category_keys
)

category_table.index.name = "Category"

# Add overall accuracy as final row
category_table.loc["overall_accuracy"] = [
    all_results[model_name][
        "average_accuracy"
    ]
    for model_name in MODEL_ORDER
]


# =============================================================================
# BUILD FIELD TABLE
# =============================================================================

field_keys = sorted(
    {
        field
        for result in all_results.values()
        for field in result["field"].keys()
    }
)

field_table = pd.DataFrame(
    {
        model_name: [
            all_results[model_name][
                "field"
            ].get(field)
            for field in field_keys
        ]
        for model_name in MODEL_ORDER
    },
    index=field_keys
)

field_table.index.name = "Field"


# =============================================================================
# BUILD UID / INDIVIDUAL TEST TABLE
# =============================================================================

uid_keys = sorted(
    {
        uid
        for result in all_results.values()
        for uid in result["uid"].keys()
    }
)

uid_table = pd.DataFrame(
    {
        model_name: [
            all_results[model_name][
                "uid"
            ].get(uid)
            for uid in uid_keys
        ]
        for model_name in MODEL_ORDER
    },
    index=uid_keys
)

uid_table.index.name = "BLiMP phenomenon"


# =============================================================================
# BUILD OVERALL SUMMARY TABLE
# =============================================================================

summary_rows = []

for model_name in MODEL_ORDER:

    summary_rows.append({
        "Model": model_name,
        "Temperature":
            all_results[model_name]["temperature"],
        "Average BLiMP accuracy":
            all_results[model_name][
                "average_accuracy"
            ],
        "Report path":
            report_paths[model_name],
    })

summary_table = pd.DataFrame(
    summary_rows
)


# =============================================================================
# MAKE LABELS MORE READABLE
# =============================================================================

CATEGORY_LABELS = {
    "binding": "Binding",
    "subject_verb_agreement":
        "Subject–Verb Agreement",
    "ellipsis": "Ellipsis",
    "determiner_noun_agreement":
        "Determiner–Noun Agreement",
    "irregular_forms": "Irregular Forms",
    "npi_licensing": "NPI Licensing",
    "quantifiers": "Quantifiers",
    "filler_gap_dependency":
        "Filler–Gap Dependency",
    "argument_structure":
        "Argument Structure",
    "island_effects": "Island Effects",
    "control_raising": "Control/Raising",
    "anaphor_agreement":
        "Anaphor Agreement",
    "s-selection":
        "Selectional Restrictions",
    "overall_accuracy":
        "Overall Accuracy",
}

category_table.index = [
    CATEGORY_LABELS.get(
        index,
        index.replace("_", " ").title()
    )
    for index in category_table.index
]

field_table.index = [
    index.replace("_", " ").title()
    for index in field_table.index
]

uid_table.index = [
    index.replace("_", " ")
    for index in uid_table.index
]


# =============================================================================
# ROUND RESULTS
# =============================================================================

category_table = category_table.round(2)
field_table = field_table.round(2)
uid_table = uid_table.round(2)
summary_table[
    "Average BLiMP accuracy"
] = summary_table[
    "Average BLiMP accuracy"
].round(2)


# =============================================================================
# SAVE CSV FILES
# =============================================================================

CATEGORY_CSV = (
    f"{OUTPUT_DIR}/BLiMP_category_results.csv"
)

FIELD_CSV = (
    f"{OUTPUT_DIR}/BLiMP_field_results.csv"
)

UID_CSV = (
    f"{OUTPUT_DIR}/BLiMP_uid_results.csv"
)

SUMMARY_CSV = (
    f"{OUTPUT_DIR}/BLiMP_overall_summary.csv"
)

category_table.to_csv(
    CATEGORY_CSV
)

field_table.to_csv(
    FIELD_CSV
)

uid_table.to_csv(
    UID_CSV
)

summary_table.to_csv(
    SUMMARY_CSV,
    index=False
)


# =============================================================================
# SAVE EXCEL WORKBOOK WITH MULTIPLE SHEETS
# =============================================================================

EXCEL_FILE = (
    f"{OUTPUT_DIR}/BLiMP_detailed_results_all_models.xlsx"
)

with pd.ExcelWriter(
    EXCEL_FILE,
    engine="openpyxl"
) as writer:

    category_table.to_excel(
        writer,
        sheet_name="Category Results"
    )

    field_table.to_excel(
        writer,
        sheet_name="Field Results"
    )

    uid_table.to_excel(
        writer,
        sheet_name="Individual Tests"
    )

    summary_table.to_excel(
        writer,
        sheet_name="Overall Summary",
        index=False
    )

    report_path_table = pd.DataFrame({
        "Model": MODEL_ORDER,
        "Report Path": [
            report_paths[model]
            for model in MODEL_ORDER
        ]
    })

    report_path_table.to_excel(
        writer,
        sheet_name="Report Paths",
        index=False
    )


# =============================================================================
# SAVE LATEX TABLE
# =============================================================================

LATEX_FILE = (
    f"{OUTPUT_DIR}/BLiMP_category_results.tex"
)

latex_table = category_table.to_latex(
    float_format="%.2f",
    caption=(
        "BLiMP accuracy by linguistic category for "
        "the six BabyRoBERTa models. Higher values "
        "indicate better selection of the grammatical "
        "sentence in each minimal pair."
    ),
    label="tab:blimp_categories",
    position="ht",
)

with open(
    LATEX_FILE,
    "w",
    encoding="utf-8"
) as file:
    file.write(
        latex_table
    )


# =============================================================================
# DISPLAY RESULTS
# =============================================================================

print("\n" + "=" * 100)
print("BLIMP CATEGORY RESULTS")
print("=" * 100)

display(category_table)

print("\n" + "=" * 100)
print("BLIMP FIELD RESULTS")
print("=" * 100)

display(field_table)

print("\n" + "=" * 100)
print("OVERALL SUMMARY")
print("=" * 100)

display(summary_table[
    [
        "Model",
        "Temperature",
        "Average BLiMP accuracy",
    ]
])


# =============================================================================
# FINAL OUTPUT LOCATIONS
# =============================================================================

print("\n" + "=" * 100)
print("FILES SAVED")
print("=" * 100)

print("\nDetailed Excel workbook:")
print(EXCEL_FILE)

print("\nCategory CSV:")
print(CATEGORY_CSV)

print("\nField CSV:")
print(FIELD_CSV)

print("\nIndividual-test CSV:")
print(UID_CSV)

print("\nOverall-summary CSV:")
print(SUMMARY_CSV)

print("\nLaTeX table:")
print(LATEX_FILE)

print("\nDONE.")

READING BLIMP REPORTS
English BabyRoBERTa                           | Overall: 51.48
  Report: /content/drive/MyDrive/BabyLM_Chinese_English_Translation/blimp_evaluation_results/Baby_EN/babyroberta_english_same_arch_95_5_final/main/zero_shot/mlm/blimp/blimp_filtered/best_temperature_report.txt
English BabyRoBERTa + Essays                  | Overall: 51.57
  Report: /content/drive/MyDrive/BabyLM_Chinese_English_Translation/blimp_evaluation_results/Baby_L2E/final_model/main/zero_shot/mlm/blimp/blimp_filtered/best_temperature_report.txt
English BabyRoBERTa + MUSE                    | Overall: 51.15
  Report: /content/drive/MyDrive/BabyLM_Chinese_English_Translation/blimp_evaluation_results/Baby_MUSE/final_model/main/zero_shot/mlm/blimp/blimp_filtered/best_temperature_report.txt
Word-to-Word Chinese BabyRoBERTa              | Overall: 62.92
  Report: /content/drive/MyDrive/BabyLM_Chinese_English_Translation/blimp_evaluation_results/Baby_W2W_ZH/babyroberta_108M_same_arch_95_5_final/main/zer

,English BabyRoBERTa,English BabyRoBERTa + Essays,English BabyRoBERTa + MUSE,Word-to-Word Chinese BabyRoBERTa,70% W2W Chinese + 30% English,70% Original Chinese + 30% English
Anaphor Agreement,57.36,58.36,56.73,63.88,51.84,67.40
Argument Structure,49.37,51.24,49.34,60.01,53.90,55.81
Binding,46.27,48.57,45.22,59.29,50.68,47.97
Control/Raising,61.95,62.62,61.38,51.85,56.70,59.00
Determiner–Noun Agreement,52.98,53.85,53.22,91.37,54.35,70.31
Ellipsis,41.67,42.51,41.18,89.13,83.70,99.03
Filler–Gap Dependency,57.74,57.98,57.81,54.38,59.61,56.03
Irregular Forms,60.59,56.44,59.01,90.75,57.02,68.26
Island Effects,39.30,36.24,38.63,49.95,34.59,42.60
NPI Licensing,51.11,49.86,50.07,59.86,65.05,50.33



BLIMP FIELD RESULTS


,English BabyRoBERTa,English BabyRoBERTa + Essays,English BabyRoBERTa + MUSE,Word-to-Word Chinese BabyRoBERTa,70% W2W Chinese + 30% English,70% Original Chinese + 30% English
Morphology,53.12,53.28,52.95,74.45,52.50,64.04
Semantics,56.97,56.04,56.93,65.46,64.13,47.52
Syntax,47.78,47.33,47.53,56.18,48.49,52.71
Syntax/Semantics,51.17,52.53,50.31,57.58,53.22,54.81



OVERALL SUMMARY


,Model,Temperature,Average BLiMP accuracy
0,English BabyRoBERTa,1.0,51.48
1,English BabyRoBERTa + Essays,1.0,51.57
2,English BabyRoBERTa + MUSE,1.0,51.15
3,Word-to-Word Chinese BabyRoBERTa,1.0,62.92
4,70% W2W Chinese + 30% English,1.0,53.09
5,70% Original Chinese + 30% English,1.0,55.78



FILES SAVED

Detailed Excel workbook:
/content/drive/MyDrive/BabyLM_Chinese_English_Translation/blimp_aggregated_results/BLiMP_detailed_results_all_models.xlsx

Category CSV:
/content/drive/MyDrive/BabyLM_Chinese_English_Translation/blimp_aggregated_results/BLiMP_category_results.csv

Field CSV:
/content/drive/MyDrive/BabyLM_Chinese_English_Translation/blimp_aggregated_results/BLiMP_field_results.csv

Individual-test CSV:
/content/drive/MyDrive/BabyLM_Chinese_English_Translation/blimp_aggregated_results/BLiMP_uid_results.csv

Overall-summary CSV:
/content/drive/MyDrive/BabyLM_Chinese_English_Translation/blimp_aggregated_results/BLiMP_overall_summary.csv

LaTeX table:
/content/drive/MyDrive/BabyLM_Chinese_English_Translation/blimp_aggregated_results/BLiMP_category_results.tex

DONE.
